### Step 1 — Mount Google Drive and establish PROJECT_ROOT

**Goal:** Connect Colab to Google Drive and define every path the rest of
this project will use, derived from a single PROJECT_ROOT variable.

**Why:** Hardcoded paths break the moment anyone else runs this notebook,
or you run it from a different machine. Every later phase imports these
same variables instead of retyping paths.

**What to expect:** A Drive authorization prompt, then confirmation that
PROJECT_ROOT exists and is writable.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = '/content/drive/MyDrive/CountryWellness'
os.makedirs(DRIVE_BASE, exist_ok=True)

PROJECT_ROOT = os.path.join(DRIVE_BASE, 'Country-Wellness-Score-Prediction')

print("DRIVE_BASE:", DRIVE_BASE)
print("PROJECT_ROOT (will exist after Step 2 clone):", PROJECT_ROOT)

In [ ]:
assert os.path.ismount('/content/drive'), "Drive did not mount."
assert os.path.isdir(DRIVE_BASE), f"{DRIVE_BASE} was not created."
print("Step 1 validation passed: Drive mounted and base folder ready.")

### Step 2 — Clone repo into Drive and create your working branch

**Goal:** Get a persistent copy of the actual GitHub repo into Drive
(not Colab's ephemeral /content), and create a branch for your work so
you never commit directly to main.

**Why:** Colab's local filesystem resets on disconnect. Drive persists.
A separate branch protects your friend's existing work from accidental
overwrites, per Rule 13 (GitHub discipline).

**What to expect:** A cloned repo folder appears in Drive, `git status`
shows a clean working tree on your new branch.

In [ ]:
import subprocess

REPO_URL = "https://github.com/Humaira201/Country-Wellness-Score-Prediction.git"
BRANCH_NAME = "modeling-phase"  # change if you prefer a different name

os.chdir(DRIVE_BASE)

if not os.path.isdir(PROJECT_ROOT):
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
else:
    print(f"{PROJECT_ROOT} already exists — pulling latest instead of re-cloning.")
    os.chdir(PROJECT_ROOT)
    result = subprocess.run(["git", "pull"], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

os.chdir(PROJECT_ROOT)

# Create (or switch to) your working branch
branch_check = subprocess.run(["git", "rev-parse", "--verify", BRANCH_NAME],
                                capture_output=True, text=True)
if branch_check.returncode != 0:
    subprocess.run(["git", "checkout", "-b", BRANCH_NAME], check=True)
    print(f"Created and switched to new branch: {BRANCH_NAME}")
else:
    subprocess.run(["git", "checkout", BRANCH_NAME], check=True)
    print(f"Switched to existing branch: {BRANCH_NAME}")

In [ ]:
status = subprocess.run(["git", "status"], capture_output=True, text=True, cwd=PROJECT_ROOT)
branch = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True, cwd=PROJECT_ROOT)

print(status.stdout)
assert branch.stdout.strip() == BRANCH_NAME, f"Expected branch {BRANCH_NAME}, got {branch.stdout.strip()}"
assert os.path.isdir(os.path.join(PROJECT_ROOT, "data", "processed")), "data/processed missing — clone may have failed."
assert os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")), "notebooks folder missing — clone may have failed."
print(f"Step 2 validation passed: on branch '{BRANCH_NAME}', repo structure present.")

### Step 3 — Record environment and install exact dependencies

**Goal:** Know exactly what Python/package versions this project ran on,
and install anything missing. Since no requirements.txt exists in the
repo (verified by inspection), we create one now so future runs are
reproducible.

**Why:** Rule 8 requires recording versions and using deterministic
setups. Silent version drift is a common source of "it worked before"
bugs.

**What to expect:** A printed environment report, and a new
requirements.txt file written into the repo (not yet committed).

In [ ]:
import sys
import platform

REQUIRED_PACKAGES = [
    "pandas", "numpy", "scikit-learn", "joblib",
    "xgboost", "shap", "matplotlib", "seaborn",
]

print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("GPU available (torch):", torch.cuda.is_available())
except ImportError:
    print("torch not yet installed — will install if needed for LSTM phase")

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        __import__(pkg.replace("-", "_") if pkg != "scikit-learn" else "sklearn")
    except ImportError:
        missing.append(pkg)

print("Missing packages:", missing if missing else "none")

if missing:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
    print(f"Installed: {missing}")

In [ ]:
import importlib

IMPORT_NAME_MAP = {"scikit-learn": "sklearn"}

failed = []
for pkg in REQUIRED_PACKAGES:
    import_name = IMPORT_NAME_MAP.get(pkg, pkg)
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"{pkg}: OK (version {version})")
    except ImportError:
        failed.append(pkg)

assert not failed, f"Still missing after install attempt: {failed}"
print("Step 3 validation passed: all required packages importable.")

### Step 4 — Freeze dependencies into requirements.txt

**Goal:** Create the missing requirements.txt so this environment is
reproducible by anyone (including future-you).

**Why:** The repo currently has no dependency record at all — this was
a gap we found during inspection, not an assumption.


In [ ]:
import subprocess

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
all_installed = freeze.stdout.splitlines()

relevant_prefixes = tuple(p.lower() for p in REQUIRED_PACKAGES) + ("torch", "keras", "tensorflow")
relevant_lines = [line for line in all_installed
                   if line.split("==")[0].lower() in [p.lower() if p != "scikit-learn" else "scikit-learn" for p in REQUIRED_PACKAGES]
                   or line.lower().startswith(("torch==", "keras==", "tensorflow=="))]

req_path = os.path.join(PROJECT_ROOT, "requirements.txt")
with open(req_path, "w") as f:
    f.write("\n".join(sorted(relevant_lines)) + "\n")

print(f"Wrote {req_path} with {len(relevant_lines)} pinned packages:")
print("\n".join(sorted(relevant_lines)))

In [ ]:
assert os.path.exists(req_path), "requirements.txt was not created."
with open(req_path) as f:
    content = f.read()
assert "pandas" in content, "requirements.txt looks incomplete."
print("Step 4 validation passed: requirements.txt written.")

### Step 5 — Re-run the existing notebook cells and confirm output matches

**Goal:** Independently confirm, on your own Colab instance, that the
existing merge + preprocessing pipeline reproduces the same numbers
documented in data/processed/data_dictionary.md, before any new code
is layered on top.

**Why:** Rule 12 — validate existing work before building on it. The
merge script pulls from live OWID URLs, which can change over time, so
this is also an early warning check for silent data drift.

**What to expect:** Row counts and country counts matching what's
already in data/raw and data/processed. If they don't match, we treat
it as a flagged discrepancy, not a silent overwrite (Rule 12).

In [ ]:
import pandas as pd

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

existing_raw = pd.read_csv(os.path.join(RAW_DIR, "whr_worldbank_merged.csv"))
existing_train = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
existing_val = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
existing_test = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print("EXISTING (already in repo) numbers:")
print(f"  Raw merged: {len(existing_raw)} rows, {existing_raw['Country'].nunique()} countries")
print(f"  Train: {len(existing_train)} rows")
print(f"  Val:   {len(existing_val)} rows")
print(f"  Test:  {len(existing_test)} rows")

In [ ]:
# These are the numbers found during inspection of the actual repo files.
# If your output above differs, that's a real discrepancy to investigate,
# not something to silently accept.
EXPECTED = {"raw_rows": 1897, "raw_countries": 159, "train_rows": 1127, "val_rows": 133, "test_rows": 138}

actual = {
    "raw_rows": len(existing_raw),
    "raw_countries": existing_raw["Country"].nunique(),
    "train_rows": len(existing_train),
    "val_rows": len(existing_val),
    "test_rows": len(existing_test),
}

comparison = pd.DataFrame({"expected": EXPECTED, "actual": actual})
comparison["match"] = comparison["expected"] == comparison["actual"]
print(comparison)

if not comparison["match"].all():
    print("\nDISCREPANCY DETECTED — do not proceed silently. "
          "Report this table before continuing to Phase 1.")
else:
    print("\nStep 5 validation passed: your clone matches the repo's committed data exactly.")

### Step 6 — Push the repo to your own GitHub account (sharifa5852)

**Goal:** Add a new git remote pointing to your own empty repo, and
push your current branch (with all of Humaira's existing history
intact) plus your Phase 0 commit, to your account.

**Why:** `origin` still points to Humaira's repo, which you don't have
write access to. A separate remote lets you push your own copy without
needing her permission, and without losing any of her original commits
or authorship history.

**What to expect:** A token prompt (hidden input, not printed to the
notebook), then confirmation output from git push showing your branch
uploaded.

In [ ]:
remotes = subprocess.run(["git", "remote", "-v"], capture_output=True, text=True, cwd=PROJECT_ROOT)

# Strip any embedded token before printing, so it's never shown or saved to the notebook
import re
safe_output = re.sub(r"https://[^:]+:[^@]+@", "https://<redacted>@", remotes.stdout)
print("Current Remotes:\n" + safe_output)

assert "origin" in remotes.stdout, "origin remote missing."

if push_result.returncode != 0:
    print("\n--- ERROR DETECTED ---")
    print("Your push failed with the following error:")
    print(push_result.stderr)
    print("\nHow to fix this 403 Forbidden error:")
    print("1. Ensure you created an empty repo named 'Country-Wellness-Score-Prediction' at https://github.com/new under your account (sharifa5852).")
    print("2. Ensure your Personal Access Token has the 'repo' scope checked in GitHub Developer Settings.")
    print("3. Once both are confirmed, re-run cell M3Jk9d5bQmuk with the correct token.")
    raise AssertionError(f"Push failed: {push_result.stderr.strip()}")
else:
    print("Push succeeded — your repo is now live under your account.")

In [ ]:
import requests
from getpass import getpass

github_token = getpass("Paste your GitHub Personal Access Token again (input hidden, used only for this check): ")

headers = {"Authorization": f"token {github_token}"}

# 1. Which account does this token actually belong to?
user_resp = requests.get("https://api.github.com/user", headers=headers)
print("Token belongs to account:", user_resp.json().get("login", "UNKNOWN — check below"))
print("Status code:", user_resp.status_code)
if user_resp.status_code != 200:
    print("Full response:", user_resp.json())

# 2. Does the target repo exist, and what can this token do with it?
repo_resp = requests.get(
    "https://api.github.com/repos/sharifa5852/Country-Wellness-Score-Prediction",
    headers=headers
)
print("\nRepo lookup status code:", repo_resp.status_code)
if repo_resp.status_code == 200:
    repo_data = repo_resp.json()
    print("Repo exists. Full name:", repo_data.get("full_name"))
    print("Your permissions on it:", repo_data.get("permissions"))
elif repo_resp.status_code == 404:
    print("Repo NOT FOUND at this exact path — either it wasn't created, the name doesn't match exactly, or it's private and this token can't see it.")
else:
    print("Unexpected response:", repo_resp.json())

del github_token

In [ ]:
import re

remotes = subprocess.run(["git", "remote", "-v"], capture_output=True, text=True, cwd=PROJECT_ROOT)
safe_output = re.sub(r"https://[^:]+:[^@]+@", "https://<redacted>@", remotes.stdout)
print(safe_output)

assert push_result.returncode == 0, f"Push still failing: {push_result.stderr}"
print("Push confirmed successful — repo is live at your account.")

In [ ]:
# CELL 1
# Goal: Set up config and paths for Phase 1, reused from Phase 0's PROJECT_ROOT.
# Why: Centralized paths per Rule 4/9 -- every later cell derives from these.
# Expected output: confirmation that all directories exist/are created.

import os
import pandas as pd
import numpy as np

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")

for d in [RESULTS_DIR, TABLES_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("TABLES_DIR:", TABLES_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("All directories ready.")

In [ ]:
# CELL 2
# Goal: Part A -- rebuild baseline audit of the EXISTING pipeline using the
# actual committed files (not re-fetched, these are what's on GitHub right now).
# Why: Rule 12 -- validate existing work with real numbers before touching it.
# Expected output: a full audit table matching what's documented in
# data_dictionary.md -- any mismatch gets flagged, not silently accepted.

raw_old = pd.read_csv(os.path.join(RAW_DIR, "whr_worldbank_merged.csv"))
train_old = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_old = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
test_old = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

feature_cols_old = [c for c in train_old.columns
                     if c not in ("Country", "Code", "Year", "HappinessScore_target")]

baseline_audit = pd.DataFrame([{
    "raw_rows": len(raw_old),
    "raw_countries": raw_old["Country"].nunique(),
    "raw_year_min": raw_old["Year"].min(),
    "raw_year_max": raw_old["Year"].max(),
    "raw_missing_total": int(raw_old.isna().sum().sum()),
    "raw_duplicate_country_year": int(raw_old.duplicated(subset=["Country", "Year"]).sum()),
    "train_rows": len(train_old),
    "train_countries": train_old["Country"].nunique(),
    "val_rows": len(val_old),
    "val_countries": val_old["Country"].nunique(),
    "test_rows": len(test_old),
    "test_countries": test_old["Country"].nunique(),
    "test_year_min": test_old["Year"].min(),
    "test_year_max": test_old["Year"].max(),
    "feature_count": len(feature_cols_old),
    "target_nan_train": int(train_old["HappinessScore_target"].isna().sum()),
}])

print(baseline_audit.T)

In [ ]:
# CELL 3
# Goal: Validate Cell 2's numbers and SURFACE a real discrepancy found
# during inspection -- do not silently accept the README's claims.
# Why: Rule 1.8 / Rule 12 require flagging doc-vs-reality mismatches.
# Expected output: a printed WARNING if test data does not actually cover
# both 2023 and 2024 as the README implies ("test > 2022").

assert baseline_audit["raw_duplicate_country_year"].iloc[0] == 0, "Unexpected duplicate rows in raw data."
assert baseline_audit["target_nan_train"].iloc[0] == 0, "Train target should have no NaNs after preprocessing."

year_gaps = sorted(set(range(raw_old["Year"].min(), raw_old["Year"].max() + 1)) - set(raw_old["Year"].unique()))
print("Missing years entirely absent from raw data:", year_gaps if year_gaps else "none")

if baseline_audit["test_year_max"].iloc[0] < 2024:
    print("\nDISCREPANCY FLAGGED (not silently accepted):")
    print(f"README says 'test > 2022' implying 2023-2024, but test set actually "
          f"only contains year {baseline_audit['test_year_max'].iloc[0]}.")
    print("REASON (verified, not guessed): the target is built with "
          "shift(-1) per country -- 2024 rows have no 2025 to shift from, "
          "so their target is NaN and they get dropped during preprocessing. "
          "This is CORRECT forecasting behavior, not a bug -- but the README's "
          "wording overstates test coverage and should be corrected.")

baseline_audit.to_csv(os.path.join(TABLES_DIR, "baseline_audit_old_pipeline.csv"), index=False)
print("\nSaved: results/tables/baseline_audit_old_pipeline.csv")

In [ ]:
# CELL 4
# Goal: Part B -- document exact source metadata for every indicator
# currently in the pipeline, pulled from the real URLs in your existing
# merge_whr_worldbank.py-equivalent notebook cell (verified against the
# notebook content, not assumed).
# Why: required documentation before adding new sources, Rule 1.8.
# Expected output: data_source_audit.csv with one row per existing indicator.

data_source_audit = pd.DataFrame([
    {
        "indicator_name": "HappinessScore",
        "source_name": "World Happiness Report (Cantril Ladder)",
        "retrieval_method": "OWID grapher CSV export",
        "source_url": "https://ourworldindata.org/grapher/happiness-cantril-ladder.csv",
        "unit": "0-10 self-reported life evaluation score",
        "definition": "Average answer to the Cantril Ladder question ('best possible life' = 10)",
        "year_availability_in_pipeline": "2011-2024 (2013 missing)",
        "country_coverage_in_pipeline": 159,
        "known_missingness": "Not every country reports every year; aggregate rows (OWID_*, WB_*) excluded by design",
    },
    {
        "indicator_name": "GDPPerCapita_WB",
        "source_name": "World Bank, via OWID",
        "retrieval_method": "OWID grapher CSV export",
        "source_url": "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv",
        "unit": "Constant international $ (PPP-adjusted)",
        "definition": "GDP per capita, World Bank national accounts data",
        "year_availability_in_pipeline": "2011-2024 (2013 missing)",
        "country_coverage_in_pipeline": 159,
        "known_missingness": "Same aggregate-row exclusion as above",
    },
])

data_source_audit.to_csv(os.path.join(TABLES_DIR, "data_source_audit.csv"), index=False)
print(data_source_audit.to_string(index=False))
print("\nSaved: results/tables/data_source_audit.csv")

In [ ]:
# CELL 5
# Goal: Part C -- define candidate World Bank/OWID indicators to evaluate.
# Why: these candidates come from your original plan's list (Section 4) --
# unemployment, internet access, health expenditure, education expenditure,
# inequality. URLs follow the EXACT same OWID grapher pattern your friend
# already used successfully for GDP.
# Expected output: a dict of candidate indicators, printed for review.
#
# IMPORTANT: these URL slugs are my best-effort guess based on OWID's naming
# conventions -- verify each one resolves in a browser BEFORE running Cell 6.
# If any 404s, find the correct slug at ourworldindata.org (search the
# indicator name, open the chart, the URL slug is in the address bar) and
# update this dict. Do not silently skip a broken URL without noting why.

CANDIDATE_INDICATORS = {
    "UnemploymentRate_WB": "https://ourworldindata.org/grapher/unemployment-rate.csv?v=1&csvType=full&useColumnShortNames=true",
    "InternetAccessPct_WB": "https://ourworldindata.org/grapher/share-of-individuals-using-the-internet.csv?v=1&csvType=full&useColumnShortNames=true",
    "HealthExpenditurePctGDP_WB": "https://ourworldindata.org/grapher/public-health-expenditure-share-of-gdp.csv?v=1&csvType=full&useColumnShortNames=true",
    "EducationExpenditurePctGDP_WB": "https://ourworldindata.org/grapher/total-government-expenditure-on-education-gdp.csv?v=1&csvType=full&useColumnShortNames=true",
    "GiniIndex_WB": "https://ourworldindata.org/grapher/economic-inequality-gini-index.csv?v=1&csvType=full&useColumnShortNames=true",
}

for name, url in CANDIDATE_INDICATORS.items():
    print(f"{name}: {url}")

In [ ]:
# CELL 6
# Goal: Part C -- fetch each candidate, reusing the SAME load_and_clean()
# pattern your friend already used for GDP, then compute every required
# vetting metric against the EXISTING merged dataset (raw_old).
# Why: Rule 11 -- empirically evaluate before deciding, no arbitrary adds.
# Expected output: feature_vetting_results.csv with one row per candidate.
# If a URL fails, the row records the failure instead of crashing the loop.

from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

AGGREGATE_PREFIXES = ("OWID_", "WB_")

def load_and_clean(url, new_name):
    df = pd.read_csv(url, storage_options={"User-Agent": "Mozilla/5.0 (research data fetch)"})
    df.columns = [c.strip() for c in df.columns]
    country_col, code_col, year_col = df.columns[0], df.columns[1], df.columns[2]
    value_col = df.columns[3]
    df = df[[country_col, code_col, year_col, value_col]]
    df.columns = ["Country", "Code", "Year", new_name]
    df = df[~df["Code"].astype(str).str.startswith(AGGREGATE_PREFIXES)]
    df = df.dropna(subset=["Code"])
    return df

vetting_rows = []
existing_features_merged = raw_old.copy()  # baseline: HappinessScore + GDPPerCapita_WB

for feature_name, url in CANDIDATE_INDICATORS.items():
    row = {"feature": feature_name, "source": "OWID/World Bank", "url": url}
    try:
        candidate_df = load_and_clean(url, feature_name)
        merged_check = pd.merge(raw_old, candidate_df[["Code", "Year", feature_name]],
                                 on=["Code", "Year"], how="left")

        total_possible = len(raw_old)
        non_missing = merged_check[feature_name].notna().sum()
        coverage_pct = round(100 * non_missing / total_possible, 2)
        missing_rate = round(100 - coverage_pct, 2)

        valid_vals = merged_check[feature_name].dropna()
        skewness = round(valid_vals.skew(), 3) if len(valid_vals) > 2 else np.nan

        corr_with_target = round(
            merged_check[["HappinessScore", feature_name]].dropna().corr().iloc[0, 1], 3
        ) if non_missing > 2 else np.nan

        existing_feature_corrs = merged_check[["GDPPerCapita_WB", feature_name]].dropna()
        max_feature_corr = round(existing_feature_corrs.corr().iloc[0, 1], 3) if len(existing_feature_corrs) > 2 else np.nan

        vif_value = np.nan
        vif_input = merged_check[["GDPPerCapita_WB", feature_name]].dropna()
        if len(vif_input) > 10:
            vif_value = round(variance_inflation_factor(vif_input.values, 1), 3)

        decision = "REVIEW"
        reason = "Automatic rule-of-thumb only -- confirm manually before finalizing"
        if coverage_pct < 60:
            decision, reason = "DROP", f"Coverage too low ({coverage_pct}%) -- would shrink usable dataset significantly"
        elif abs(max_feature_corr) > 0.85:
            decision, reason = "DROP", f"Near-duplicate of existing feature (corr={max_feature_corr})"
        elif vif_value > 10:
            decision, reason = "REVIEW", f"High multicollinearity (VIF={vif_value})"
        elif coverage_pct >= 60:
            decision, reason = "KEEP-CANDIDATE", f"Coverage {coverage_pct}% acceptable, low redundancy"

        row.update({
            "coverage_pct": coverage_pct, "missing_rate_pct": missing_rate,
            "skewness": skewness, "target_correlation": corr_with_target,
            "max_feature_correlation": max_feature_corr, "VIF": vif_value,
            "decision": decision, "reason": reason, "fetch_status": "OK",
        })
    except Exception as e:
        row.update({"fetch_status": "FAILED", "error": str(e), "decision": "ERROR",
                     "reason": "Fetch failed -- verify URL slug on ourworldindata.org"})
    vetting_rows.append(row)

feature_vetting_results = pd.DataFrame(vetting_rows)
print(feature_vetting_results.to_string(index=False))

In [ ]:
# CELL 7
# Goal: Save Part C's real output and validate no silent failures were missed.
# Why: Rule 9 -- outputs go in results/tables/, not scattered in /content.
# Expected output: confirmation file saved; any FAILED fetch explicitly listed.

feature_vetting_results.to_csv(os.path.join(TABLES_DIR, "feature_vetting_results.csv"), index=False)

failed = feature_vetting_results[feature_vetting_results.get("fetch_status", pd.Series(dtype=str)) == "FAILED"]
if len(failed) > 0:
    print("FAILED fetches -- these need a corrected URL before Part D can proceed:")
    print(failed[["feature", "url", "error"]].to_string(index=False))
else:
    print("All candidate indicators fetched successfully.")

print("\nSaved: results/tables/feature_vetting_results.csv")
assert os.path.exists(os.path.join(TABLES_DIR, "feature_vetting_results.csv"))

In [ ]:
# CELL 8
# Goal: Re-fetch the one failed candidate with the corrected, verified URL.
# Why: Rule 7 -- when something fails, fix root cause and resume exactly
# where it broke, don't redo everything.
# Expected output: one new row appended to feature_vetting_results with
# real computed metrics, fetch_status = OK.

CANDIDATE_INDICATORS["HealthExpenditurePctGDP_WB"] = (
    "https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv"
    "?v=1&csvType=full&useColumnShortNames=true"
)

feature_name = "HealthExpenditurePctGDP_WB"
url = CANDIDATE_INDICATORS[feature_name]
row = {"feature": feature_name, "source": "OWID/World Bank (WHO GHO)", "url": url}

try:
    candidate_df = load_and_clean(url, feature_name)
    merged_check = pd.merge(raw_old, candidate_df[["Code", "Year", feature_name]],
                             on=["Code", "Year"], how="left")
    total_possible = len(raw_old)
    non_missing = merged_check[feature_name].notna().sum()
    coverage_pct = round(100 * non_missing / total_possible, 2)
    missing_rate = round(100 - coverage_pct, 2)
    valid_vals = merged_check[feature_name].dropna()
    skewness = round(valid_vals.skew(), 3) if len(valid_vals) > 2 else np.nan
    corr_with_target = round(merged_check[["HappinessScore", feature_name]].dropna().corr().iloc[0, 1], 3)
    max_feature_corr = round(merged_check[["GDPPerCapita_WB", feature_name]].dropna().corr().iloc[0, 1], 3)
    vif_input = merged_check[["GDPPerCapita_WB", feature_name]].dropna()
    vif_value = round(variance_inflation_factor(vif_input.values, 1), 3) if len(vif_input) > 10 else np.nan

    decision = "DROP" if coverage_pct < 60 else ("DROP" if abs(max_feature_corr) > 0.85 else "KEEP-CANDIDATE")
    reason = f"Coverage {coverage_pct}%, max_corr {max_feature_corr}"

    row.update({"coverage_pct": coverage_pct, "missing_rate_pct": missing_rate,
                "skewness": skewness, "target_correlation": corr_with_target,
                "max_feature_correlation": max_feature_corr, "VIF": vif_value,
                "decision": decision, "reason": reason, "fetch_status": "OK"})
except Exception as e:
    row.update({"fetch_status": "FAILED", "error": str(e), "decision": "ERROR"})

feature_vetting_results = feature_vetting_results[feature_vetting_results["feature"] != "HealthExpenditurePctGDP_WB"]
feature_vetting_results = pd.concat([feature_vetting_results, pd.DataFrame([row])], ignore_index=True)
feature_vetting_results.to_csv(os.path.join(TABLES_DIR, "feature_vetting_results.csv"), index=False)
print(feature_vetting_results.to_string(index=False))

In [ ]:
# CELL 9
# Goal: Part D -- merge all 4 shortlisted candidates into the existing raw
# dataset with a LEFT join (not inner), so we can see exactly how much data
# each feature is missing without silently losing rows yet.
# Why: Rule Part D -- explicitly test coverage impact before committing to
# an inner merge that could silently shrink the dataset.
# Expected output: one combined dataframe, missingness per new column.

FINAL_CANDIDATES = ["UnemploymentRate_WB", "InternetAccessPct_WB",
                     "EducationExpenditurePctGDP_WB", "HealthExpenditurePctGDP_WB"]

enriched_check = raw_old.copy()
for feature_name in FINAL_CANDIDATES:
    url = CANDIDATE_INDICATORS[feature_name]
    candidate_df = load_and_clean(url, feature_name)
    enriched_check = pd.merge(enriched_check, candidate_df[["Code", "Year", feature_name]],
                               on=["Code", "Year"], how="left")

print("Enriched shape (LEFT merge, no rows dropped yet):", enriched_check.shape)
print("\nMissing count per new feature:")
print(enriched_check[FINAL_CANDIDATES].isna().sum())
print("\nRows with ALL 4 new features present:", enriched_check[FINAL_CANDIDATES].notna().all(axis=1).sum())
print("Rows missing AT LEAST ONE new feature:", enriched_check[FINAL_CANDIDATES].isna().any(axis=1).sum())

In [ ]:
# CELL 10
# Goal: Part D -- coverage safety check SPECIFICALLY within the existing
# train/val/test year windows, not just pooled across all years.
# Why: a feature can look fine overall but be disproportionately missing
# in the exact year that matters most (your test year, 2023).
# Expected output: a coverage-by-split table -- this is the real test of
# whether enrichment is safe.

SPLIT_YEARS = {
    "train (<=2021)": enriched_check["Year"] <= 2021,
    "val (2022)": enriched_check["Year"] == 2022,
    "test (2023)": enriched_check["Year"] == 2023,
}

coverage_by_split = []
for split_name, mask in SPLIT_YEARS.items():
    split_df = enriched_check[mask]
    row = {"split": split_name, "total_rows": len(split_df),
           "countries": split_df["Country"].nunique()}
    for feature_name in FINAL_CANDIDATES:
        coverage = round(100 * split_df[feature_name].notna().sum() / len(split_df), 2) if len(split_df) > 0 else np.nan
        row[f"{feature_name}_coverage_pct"] = coverage
    coverage_by_split.append(row)

coverage_by_split_df = pd.DataFrame(coverage_by_split)
print(coverage_by_split_df.T)

coverage_by_split_df.to_csv(os.path.join(TABLES_DIR, "coverage_by_split.csv"), index=False)
print("\nSaved: results/tables/coverage_by_split.csv")
print("\nCHECK: if any feature's coverage in the TEST row is notably lower than")
print("its coverage in TRAIN, that feature will be weaker at exactly the years")
print("your forecast is actually evaluated on -- flag it, don't ignore it.")

In [ ]:
# CELL 11
# Goal: Part D -- TRUE multivariate VIF across all final candidates +
# existing GDP together, not pairwise. This is what actually determines
# multicollinearity once all features sit in the same model.
# Why: a pairwise VIF under 10 does not guarantee the full-model VIF does.
# Expected output: one VIF value per feature, computed jointly.

vif_feature_set = ["GDPPerCapita_WB"] + FINAL_CANDIDATES
vif_data = enriched_check[vif_feature_set].dropna()  # complete-case for this check only

print(f"Rows available for joint VIF check (complete cases): {len(vif_data)}")

joint_vif = pd.DataFrame({
    "feature": vif_feature_set,
    "VIF": [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_feature_set))]
})
joint_vif["VIF"] = joint_vif["VIF"].round(3)
print(joint_vif)

print("\nRule of thumb: VIF > 10 is a real multicollinearity concern, 5-10 is worth watching.")
joint_vif.to_csv(os.path.join(TABLES_DIR, "joint_vif.csv"), index=False)

In [ ]:
# CELL 12
# Goal: Part D -- full pairwise correlation matrix across ALL features
# (existing + candidates), not just each candidate vs GDP individually.
# Why: catches candidate-vs-candidate redundancy (e.g. Internet Access vs
# Education Expenditure) that the earlier one-at-a-time checks could not see.
# Expected output: correlation matrix, saved as both CSV and a figure.

import matplotlib.pyplot as plt
import seaborn as sns

corr_feature_set = ["HappinessScore", "GDPPerCapita_WB"] + FINAL_CANDIDATES
full_corr_matrix = enriched_check[corr_feature_set].corr().round(3)
print(full_corr_matrix)

full_corr_matrix.to_csv(os.path.join(TABLES_DIR, "full_correlation_matrix.csv"))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(full_corr_matrix, annot=True, cmap="coolwarm", center=0, ax=ax,
            vmin=-1, vmax=1, square=True)
ax.set_title("Correlation Matrix -- Existing + Candidate Features", fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "full_correlation_heatmap.png"), dpi=150)
plt.show()
print("\nSaved figure: results/figures/full_correlation_heatmap.png")

In [ ]:
# CELL 13
# Goal: Part D final checkpoint -- decide merge strategy based on real
# numbers from Cells 9-12, not assumption. Compare row/country loss under
# an INNER merge (require all 4 features present) vs keeping the current
# LEFT-merge + imputation approach.
# Why: Part D explicitly requires this before Part E (actual merge).
# Expected output: a before/after coverage comparison table.

inner_merge_result = enriched_check.dropna(subset=FINAL_CANDIDATES)

coverage_comparison = pd.DataFrame([
    {"stage": "before enrichment (existing pipeline)",
     "rows": len(raw_old), "countries": raw_old["Country"].nunique()},
    {"stage": "after enrichment, LEFT merge (features may have NaN)",
     "rows": len(enriched_check), "countries": enriched_check["Country"].nunique()},
    {"stage": "after enrichment, INNER merge (all 4 features required)",
     "rows": len(inner_merge_result), "countries": inner_merge_result["Country"].nunique()},
])
print(coverage_comparison)

countries_lost_if_inner = set(raw_old["Country"].unique()) - set(inner_merge_result["Country"].unique())
print(f"\nCountries that would be LOST ENTIRELY under an inner merge: {len(countries_lost_if_inner)}")
print(sorted(countries_lost_if_inner) if countries_lost_if_inner else "none")

coverage_comparison.to_csv(os.path.join(TABLES_DIR, "coverage_comparison_before_after.csv"), index=False)
print("\nSaved: results/tables/coverage_comparison_before_after.csv")

In [ ]:
# ==================================================
# CELL 14 — FINAL ENRICHED DATASET (LEFT MERGE, 3 FEATURES)
#
# GOAL:
#   Download the 3 approved indicators (unemployment, internet access,
#   health expenditure) and LEFT-merge them onto the untouched raw
#   WHR+GDP file. Save as a NEW file; the original raw file is not modified.
#
# WHY:
#   Cells 9-13 showed LEFT keeps all 1,897 rows / 159 countries, while
#   INNER would delete 11 countries. Education (test-year coverage 56%)
#   and Gini (48.5%) are excluded.
#
# WHAT HAPPENS:
#   1) read raw file  2) download each indicator, keep real countries only
#   3) merge on [Code, Year]  4) save whr_worldbank_enriched.csv
#   5) save a provenance table (URL, download time, coverage)
#
# EXPECTED OUTPUT:
#   shape (1897, 8), 159 countries, missing counts 6 / 24 / 159,
#   then "CELL 14 PASSED".
#
# POSSIBLE FAILURES: see the table under this cell.
# ==================================================
import os, time, datetime
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
assert os.path.isdir(PROJECT_ROOT), (
    f"{PROJECT_ROOT} not found. Run the Step 1 Drive-mount cell, then re-run this cell.")

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
os.makedirs(TABLES_DIR, exist_ok=True)

RAW_PATH = os.path.join(RAW_DIR, "whr_worldbank_merged.csv")          # original, never modified
ENRICHED_PATH = os.path.join(RAW_DIR, "whr_worldbank_enriched.csv")   # new file
PROVENANCE_PATH = os.path.join(TABLES_DIR, "data_source_audit_new_features.csv")
OVERWRITE = False          # False = never silently replace an existing enriched file
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Self-contained URL list (does NOT depend on CELL 5, which still holds the 404 health URL)
FINAL_FEATURES = {
    "UnemploymentRate_WB":
        "https://ourworldindata.org/grapher/unemployment-rate.csv?v=1&csvType=full&useColumnShortNames=true",
    "InternetAccessPct_WB":
        "https://ourworldindata.org/grapher/share-of-individuals-using-the-internet.csv?v=1&csvType=full&useColumnShortNames=true",
    "HealthExpenditurePctGDP_WB":
        "https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true",
}
ORIGINAL_COLS = ["Country", "Code", "Year", "HappinessScore", "GDPPerCapita_WB"]


def fetch_owid_indicator(url, new_name, retries=3, wait=5):
    """Download one OWID indicator and return (clean_df, name_of_source_value_column)."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            df = pd.read_csv(url, storage_options={"User-Agent": "Mozilla/5.0 (research data fetch)"})
            break
        except Exception as e:                       # network / HTTP errors
            last_err = e
            print(f"  attempt {attempt}/{retries} failed for {new_name}: {e}")
            time.sleep(wait)
    else:                                            # runs only if the loop never hit 'break'
        raise RuntimeError(f"Could not download {new_name}. URL: {url}. Last error: {last_err}")

    df.columns = [c.strip() for c in df.columns]
    assert [c.lower() for c in df.columns[:3]] == ["entity", "code", "year"], \
        f"Unexpected leading columns for {new_name}: {list(df.columns)}"
    value_cols = [c for c in df.columns[3:] if "annotation" not in c.lower()]
    assert len(value_cols) == 1, f"Expected exactly 1 value column for {new_name}, got {value_cols}"

    out = df[[df.columns[0], df.columns[1], df.columns[2], value_cols[0]]].copy()
    out.columns = ["Country", "Code", "Year", new_name]
    out = out.dropna(subset=["Code"])                                       # drop unnamed regions
    out = out[~out["Code"].astype(str).str.startswith(("OWID_", "WB_"))]    # drop aggregates
    out["Year"] = pd.to_numeric(out["Year"], errors="raise").astype("int64")
    out[new_name] = pd.to_numeric(out[new_name], errors="coerce")
    assert not out.duplicated(["Code", "Year"]).any(), f"Duplicate Code-Year rows in {new_name}"
    assert len(out) > 0, f"{new_name}: empty after cleaning"
    return out, value_cols[0]


raw = pd.read_csv(RAW_PATH)
assert set(raw.columns) == set(ORIGINAL_COLS), f"Raw columns differ from expectation: {list(raw.columns)}"
assert not raw.duplicated(["Code", "Year"]).any(), "Raw file has duplicate Code-Year rows"
print("Raw file:", raw.shape)

enriched = raw.copy()
provenance = []
for name, url in FINAL_FEATURES.items():
    print(f"Downloading {name} ...")
    cand, src_col = fetch_owid_indicator(url, name)
    n_before = len(enriched)
    enriched = enriched.merge(cand[["Code", "Year", name]], on=["Code", "Year"],
                              how="left", validate="one_to_one")
    assert len(enriched) == n_before, f"LEFT merge changed the row count for {name}"
    matched = int(enriched[name].notna().sum())
    provenance.append({
        "feature": name, "source_url": url, "owid_value_column": src_col,
        "downloaded_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "owid_rows_after_cleaning": len(cand), "owid_countries": cand["Code"].nunique(),
        "owid_year_min": int(cand["Year"].min()), "owid_year_max": int(cand["Year"].max()),
        "project_rows_matched": matched,
        "project_coverage_pct": round(100 * matched / len(enriched), 2),
    })

enriched = enriched.sort_values(["Code", "Year"]).reset_index(drop=True)
pd.DataFrame(provenance).to_csv(PROVENANCE_PATH, index=False)

# Save without silently overwriting
if os.path.exists(ENRICHED_PATH) and not OVERWRITE:
    existing = pd.read_csv(ENRICHED_PATH)
    num_cols = [c for c in enriched.columns if c not in ("Country", "Code")]
    same = (list(existing.columns) == list(enriched.columns) and existing.shape == enriched.shape
            and np.allclose(existing[num_cols].to_numpy(float), enriched[num_cols].to_numpy(float),
                            equal_nan=True))
    if same:
        print("Enriched file already exists and is IDENTICAL to a fresh download. Kept as is.")
    else:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fresh_path = ENRICHED_PATH.replace(".csv", f"_FRESH_{stamp}.csv")
        enriched.to_csv(fresh_path, index=False)
        print("WARNING: existing enriched file DIFFERS from a fresh download (source data may have been revised).")
        print("Existing file kept; fresh copy saved as:", fresh_path)
else:
    enriched.to_csv(ENRICHED_PATH, index=False)
    print("Saved:", ENRICHED_PATH)

# VALIDATION
final = pd.read_csv(ENRICHED_PATH)
print("\nShape:", final.shape, "| countries:", final["Country"].nunique())
print("Columns:", list(final.columns))
print("\nMissing per new feature:")
print(final[list(FINAL_FEATURES)].isna().sum())
print(pd.DataFrame(provenance)[["feature", "owid_value_column", "project_coverage_pct"]])

assert final.shape == (1897, 8), f"Expected (1897, 8), got {final.shape}"
assert final["Country"].nunique() == 159, "Country count changed"
assert not final.duplicated(["Code", "Year"]).any(), "Duplicate Code-Year rows"
assert all(f in final.columns and final[f].notna().any() for f in FINAL_FEATURES), "A new feature is empty"
print("\nCELL 14 PASSED")

In [ ]:
# ==================================================
# CELL 15 — VALIDATE ENRICHED DATASET + COVERAGE TABLES
#
# GOAL:
#   Run an automated validation function on the enriched file, recompute
#   coverage by year and by split, list countries with a feature entirely
#   missing, save the final feature decisions, and check Phase 1 files exist.
#
# WHY:
#   We do not build lags on top of a dataset that has not passed validation.
#   Original columns must be bit-for-bit unchanged; new columns must be sane.
#
# EXPECTED OUTPUT:
#   "problems: none", at most soft warnings, coverage tables, "CELL 15 PASSED".
#
# POSSIBLE FAILURES: see the table under this cell.
# ==================================================
import os
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
RAW_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "whr_worldbank_merged.csv")
ENRICHED_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "whr_worldbank_enriched.csv")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")

ORIGINAL_COLS = ["Country", "Code", "Year", "HappinessScore", "GDPPerCapita_WB"]
NEW_FEATURES = ["UnemploymentRate_WB", "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB"]
EXPECTED_MISSING = {"UnemploymentRate_WB": 6, "InternetAccessPct_WB": 24, "HealthExpenditurePctGDP_WB": 159}  # from CELL 9


def validate_enriched(enr, raw):
    """Return (problems, warnings). 'problems' must be empty to continue."""
    problems, warnings = [], []
    if set(enr.columns) != set(ORIGINAL_COLS + NEW_FEATURES):
        problems.append(f"Unexpected columns: {list(enr.columns)}")
    if len(enr) != len(raw):
        problems.append(f"Row count {len(enr)} != raw {len(raw)}")
    if enr.duplicated(["Code", "Year"]).any():
        problems.append("Duplicate Code-Year rows")
    if enr[["Code", "Year"]].isna().any().any():
        problems.append("Missing Code or Year")
    if not problems:
        m = raw.merge(enr[ORIGINAL_COLS], on=["Code", "Year"], suffixes=("_raw", "_enr"), validate="one_to_one")
        if len(m) != len(raw):
            problems.append("Some raw rows are missing from the enriched file")
        else:
            for c in ["HappinessScore", "GDPPerCapita_WB"]:
                if not np.allclose(m[c + "_raw"], m[c + "_enr"], equal_nan=True):
                    problems.append(f"Original column {c} was altered")
            if (m["Country_raw"] != m["Country_enr"]).any():
                problems.append("Country names were altered")
    for f in NEW_FEATURES:
        v = enr[f].dropna()
        if v.empty:
            problems.append(f"{f} entirely missing")
        elif (v < 0).any() or (v > 100).any():
            problems.append(f"{f} has values outside 0-100")
        if np.isinf(enr[f].to_numpy(float)).any():
            problems.append(f"{f} has infinite values")
    if 2013 in set(enr["Year"]):
        warnings.append("Year 2013 is present (it was absent before).")
    for f, n in EXPECTED_MISSING.items():
        if enr[f].isna().sum() != n:
            warnings.append(f"{f}: missing {enr[f].isna().sum()} vs {n} seen in CELL 9 (source drift?)")
    if enr["Country"].nunique() != 159:
        warnings.append(f"Countries = {enr['Country'].nunique()} (expected 159)")
    return problems, warnings


raw = pd.read_csv(RAW_PATH)
enr = pd.read_csv(ENRICHED_PATH)
problems, warnings = validate_enriched(enr, raw)
print("problems:", problems if problems else "none")
print("warnings:", warnings if warnings else "none")

# Coverage by calendar year (feature year)
cov_year = enr.groupby("Year")[NEW_FEATURES].agg(lambda s: round(100 * s.notna().mean(), 2))
cov_year.insert(0, "rows", enr.groupby("Year").size())
print("\nCoverage % by year:\n", cov_year)
cov_year.to_csv(os.path.join(TABLES_DIR, "coverage_by_year_enriched.csv"))

# Coverage by split (same windows as CELL 10)
splits = {"train (<=2021)": enr["Year"] <= 2021, "val (2022)": enr["Year"] == 2022, "test (2023)": enr["Year"] == 2023}
rows = []
for name, mask in splits.items():
    d = enr[mask]
    r = {"split": name, "rows": len(d), "countries": d["Country"].nunique()}
    for f in NEW_FEATURES:
        r[f + "_coverage_pct"] = round(100 * d[f].notna().mean(), 2)
    rows.append(r)
cov_split = pd.DataFrame(rows)
print("\nCoverage by split:\n", cov_split.T)
cov_split.to_csv(os.path.join(TABLES_DIR, "coverage_by_split_enriched.csv"), index=False)

# Countries with a feature missing in EVERY year (drives the missing-value policy in CELL 19)
print("\nCountries with a feature missing in every year:")
for f in NEW_FEATURES:
    all_nan = enr.groupby("Country")[f].apply(lambda s: s.isna().all())
    names = sorted(all_nan[all_nan].index)
    print(f"  {f}: {len(names)} countries", names if len(names) <= 15 else "(list too long, see saved table)")

# Final feature decisions (numbers from CELLS 6-11 and 10)
decisions = pd.DataFrame([
    ["UnemploymentRate_WB", 99.68, 98.56, "KEEP", "High coverage, low redundancy (joint VIF 1.099)"],
    ["InternetAccessPct_WB", 98.73, 97.12, "KEEP", "High coverage; corr with GDP 0.722 but joint VIF 2.430"],
    ["HealthExpenditurePctGDP_WB", 91.62, 98.56, "KEEP", "Coverage ok in all splits (joint VIF 1.530)"],
    ["EducationExpenditurePctGDP_WB", 77.33, 56.12, "DROP (core)", "Test-year coverage 56.12%: weak exactly where we evaluate"],
    ["GiniIndex_WB", 48.50, np.nan, "DROP", "Coverage 48.5% < 60% threshold"],
], columns=["feature", "overall_coverage_pct", "test_year_coverage_pct", "final_decision", "reason"])
decisions.to_csv(os.path.join(TABLES_DIR, "feature_decisions_final.csv"), index=False)

# Do the Phase 1 outputs listed in your handoff exist in Drive?
expected_files = [os.path.join(TABLES_DIR, f) for f in [
    "baseline_audit_old_pipeline.csv", "data_source_audit.csv", "feature_vetting_results.csv",
    "coverage_by_split.csv", "joint_vif.csv", "full_correlation_matrix.csv",
    "coverage_comparison_before_after.csv"]] + [os.path.join(FIGURES_DIR, "full_correlation_heatmap.png")]
missing_files = [os.path.basename(p) for p in expected_files if not os.path.exists(p)]
print("\nPhase 1 files missing from Drive:", missing_files if missing_files else "none")

assert not problems, f"Validation failed: {problems}"
print("\nCELL 15 PASSED")

In [ ]:
# ==================================================
# CELL 16 — GIT SECURITY CLEANUP + COMMIT IDENTITY
#
# GOAL:
#   Remove any credentials embedded in remote URLs, set a real commit
#   identity (repo-local, so it survives runtime resets), and scan
#   .git/config for token-shaped strings.
#
# WHY:
#   Your 'origin' URL contains a token inside .git/config on Google Drive.
#   Tokens must never live in the remote URL, notebook, or history.
#
# EXPECTED OUTPUT:
#   "SCRUBBED ..." lines if anything was embedded, then clean redacted
#   remote list, identity, branch, short status, "CELL 16 PASSED".
#   This cell asks for NO token and uses NO network.
#
# POSSIBLE FAILURES: see the table under this cell.
# ==================================================
import os, re, subprocess

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
BRANCH_NAME = "modeling-phase"
EXPECTED_ORIGIN = "https://github.com/sharifa5852/Country-Wellness-Score-Prediction.git"
EXPECTED_UPSTREAM = "https://github.com/Humaira201/Country-Wellness-Score-Prediction.git"
NOREPLY_EMAIL = "144588197+sharifa5852@users.noreply.github.com"   # GitHub no-reply format: <id>+<username>@users.noreply.github.com
URL_KEYS = r"^remote\..*\.(url|pushurl)$"


def git(*args):
    """Run a git command inside the project folder and capture its text output."""
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True)


# A) Tell Git this Drive-mounted folder is safe (prevents 'dubious ownership' errors)
if PROJECT_ROOT not in git("config", "--global", "--get-all", "safe.directory").stdout.splitlines():
    git("config", "--global", "--add", "safe.directory", PROJECT_ROOT)

# B) Ensure 'upstream' is set to the correct original read-only repo URL if misconfigured
if "upstream" not in git("remote").stdout.splitlines():
    git("remote", "add", "upstream", EXPECTED_UPSTREAM)
else:
    current_upstream = git("remote", "get-url", "upstream").stdout.strip()
    if current_upstream != EXPECTED_UPSTREAM:
        git("remote", "set-url", "upstream", EXPECTED_UPSTREAM)
        print("CORRECTED 'upstream' remote URL to point to original repository.")

# C) Remove user:token@ from every remote url / pushurl
scrubbed = 0
for line in git("config", "--local", "--get-regexp", URL_KEYS).stdout.splitlines():
    key, _, value = line.partition(" ")
    clean = re.sub(r"^(https?://)[^/@]+@", r"\1", value)
    if clean != value:
        git("config", "--local", key, clean)
        scrubbed += 1
        print(f"SCRUBBED credentials from {key}")
print("Entries scrubbed:", scrubbed)

# D) Show the remotes (now clean)
print("\nRemotes:\n" + git("remote", "-v").stdout)

# E) Commit identity (repo-local) if missing or still the placeholder
cur_email = git("config", "--get", "user.email").stdout.strip()
if cur_email in ("", "you@example.com"):
    git("config", "--local", "user.email", NOREPLY_EMAIL)
    git("config", "--local", "user.name", "sharifa5852")
print("Commit identity:", git("config", "--get", "user.name").stdout.strip(),
      "<" + git("config", "--get", "user.email").stdout.strip() + ">")

# F) Scan .git/config for token-shaped strings (matches are never printed)
cfg_text = open(os.path.join(PROJECT_ROOT, ".git", "config")).read()
token_hits = re.findall(r"gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}", cfg_text)
print("Token-shaped strings in .git/config:", len(token_hits))
print("~/.git-credentials exists:", os.path.exists(os.path.expanduser("~/.git-credentials")))

# G) Branch and working-tree state
print("\nBranch:", git("branch", "--show-current").stdout.strip())
print("Status (short):\n" + (git("status", "--short").stdout or "  clean\n"))
print("Last commits (author check):\n" + git("log", "-3", "--format=%h %an <%ae> %s").stdout)

# VALIDATION
cfg_after = git("config", "--local", "--get-regexp", URL_KEYS).stdout
assert not re.search(r"https?://[^/\s@]+@", cfg_after), "Credentials are STILL embedded in a remote URL"
assert len(token_hits) == 0, "A token-shaped string is still in .git/config"
assert git("remote", "get-url", "origin").stdout.strip() == EXPECTED_ORIGIN, "origin URL is not the expected clean URL"
assert git("remote", "get-url", "upstream").stdout.strip() == EXPECTED_UPSTREAM, "upstream URL unexpected"
assert git("branch", "--show-current").stdout.strip() == BRANCH_NAME, f"Not on {BRANCH_NAME}"
print("\nCELL 16 PASSED")

In [ ]:
# ==================================================
# CELL 17 — TEMPORAL ALIGNMENT AUDIT (is the old shift() logic wrong?)
#
# GOAL:
#   (1) Measure how often row-based shift(1/2/-1) fails to equal the real
#       calendar year -1 / -2 / +1.
#   (2) List countries with year gaps besides the global missing year.
#   (3) Count how many supervised samples exist under each candidate rule:
#       STRICT  : needs t, t-1, t-2 and t+1 as real observed years
#       STRICT4 : STRICT plus t-3 (sequence length 4 for the LSTM)
#       GRIDFILL: a globally missing year (2013) may be filled ONLY as a
#                 flagged lag feature, using observations <= t, never as a target
#
# WHY:
#   The strategy decision (CELL 18) must rest on counts from your data.
#
# EXPECTED OUTPUT:
#   Misaligned counts concentrated at a few origin years, tables per origin
#   year and per split, "CELL 17 PASSED". Nothing here changes any dataset.
#
# POSSIBLE FAILURES: see the table under this cell.
# ==================================================
import os
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
ENRICHED_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "whr_worldbank_enriched.csv")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")

df = pd.read_csv(ENRICHED_PATH).sort_values(["Code", "Year"]).reset_index(drop=True)
present = set(zip(df["Code"], df["Year"]))
GLOBAL_MISSING = set(range(int(df["Year"].min()), int(df["Year"].max()) + 1)) - set(df["Year"].unique())
print("Years absent for EVERY country:", sorted(GLOBAL_MISSING))

# ---- Part 1: what row-based shifting really did -------------------------------
g = df.groupby("Code")["Year"]
df["target_gap"] = g.shift(-1) - df["Year"]      # calendar years to the "next row"; must be 1
df["lag1_gap"] = df["Year"] - g.shift(1)         # must be 1
df["lag2_gap"] = df["Year"] - g.shift(2)         # must be 2
df["bad_target"] = df["target_gap"].notna() & (df["target_gap"] != 1)
df["bad_lag1"] = df["lag1_gap"].notna() & (df["lag1_gap"] != 1)
df["bad_lag2"] = df["lag2_gap"].notna() & (df["lag2_gap"] != 2)

shift_audit = df.groupby("Year").agg(rows=("Code", "size"), misaligned_target=("bad_target", "sum"),
                                     misaligned_lag1=("bad_lag1", "sum"),
                                     misaligned_lag2=("bad_lag2", "sum"))
print("\nRow-shift misalignment by origin year (non-zero rows only):")
print(shift_audit[(shift_audit[["misaligned_target", "misaligned_lag1", "misaligned_lag2"]] > 0).any(axis=1)])
shift_audit.to_csv(os.path.join(TABLES_DIR, "temporal_alignment_audit_by_year.csv"))

# ---- Part 2: per-country gaps other than the global missing year -----------------
gap_rows = []
for code, sub in df.groupby("Code"):
    yrs = set(sub["Year"])
    missing = [y for y in range(min(yrs), max(yrs) + 1) if y not in yrs and y not in GLOBAL_MISSING]
    if missing:
        gap_rows.append({"Code": code, "Country": sub["Country"].iloc[0],
                         "first_year": min(yrs), "last_year": max(yrs), "missing_years": missing})
country_gaps = pd.DataFrame(gap_rows)
print(f"\nCountries with extra internal gaps (besides {sorted(GLOBAL_MISSING)}): {len(country_gaps)}")
if len(country_gaps):
    print(country_gaps.head(15).to_string(index=False))
    country_gaps.to_csv(os.path.join(TABLES_DIR, "country_gaps_other_than_global.csv"), index=False)


# ---- Part 3: sample eligibility under each rule ------------------------------------
def has(c, y):
    return (c, y) in present


def grid_ok(c, y):
    """Available, or a globally missing year bracketed by observed years y-1 and y+1."""
    return has(c, y) or (y in GLOBAL_MISSING and has(c, y - 1) and has(c, y + 1))


C, Y = df["Code"].to_numpy(), df["Year"].to_numpy()
df["has_target"] = [has(c, y + 1) for c, y in zip(C, Y)]
df["strict"] = [has(c, y + 1) and has(c, y - 1) and has(c, y - 2) for c, y in zip(C, Y)]
df["strict4"] = [s and has(c, y - 3) for s, c, y in zip(df["strict"], C, Y)]
df["gridfill"] = [has(c, y + 1) and grid_ok(c, y - 1) and grid_ok(c, y - 2) for c, y in zip(C, Y)]

elig_year = df.groupby("Year").agg(rows=("Code", "size"), has_target=("has_target", "sum"),
                                   strict=("strict", "sum"), strict4=("strict4", "sum"),
                                   gridfill=("gridfill", "sum"))
print("\nEligible samples by ORIGIN year (feature year t):")
print(elig_year)
elig_year.to_csv(os.path.join(TABLES_DIR, "calendar_eligibility_by_origin_year.csv"))

SPLITS = {"train (origin<=2021)": df["Year"] <= 2021, "val (origin=2022)": df["Year"] == 2022,
          "test (origin=2023)": df["Year"] == 2023}
rows = []
for rule in ["has_target", "strict", "strict4", "gridfill"]:
    for sname, mask in SPLITS.items():
        sel = df[mask & df[rule]]
        rows.append({"rule": rule, "split": sname, "samples": len(sel), "countries": sel["Code"].nunique()})
elig_split = pd.DataFrame(rows).pivot(index="rule", columns="split", values=["samples", "countries"])
print("\nSamples / countries by rule and split:")
print(elig_split)
print("\nOLD pipeline for reference (from your CELL 2): train 1127 rows/144 countries, val 133, test 138")
elig_split.to_csv(os.path.join(TABLES_DIR, "eligibility_by_split.csv"))

# ---- VALIDATION ----------------------------------------------------------------------
assert GLOBAL_MISSING == {2013}, f"Expected only 2013 missing globally, got {GLOBAL_MISSING}"
assert (elig_year["strict4"] <= elig_year["strict"]).all(), "strict4 cannot exceed strict"
assert (elig_year["strict"] <= elig_year["has_target"]).all(), "strict cannot exceed has_target"
assert (elig_year["gridfill"] >= elig_year["strict"]).all(), "gridfill must include every strict sample"
assert elig_year.loc[df["Year"].max(), "has_target"] == 0, "Last year must have no next-year target"
print("\nCELL 17 PASSED")

In [ ]:
# ==================================================
# CELL 16B — CORRECT THE COMMIT EMAIL
#
# GOAL:   Look up your real GitHub numeric ID from GitHub's PUBLIC API
#         (no token) and set the repo-local commit email to match.
# WHY:    CELL 16 used an ID I had no way of knowing. A wrong ID means
#         commits won't link to your GitHub profile.
# EXPECTED OUTPUT: your real ID, the new email, "CELL 16B PASSED".
# ==================================================
import os, subprocess, requests

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
USER = "sharifa5852"

r = requests.get(f"https://api.github.com/users/{USER}", timeout=20)   # read-only, public
assert r.status_code == 200, f"GitHub API returned {r.status_code}; wait a minute and re-run."
info = r.json()
assert info["login"].lower() == USER.lower(), "Login mismatch"
email = f"{info['id']}+{info['login']}@users.noreply.github.com"

def git(*a):
    return subprocess.run(["git", *a], cwd=PROJECT_ROOT, capture_output=True, text=True)

git("config", "--local", "user.email", email)
git("config", "--local", "user.name", info["login"])
print("GitHub ID:", info["id"])
print("Commit identity now:", git("config", "--get", "user.name").stdout.strip(),
      "<" + git("config", "--get", "user.email").stdout.strip() + ">")
assert git("config", "--get", "user.email").stdout.strip() == email
print("CELL 16B PASSED")

In [ ]:
# ==================================================
# CELL 18 — TEMPORAL STRATEGY DECISION (A: grid-fill vs B: exact join)
#
# GOAL:
#   Turn the CELL 17 counts into a written, machine-readable decision that
#   every later cell reads (splits, features, fill policy). Also compute the
#   rolling-origin training sizes we will get under the chosen rule.
#
# DECISION: STRICT (strategy B, exact calendar-year joins)
#   A sample needs REAL observed values at t, t-1, t-2 and a REAL target at t+1.
#   - Validation and test sizes are identical under STRICT and GRIDFILL
#     (133 / 133), so the headline result does not depend on this choice.
#   - GRIDFILL would add 285 training rows (origins 2014-2015) but only by
#     inventing a 2013 lag value. STRICT keeps every number a real observation.
#   - Cost of STRICT: smaller training set; the earliest rolling-origin years
#     are small. We report this as a limitation.
#
# EXPECTED OUTPUT: counts table, rolling-origin size table, "CELL 18 PASSED".
# ==================================================
import os, json
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
PROCESSED_V2_DIR = os.path.join(PROJECT_ROOT, "data", "processed_v2")   # NEW folder; data/processed untouched
os.makedirs(PROCESSED_V2_DIR, exist_ok=True)

elig = pd.read_csv(os.path.join(TABLES_DIR, "calendar_eligibility_by_origin_year.csv"), index_col="Year")

def n(rule, lo, hi):
    return int(elig.loc[(elig.index >= lo) & (elig.index <= hi), rule].sum())

counts = pd.DataFrame({
    rule: {"train (origin<=2021)": n(rule, 0, 2021), "val (origin=2022)": n(rule, 2022, 2022),
           "test (origin=2023)": n(rule, 2023, 2023)}
    for rule in ["strict", "gridfill"]})
print("Samples per split:\n", counts)

# Rolling origin: forecast year Y (target year Y, origin Y-1);
# train only on samples whose TARGET year is < Y, i.e. origins <= Y-2.
roll = []
for Y in range(2018, 2025):
    roll.append({"forecast_year_Y": Y, "train_origins": "<=" + str(Y - 2),
                 "train_n_strict": n("strict", 0, Y - 2), "train_n_gridfill": n("gridfill", 0, Y - 2),
                 "eval_origin": Y - 1, "eval_n_strict": n("strict", Y - 1, Y - 1)})
roll = pd.DataFrame(roll)
print("\nRolling-origin sizes:\n", roll.to_string(index=False))
roll.to_csv(os.path.join(TABLES_DIR, "rolling_origin_sizes_strict_vs_gridfill.csv"), index=False)

decision = {
    "primary_rule": "STRICT",
    "sensitivity_rule": "GRIDFILL (optional, only if time remains; 2013 may be filled only as a flagged lag, never as a target)",
    "rejected": "Row-based groupby().shift(): misaligns target/lags around the missing 2013 (see temporal_alignment_audit_by_year.csv)",
    "split_definition": {"train_origin_max": 2021, "val_origin": 2022, "test_origin": 2023, "test_target_year": 2024},
    "base_features": ["HappinessScore", "log_GDPPerCapita_WB", "UnemploymentRate_WB",
                      "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB"],
    "lags_included": [0, 1, 2],
    "covariate_fill": "past-only calendar forward-fill from the last REAL observation, max gap 2 years; "
                      "HappinessScore is never filled; remaining NaN imputed by a TRAIN-fitted median (CELL 23)",
    "random_seed": 42,
    "sample_counts": {"strict": counts["strict"].to_dict(), "gridfill": counts["gridfill"].to_dict()},
}
with open(os.path.join(TABLES_DIR, "temporal_strategy_decision.json"), "w") as f:
    json.dump(decision, f, indent=2)

# VALIDATION (numbers verified from your CELL 17 output)
assert counts["strict"].tolist() == [842, 133, 133], f"Unexpected strict counts: {counts['strict'].tolist()}"
assert counts["gridfill"].tolist() == [1127, 133, 133], f"Unexpected gridfill counts: {counts['gridfill'].tolist()}"
assert counts.loc["val (origin=2022)", "strict"] == counts.loc["val (origin=2022)", "gridfill"]
print("\nCELL 18 PASSED")

In [ ]:
# ==================================================
# CELL 19 — MISSING-FEATURE POLICY (leakage-safe)
#
# GOAL:   Handle missing covariate values (unemployment 6, internet 24,
#         health 159 rows, of which 143 are 2024 and never used as features).
# POLICY: 1) HappinessScore and GDP are NEVER filled.
#         2) A missing covariate may be copied FORWARD from the same country's
#            last REAL observation if it is at most 2 calendar years old.
#            (Only earlier years are used -> no future information.)
#         3) Every originally-missing value gets a flag column <feature>_was_missing.
#         4) Anything still missing is imputed later with a median computed on
#            TRAINING rows only (CELL 23).
# WHY:    Interpolation would use future values; dropping rows would delete
#         whole countries (Hong Kong, Puerto Rico have no health data at all).
# EXPECTED OUTPUT: a per-feature report and "CELL 19 PASSED".
# ==================================================
import os, json
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
PROCESSED_V2_DIR = os.path.join(PROJECT_ROOT, "data", "processed_v2")
os.makedirs(PROCESSED_V2_DIR, exist_ok=True)
ENRICHED_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "whr_worldbank_enriched.csv")
FILLED_PATH = os.path.join(PROCESSED_V2_DIR, "enriched_filled.csv")

cfg = json.load(open(os.path.join(TABLES_DIR, "temporal_strategy_decision.json")))
MAX_GAP = 2
COVS = ["UnemploymentRate_WB", "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB"]

df = pd.read_csv(ENRICHED_PATH).sort_values(["Code", "Year"]).reset_index(drop=True)
assert df[["HappinessScore", "GDPPerCapita_WB"]].notna().all().all(), "Happiness/GDP must have no NaN"
years = df["Year"].to_numpy()
by_country = df.groupby("Code").indices          # country -> row positions (already in year order)

filled = df.copy()
report, problems = [], []
for col in COVS:
    vals = df[col].to_numpy(float)
    out = vals.copy()
    for code, idx in by_country.items():
        last_v, last_y = np.nan, None
        for i in idx:
            if not np.isnan(vals[i]):                       # real observation: remember it
                last_v, last_y = vals[i], years[i]
            elif last_y is not None and years[i] - last_y <= MAX_GAP:
                out[i] = last_v                             # copy from the PAST real value only
    filled[col] = out
    filled[col + "_was_missing"] = df[col].isna().astype(int)

    # Independent checks
    was = df[col].isna().to_numpy()
    if not np.allclose(filled.loc[~was, col], df.loc[~was, col]):
        problems.append(f"{col}: an observed value was changed")
    lookup = {(c, y): v for c, y, v in zip(df["Code"], df["Year"], df[col]) if not np.isnan(v)}
    newly = filled[was & filled[col].notna()]
    for c, y, v in zip(newly["Code"], newly["Year"], newly[col]):
        if not any(lookup.get((c, y - k)) == v for k in range(1, MAX_GAP + 1)):
            problems.append(f"{col}: filled value for {c}-{y} has no earlier real source")
            break

    m = (df["Year"] <= 2023).to_numpy()                     # only years that can be feature origins
    report.append({"feature": col, "missing_origin_rows": int(was[m].sum()),
                   "filled_by_past_value": int((was & ~np.isnan(out))[m].sum()),
                   "still_missing_origin_rows": int(np.isnan(out[m]).sum()),
                   "countries_still_missing": sorted(df.loc[m & np.isnan(out), "Country"].unique())})

report = pd.DataFrame(report)
print(report.to_string(index=False))
report.to_csv(os.path.join(TABLES_DIR, "missing_value_policy_report.csv"), index=False)
filled.to_csv(FILLED_PATH, index=False)
print("\nSaved:", FILLED_PATH)

assert len(filled) == len(df) == 1897
assert filled[["HappinessScore", "GDPPerCapita_WB"]].equals(df[["HappinessScore", "GDPPerCapita_WB"]]), "Happiness/GDP altered"
assert not problems, problems
print("\nCELL 19 PASSED")

In [ ]:
# ==================================================
# CELL 20 — YEAR-AWARE SUPERVISED TABLE (STRICT rule)
#
# GOAL:   One row per (country, origin year t) containing
#           features at t, t-1, t-2  (5 base features each, 15 columns)
#           + 3 "was_missing" flags (at t)
#           + target = HappinessScore at t+1
# HOW:    Exact calendar-year MERGES (not row shifts). A sample exists only if
#         t-1, t-2 and t+1 are REAL rows, so TargetYear == Year+1 always.
#         Merge trick: to get the value from year s as "lag k" of origin t=s+k,
#         we add k to the lag table's Year before merging.
# WHY:    Fixes the 2013 misalignment found in CELL 17.
# EXPECTED OUTPUT: 1108 rows total; counts per origin year equal CELL 17's
#         'strict' column; "CELL 20 PASSED". Old data/processed is untouched.
# ==================================================
import os, json
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
PROCESSED_V2_DIR = os.path.join(PROJECT_ROOT, "data", "processed_v2")
FILLED_PATH = os.path.join(PROCESSED_V2_DIR, "enriched_filled.csv")
SUP_PATH = os.path.join(PROCESSED_V2_DIR, "supervised_all.csv")
SPEC_PATH = os.path.join(PROCESSED_V2_DIR, "feature_spec.json")

COVS = ["UnemploymentRate_WB", "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB"]
BASE = ["HappinessScore", "log_GDPPerCapita_WB"] + COVS
FLAGS = [c + "_was_missing" for c in COVS]
TARGET = "HappinessScore_target"

filled = pd.read_csv(FILLED_PATH)
assert (filled["GDPPerCapita_WB"] > 0).all(), "GDP must be positive for log"
filled["log_GDPPerCapita_WB"] = np.log(filled["GDPPerCapita_WB"])      # row-wise transform: no leakage

base_tbl = filled[["Code", "Year"] + BASE]

# origin rows (2024 can never have a target)
sup = filled.loc[filled["Year"] <= 2023, ["Country", "Code", "Year"] + BASE + FLAGS].copy()

# target: take the row for year t+1 and relabel it as year t
tgt = filled[["Code", "Year", "HappinessScore"]].copy()
tgt["Year"] = tgt["Year"] - 1
tgt = tgt.rename(columns={"HappinessScore": TARGET})
sup = sup.merge(tgt, on=["Code", "Year"], how="inner", validate="one_to_one")

# calendar lags 1 and 2 (inner merge = row must really exist)
for k in (1, 2):
    lag = base_tbl.copy()
    lag["Year"] = lag["Year"] + k
    lag = lag.rename(columns={b: f"{b}_lag{k}" for b in BASE})
    sup = sup.merge(lag, on=["Code", "Year"], how="inner", validate="one_to_one")

# marker used later by the LSTM (sequence length 4 needs a real t-3); NOT a model feature
l3 = filled[["Code", "Year"]].copy()
l3["Year"] = l3["Year"] + 3
l3["has_lag3"] = 1
sup = sup.merge(l3, on=["Code", "Year"], how="left", validate="one_to_one")
sup["has_lag3"] = sup["has_lag3"].fillna(0).astype(int)

sup["TargetYear"] = sup["Year"] + 1
FEATURE_COLS = BASE + [f"{b}_lag1" for b in BASE] + [f"{b}_lag2" for b in BASE] + FLAGS
ID_COLS = ["Country", "Code", "Year", "TargetYear"]
sup = sup[ID_COLS + [TARGET, "has_lag3"] + FEATURE_COLS].sort_values(["Year", "Code"]).reset_index(drop=True)

sup.to_csv(SUP_PATH, index=False)
with open(SPEC_PATH, "w") as f:
    json.dump({"id_cols": ID_COLS, "target_col": TARGET, "persistence_col": "HappinessScore",
               "base_features": BASE, "flag_cols": FLAGS, "feature_cols": FEATURE_COLS}, f, indent=2)

print("Rows:", len(sup), "| countries:", sup["Code"].nunique(), "| feature columns:", len(FEATURE_COLS))
print(sup.groupby("Year").size().rename("samples").to_frame().T)
print("Remaining NaN per column (only unfilled covariates expected):")
print(sup[FEATURE_COLS].isna().sum()[lambda s: s > 0])

# VALIDATION
elig = pd.read_csv(os.path.join(TABLES_DIR, "calendar_eligibility_by_origin_year.csv"), index_col="Year")
expected = elig["strict"][elig["strict"] > 0]
got = sup.groupby("Year").size()
assert got.equals(expected.astype(got.dtype).rename(got.name)) or (got.to_dict() == expected.to_dict()), \
    f"Counts per origin year differ from CELL 17 strict column:\n{got.to_dict()}\nvs\n{expected.to_dict()}"
assert len(sup) == 1108, f"Expected 1108 rows, got {len(sup)}"
assert (sup["TargetYear"] == sup["Year"] + 1).all()
assert sup[TARGET].notna().all() and sup[["HappinessScore", "log_GDPPerCapita_WB"]].notna().all().all()
print("\nCELL 20 PASSED")

In [ ]:
# ==================================================
# CELL 21 — validate_supervised(): AUTOMATED LEAKAGE / ALIGNMENT GATE
#
# GOAL:   Re-derive every key column of the supervised table INDEPENDENTLY
#         from the ORIGINAL raw file (whr_worldbank_merged.csv) with plain
#         (Code, Year) lookups and compare. Then prove the validator works by
#         feeding it a deliberately corrupted copy.
# WHY:    We do not split, scale or model until this passes.
# EXPECTED OUTPUT: "problems: none", "validator caught the corrupted copy",
#         "CELL 21 PASSED".
# ==================================================
import os, json
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
PROCESSED_V2_DIR = os.path.join(PROJECT_ROOT, "data", "processed_v2")

spec = json.load(open(os.path.join(PROCESSED_V2_DIR, "feature_spec.json")))
sup = pd.read_csv(os.path.join(PROCESSED_V2_DIR, "supervised_all.csv"))
raw = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "whr_worldbank_merged.csv"))       # original, untouched
filled = pd.read_csv(os.path.join(PROCESSED_V2_DIR, "enriched_filled.csv"))
elig = pd.read_csv(os.path.join(TABLES_DIR, "calendar_eligibility_by_origin_year.csv"), index_col="Year")
COVS = ["UnemploymentRate_WB", "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB"]


def validate_supervised(sup, raw, filled, elig):
    """Return a list of problems (empty list = passed)."""
    P = []
    T = spec["target_col"]
    if sup.duplicated(["Code", "Year"]).any():
        P.append("duplicate Code-Year rows")
    if not (sup["TargetYear"] == sup["Year"] + 1).all():
        P.append("TargetYear != Year + 1 for some rows")
    if sup["Year"].max() > 2023 or sup["Year"].min() < 2016:
        P.append(f"origin years outside 2016-2023: {sup['Year'].min()}-{sup['Year'].max()}")
    X = sup[spec["feature_cols"]].to_numpy(float)
    if np.isinf(X).any():
        P.append("infinite values in features")
    if sup[T].isna().any() or np.isinf(sup[T].to_numpy(float)).any():
        P.append("target has NaN/inf")

    hs = dict(zip(zip(raw["Code"], raw["Year"]), raw["HappinessScore"]))
    gd = dict(zip(zip(raw["Code"], raw["Year"]), raw["GDPPerCapita_WB"]))

    def expect(d, off, f=lambda v: v):
        return np.array([f(d.get((c, y + off), np.nan)) for c, y in zip(sup["Code"], sup["Year"])], float)

    def same(e, s):
        return np.isfinite(e).all() and np.allclose(e, s.to_numpy(float), atol=1e-9, rtol=0)

    for col, off in [(T, 1), ("HappinessScore", 0), ("HappinessScore_lag1", -1), ("HappinessScore_lag2", -2)]:
        if not same(expect(hs, off), sup[col]):
            P.append(f"{col} != raw HappinessScore at calendar year t{off:+d}")
    for col, off in [("log_GDPPerCapita_WB", 0), ("log_GDPPerCapita_WB_lag1", -1), ("log_GDPPerCapita_WB_lag2", -2)]:
        if not same(expect(gd, off, np.log), sup[col]):
            P.append(f"{col} != log(raw GDP) at calendar year t{off:+d}")

    for cov in COVS:                       # covariates must match the policy-filled table at the right calendar year
        cd = dict(zip(zip(filled["Code"], filled["Year"]), filled[cov]))
        for col, off in [(cov, 0), (cov + "_lag1", -1), (cov + "_lag2", -2)]:
            e = np.array([cd.get((c, y + off), np.nan) for c, y in zip(sup["Code"], sup["Year"])], float)
            if not np.allclose(e, sup[col].to_numpy(float), atol=1e-9, rtol=0, equal_nan=True):
                P.append(f"{col} misaligned with calendar year t{off:+d}")

    exp_n = elig["strict"][elig["strict"] > 0].to_dict()
    if sup.groupby("Year").size().to_dict() != exp_n:
        P.append("per-year sample counts differ from CELL 17 'strict'")
    return P


problems = validate_supervised(sup, raw, filled, elig)
print("problems:", problems if problems else "none")

# Prove the validator can actually fail
bad = sup.copy()
bad.loc[bad.index[0], "HappinessScore_lag1"] += 0.5
bad2 = sup.copy()
bad2["TargetYear"] = bad2["TargetYear"] + 1
assert validate_supervised(bad, raw, filled, elig), "Validator missed a corrupted lag!"
assert validate_supervised(bad2, raw, filled, elig), "Validator missed a wrong TargetYear!"
print("validator caught both corrupted copies")

pd.DataFrame({"check_result": problems or ["all checks passed"]}).to_csv(
    os.path.join(TABLES_DIR, "supervised_validation_report.csv"), index=False)
assert not problems, f"Supervised table FAILED validation: {problems}"
print("\nCELL 21 PASSED")

In [ ]:
# ==================================================
# CELL 22 — CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
#
# GOAL:
#   Split supervised_all.csv by ORIGIN year (the "Year" column):
#   train <= 2021, validation = 2022, test = 2023 (test target year = 2024).
#   Save to data/processed_v2/ (the old data/processed is never touched).
#
# WHY:
#   Forecasting must train on the past and test on the future. A training
#   row with origin 2021 has target year 2022. That is NOT leakage: at the
#   2022 forecast origin, the 2022 outcome is already known history.
#
# WHAT HAPPENS:
#   1) read table + feature_spec.json  2) filter rows with boolean masks
#   3) prove the splits are disjoint, complete and chronological
#   4) print a summary and save it  5) write 3 CSVs (never silently overwrite)
#
# EXPECTED OUTPUT (counts verified by your CELL 17/18 output):
#   train 842 rows / 152 countries, val 133, test 133, "CELL 22 PASSED".
#
# POSSIBLE FAILURES: see the table below the cell.
# ==================================================
import os, json
import numpy as np
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
PROCESSED_V2_DIR = os.path.join(PROJECT_ROOT, "data", "processed_v2")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
SUP_PATH = os.path.join(PROCESSED_V2_DIR, "supervised_all.csv")
assert os.path.exists(SUP_PATH), f"Missing {SUP_PATH}. Re-run CELLS 19-21."

OVERWRITE = False          # False = never silently replace an existing split file
TRAIN_MAX, VAL_YEAR, TEST_YEAR = 2021, 2022, 2023
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

spec = json.load(open(os.path.join(PROCESSED_V2_DIR, "feature_spec.json")))
FEATURE_COLS, T = spec["feature_cols"], spec["target_col"]
sup = pd.read_csv(SUP_PATH)

train = sup[sup["Year"] <= TRAIN_MAX].reset_index(drop=True)
val = sup[sup["Year"] == VAL_YEAR].reset_index(drop=True)
test = sup[sup["Year"] == TEST_YEAR].reset_index(drop=True)


def same_df(a, b):
    """True if two tables hold the same values (text exact, numbers within 1e-9)."""
    if list(a.columns) != list(b.columns) or a.shape != b.shape:
        return False
    for c in a.columns:
        if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]):
            if not np.allclose(a[c].to_numpy(float), b[c].to_numpy(float), atol=1e-9, equal_nan=True):
                return False
        elif not (a[c].astype(str) == b[c].astype(str)).all():
            return False
    return True


def write_or_verify_csv(df, path):
    if os.path.exists(path) and not OVERWRITE:
        if same_df(pd.read_csv(path), df):
            print("  exists & identical, kept:", os.path.basename(path))
            return
        raise FileExistsError(f"{path} exists and DIFFERS from the fresh result. "
                              "Inspect it; set OVERWRITE = True only if you are sure.")
    df.to_csv(path, index=False)
    print("  saved:", os.path.basename(path))


# ---- summary --------------------------------------------------------------------
rows = []
for name, d in [("train", train), ("val", val), ("test", test)]:
    rows.append({"split": name, "rows": len(d), "countries": d["Code"].nunique(),
                 "origin_min": d["Year"].min(), "origin_max": d["Year"].max(),
                 "target_min": d["TargetYear"].min(), "target_max": d["TargetYear"].max(),
                 "n_features": len(FEATURE_COLS),
                 "rows_with_any_NaN_feature": int(d[FEATURE_COLS].isna().any(axis=1).sum()),
                 "NaN_cells": int(d[FEATURE_COLS].isna().sum().sum())})
summary = pd.DataFrame(rows)
print(summary.T)
tr_c = set(train["Code"])
print("\nVal countries never seen in train:", len(set(val["Code"]) - tr_c),
      "| Test countries never seen in train:", len(set(test["Code"]) - tr_c))
summary.to_csv(os.path.join(TABLES_DIR, "split_summary_v2.csv"), index=False)

print("\nWriting files:")
for nm, d in [("train", train), ("val", val), ("test", test)]:
    write_or_verify_csv(d, os.path.join(PROCESSED_V2_DIR, f"{nm}.csv"))

# VALIDATION
assert len(train) + len(val) + len(test) == len(sup), "Splits do not cover every row exactly once"
key = lambda d: set(zip(d["Code"], d["Year"]))
assert not (key(train) & key(val)) and not (key(train) & key(test)) and not (key(val) & key(test)), "Splits overlap"
assert train["Year"].max() < val["Year"].min() <= val["Year"].max() < test["Year"].min(), "Not chronological"
assert (train["Year"] <= 2021).all() and (val["Year"] == 2022).all() and (test["Year"] == 2023).all()
assert (train["TargetYear"] == train["Year"] + 1).all() and (val["TargetYear"] == val["Year"] + 1).all() \
    and (test["TargetYear"] == test["Year"] + 1).all()
assert test["TargetYear"].eq(2024).all(), "Test targets must all be 2024"
assert train["TargetYear"].max() == 2022, "Train targets must stop at 2022 (known history at origin 2022)"
assert [len(train), len(val), len(test)] == [842, 133, 133], "Counts differ from CELL 18 STRICT decision"
for nm in ("train", "val", "test"):
    back = pd.read_csv(os.path.join(PROCESSED_V2_DIR, f"{nm}.csv"))
    assert list(back.columns) == list(sup.columns) and back[T].notna().all(), f"{nm}.csv failed reload check"
print("\nCELL 22 PASSED")

In [ ]:
# ==================================================
# CELL 23 — TRAIN-ONLY IMPUTATION AND SCALING
#
# GOAL:
#   Fit median imputers and StandardScalers on TRAINING rows only, apply them
#   unchanged to validation and test, save matrices + joblib artifacts.
#
# WHY:
#   Fitting on val/test would leak their statistics into training.
#   Trees do not need scaling, so we save imputed-unscaled AND scaled versions.
#
# WHAT HAPPENS:
#   imputer.fit(train)       -> learns one median per column from train
#   imputer.transform(any)   -> fills NaN with those train medians
#   scaler.fit(train)        -> learns mean and std per column from train
#   scaler.transform(any)    -> (x - train_mean) / train_std
#
# EXPECTED OUTPUT:
#   shapes (842,18)/(133,18)/(133,18), seq shapes (n,3,5),
#   proof lines about n_samples_seen_ = 842, "CELL 23 PASSED".
#
# POSSIBLE FAILURES: see the table below the cell.
# ==================================================
import os, json
import numpy as np
import pandas as pd
import joblib, sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
PROCESSED_V2_DIR = os.path.join(PROJECT_ROOT, "data", "processed_v2")
PREP_DIR = os.path.join(PROJECT_ROOT, "models", "preprocessing")
os.makedirs(PREP_DIR, exist_ok=True)
OVERWRITE = False
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

spec = json.load(open(os.path.join(PROCESSED_V2_DIR, "feature_spec.json")))
FEATURE_COLS, FLAGS, BASE = spec["feature_cols"], spec["flag_cols"], spec["base_features"]
ID_COLS, T = spec["id_cols"], spec["target_col"]
CONT = [c for c in FEATURE_COLS if c not in FLAGS]            # 15 continuous columns
assert len(FEATURE_COLS) == 18 and len(CONT) == 15

train, val, test = [pd.read_csv(os.path.join(PROCESSED_V2_DIR, f"{n}.csv")) for n in ("train", "val", "test")]
assert len(train) == 842, "Train must have 842 rows; re-run CELL 22"

# ---- pre-fit safety checks (TRAIN only) ---------------------------------------------
empty = [c for c in FEATURE_COLS if train[c].notna().sum() == 0]
assert not empty, f"Columns entirely missing in TRAIN (cannot compute a median): {empty}"
const = [c for c in CONT if train[c].nunique(dropna=True) < 2]
assert not const, f"Constant columns in TRAIN (scaler would divide by 0): {const}"

# ======================= A) TABULAR artifacts ===========================================
imp_tab = SimpleImputer(strategy="median", keep_empty_features=True)
imp_tab.fit(train[FEATURE_COLS])                     # <-- learns medians from TRAIN only


def impute(df):
    return pd.DataFrame(imp_tab.transform(df[FEATURE_COLS]), columns=FEATURE_COLS, index=df.index)


Xi = {"train": impute(train), "val": impute(val), "test": impute(test)}
sc_tab = StandardScaler()
sc_tab.fit(Xi["train"][CONT])                        # <-- learns mean/std from TRAIN only


def scale(X):
    out = X.copy()
    out[CONT] = sc_tab.transform(X[CONT])
    return out                                       # flags stay 0/1


Xs = {k: scale(v) for k, v in Xi.items()}
frames = {"train": train, "val": val, "test": test}


def assemble(name, X):
    return pd.concat([frames[name][ID_COLS + [T]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)


# ======================= B) LSTM artifacts ==============================================
STEPS = [("t-2", "_lag2"), ("t-1", "_lag1"), ("t", "")]          # oldest -> newest


def seq_array(df):
    return np.stack([df[[b + suf for b in BASE]].to_numpy(float) for _, suf in STEPS], axis=1)   # (n,3,5)


S = {k: seq_array(v) for k, v in frames.items()}
imp_seq = SimpleImputer(strategy="median", keep_empty_features=True)
sc_seq = StandardScaler()
tr_flat = pd.DataFrame(S["train"].reshape(-1, 5), columns=BASE)
imp_seq.fit(tr_flat)
sc_seq.fit(pd.DataFrame(imp_seq.transform(tr_flat), columns=BASE))


def seq_prepare(a):
    flat = pd.DataFrame(a.reshape(-1, 5), columns=BASE)
    out = sc_seq.transform(pd.DataFrame(imp_seq.transform(flat), columns=BASE))
    return out.reshape(a.shape)


SEQ = {k: seq_prepare(v) for k, v in S.items()}

# ======================= C) SAVE (verify instead of overwrite) ===========================
def same_df(a, b):
    return list(a.columns) == list(b.columns) and a.shape == b.shape and all(
        np.allclose(a[c].to_numpy(float), b[c].to_numpy(float), atol=1e-9, equal_nan=True)
        if pd.api.types.is_numeric_dtype(a[c]) else (a[c].astype(str) == b[c].astype(str)).all()
        for c in a.columns)


def write_or_verify_csv(df, path):
    if os.path.exists(path) and not OVERWRITE:
        assert same_df(pd.read_csv(path), df), f"{path} exists and differs. Set OVERWRITE=True only if intended."
        print("  exists & identical, kept:", os.path.basename(path)); return
    df.to_csv(path, index=False); print("  saved:", os.path.basename(path))


for nm in ("train", "val", "test"):
    write_or_verify_csv(assemble(nm, Xi[nm]), os.path.join(PROCESSED_V2_DIR, f"{nm}_imputed.csv"))
    write_or_verify_csv(assemble(nm, Xs[nm]), os.path.join(PROCESSED_V2_DIR, f"{nm}_scaled.csv"))

ART = {"tabular_imputer.joblib": imp_tab, "tabular_scaler.joblib": sc_tab,
       "lstm_imputer.joblib": imp_seq, "lstm_scaler.joblib": sc_seq}
for fn, obj in ART.items():
    p = os.path.join(PREP_DIR, fn)
    if os.path.exists(p) and not OVERWRITE:
        print("  exists, will verify against fresh fit:", fn)
    else:
        joblib.dump(obj, p); print("  saved:", fn)

NPZ = os.path.join(PROCESSED_V2_DIR, "lstm_sequences.npz")
if not (os.path.exists(NPZ) and not OVERWRITE):
    np.savez_compressed(
        NPZ, X_train=SEQ["train"], X_val=SEQ["val"], X_test=SEQ["test"],
        y_train=train[T].to_numpy(float), y_val=val[T].to_numpy(float), y_test=test[T].to_numpy(float),
        flags_train=train[FLAGS].to_numpy(float), flags_val=val[FLAGS].to_numpy(float), flags_test=test[FLAGS].to_numpy(float),
        code_train=train["Code"].to_numpy(str), code_val=val["Code"].to_numpy(str), code_test=test["Code"].to_numpy(str),
        year_train=train["Year"].to_numpy(int), year_val=val["Year"].to_numpy(int), year_test=test["Year"].to_numpy(int),
        timesteps=np.array([s for s, _ in STEPS]), features=np.array(BASE))
    print("  saved: lstm_sequences.npz")
else:
    print("  exists, will verify against fresh arrays: lstm_sequences.npz")

meta = {
    "sklearn_version": sklearn.__version__, "random_seed": RANDOM_SEED,
    "fit_rows": "train only (origin year <= 2021)", "n_train_rows": int(len(train)),
    "feature_order": FEATURE_COLS, "continuous_scaled_columns": CONT, "binary_flag_columns_not_scaled": FLAGS,
    "tabular_imputer": {"strategy": "median", "statistics": dict(zip(FEATURE_COLS, map(float, imp_tab.statistics_)))},
    "tabular_scaler": {"mean": dict(zip(CONT, map(float, sc_tab.mean_))), "scale": dict(zip(CONT, map(float, sc_tab.scale_)))},
    "lstm": {"timesteps_oldest_to_newest": [s for s, _ in STEPS], "features": BASE,
             "imputer_medians": dict(zip(BASE, map(float, imp_seq.statistics_))),
             "scaler_mean": dict(zip(BASE, map(float, sc_seq.mean_))), "scaler_scale": dict(zip(BASE, map(float, sc_seq.scale_))),
             "note": "Fit on the 3 stacked time steps of TRAIN rows (all values from years <= 2021). "
                     "Length-4 sequences are built in Phase 3 from enriched_filled.csv with their own artifacts."},
    "target_scaled": False,
    "artifact_files": list(ART) + ["lstm_sequences.npz"],
}
with open(os.path.join(PROCESSED_V2_DIR, "preprocessing_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

# ======================= VALIDATION ======================================================
# 1. Reload artifacts from disk and prove they equal the fresh TRAIN-only fits
L = {fn: joblib.load(os.path.join(PREP_DIR, fn)) for fn in ART}
assert np.allclose(L["tabular_imputer.joblib"].statistics_, train[FEATURE_COLS].median().to_numpy(float)), \
    "Imputer medians are not the TRAIN medians"
assert np.allclose(L["tabular_scaler.joblib"].mean_, Xi["train"][CONT].mean().to_numpy(float), atol=1e-9)
assert np.allclose(L["tabular_scaler.joblib"].scale_, Xi["train"][CONT].std(ddof=0).to_numpy(float), atol=1e-9)
assert L["tabular_scaler.joblib"].n_samples_seen_ == len(train) == 842, "Scaler saw rows other than TRAIN"
assert L["lstm_scaler.joblib"].n_samples_seen_ == len(train) * 3, "LSTM scaler saw rows other than TRAIN steps"
assert np.allclose(L["lstm_imputer.joblib"].statistics_, imp_seq.statistics_)
assert np.allclose(L["lstm_scaler.joblib"].mean_, sc_seq.mean_)
print("Proof: scaler.n_samples_seen_ =", int(L["tabular_scaler.joblib"].n_samples_seen_),
      "(= train rows only). Val+test rows would make it", len(train) + len(val) + len(test))

# 2. Informational: would including val/test have changed anything? (it should)
all_med = pd.concat([train, val, test])[FEATURE_COLS].median().to_numpy(float)
print("Columns whose median would change if val/test were included:",
      int((~np.isclose(all_med, imp_tab.statistics_)).sum()), "of", len(FEATURE_COLS))
print("Mean |val scaled mean| (not exactly 0 = proof val was not used to fit):",
      round(float(np.abs(Xs["val"][CONT].mean()).mean()), 4))

# 3. Train scaled columns have mean 0, std 1
assert np.allclose(Xs["train"][CONT].mean(), 0, atol=1e-9) and np.allclose(Xs["train"][CONT].std(ddof=0), 1, atol=1e-9)

# 4. No NaN / inf anywhere
for k in ("train", "val", "test"):
    for nm, M in (("imputed", Xi[k]), ("scaled", Xs[k])):
        arr = M.to_numpy(float)
        assert np.isfinite(arr).all(), f"NaN/inf in {k} {nm}"
    assert np.isfinite(SEQ[k]).all(), f"NaN/inf in {k} sequences"

# 5. Order, flags, target
for k in ("train", "val", "test"):
    assert list(Xi[k].columns) == FEATURE_COLS == list(Xs[k].columns), "Feature order changed"
    assert set(np.unique(Xs[k][FLAGS].to_numpy())) <= {0.0, 1.0}, "Flags are not 0/1"
    on_disk = pd.read_csv(os.path.join(PROCESSED_V2_DIR, f"{k}_scaled.csv"))
    assert np.allclose(on_disk[T], frames[k][T]), "Target in scaled file was changed"
    assert list(on_disk[FEATURE_COLS].columns) == FEATURE_COLS
z = np.load(NPZ)
assert z["X_train"].shape == (842, 3, 5) and z["X_val"].shape == (133, 3, 5) and z["X_test"].shape == (133, 3, 5)
assert np.allclose(z["X_train"], SEQ["train"]) and np.allclose(z["y_test"], test[T])
print("Shapes tabular:", Xs["train"].shape, Xs["val"].shape, Xs["test"].shape,
      "| sequences:", z["X_train"].shape, z["X_val"].shape, z["X_test"].shape)
print("\nCELL 23 PASSED")

In [ ]:
# ==================================================
# CELL 24 — FINAL ARTIFACT ORGANIZATION + RELOAD CHECK
#
# GOAL:
#   Write feature_metadata.json, missing_data_policy.json and
#   dataset_metadata.json, then verify EVERY Phase 1 file exists and loads.
#   Confirm the old data/processed and original raw files are unmodified.
#
# EXPECTED OUTPUT:
#   a manifest table with all rows loaded_ok = True, a clean git check,
#   "CELL 24 PASSED". Also saves results/tables/file_manifest_phase1.csv.
#
# POSSIBLE FAILURES: see the table below the cell.
# ==================================================
import os, json, subprocess
import numpy as np
import pandas as pd
import joblib, sklearn

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction"
P = lambda *a: os.path.join(PROJECT_ROOT, *a)
V2, TABLES, PREP = P("data", "processed_v2"), P("results", "tables"), P("models", "preprocessing")
for d in (V2, TABLES, P("results", "figures"), PREP):
    os.makedirs(d, exist_ok=True)

spec = json.load(open(os.path.join(V2, "feature_spec.json")))
FEATURE_COLS, FLAGS, BASE = spec["feature_cols"], spec["flag_cols"], spec["base_features"]
CONT = [c for c in FEATURE_COLS if c not in FLAGS]

# ---- feature_metadata.json -------------------------------------------------------------
DESC = {
    "HappinessScore": ("WHR Cantril ladder average, 0-10", "OWID/WHR", "points", "none"),
    "log_GDPPerCapita_WB": ("GDP per capita (PPP, World Bank)", "OWID/World Bank", "ln(international $)", "natural log; GDP verified > 0"),
    "UnemploymentRate_WB": ("Unemployment rate", "OWID/World Bank (SL.UEM.TOTL.ZS)", "% of labour force", "none"),
    "InternetAccessPct_WB": ("Individuals using the internet", "OWID/World Bank (IT.NET.USER.ZS)", "% of population", "none"),
    "HealthExpenditurePctGDP_WB": ("Current health expenditure", "OWID (WHO GHED)", "% of GDP", "none"),
}
LAGN = {"": "t (current year)", "_lag1": "t-1 (calendar)", "_lag2": "t-2 (calendar)"}
cols = []
for col in FEATURE_COLS:
    if col in FLAGS:
        b = col.replace("_was_missing", "")
        cols.append({"name": col, "role": "feature", "meaning": f"1 if {b} was missing at year t in the source",
                     "source": "derived (CELL 19)", "unit": "0/1", "lag": "t", "transformation": "none",
                     "scaled_for_linear": False, "classical_models": True, "lstm": False})
        continue
    base = next(b for b in BASE if col.startswith(b))
    suf = col[len(base):]
    d = DESC[base]
    cols.append({"name": col, "role": "feature", "meaning": d[0], "source": d[1], "unit": d[2], "lag": LAGN[suf],
                 "transformation": d[3] + ("; past-only forward-fill <=2y, median train-imputed" if base in
                                           ("UnemploymentRate_WB", "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB") else ""),
                 "scaled_for_linear": True, "classical_models": True, "lstm": True})
cols += [
    {"name": "HappinessScore_target", "role": "TARGET", "meaning": "WHR happiness at year t+1", "unit": "points",
     "transformation": "none, never scaled/imputed"},
    {"name": "Country", "role": "identifier"}, {"name": "Code", "role": "identifier"},
    {"name": "Year", "role": "identifier (origin year t)"}, {"name": "TargetYear", "role": "identifier (= Year+1)"},
    {"name": "has_lag3", "role": "metadata (real t-3 exists); NOT a model feature"}]
feature_meta = {"feature_order": FEATURE_COLS, "continuous_scaled_columns": CONT, "binary_flag_columns": FLAGS,
                "lstm_timesteps_oldest_to_newest": ["t-2", "t-1", "t"], "lstm_features": BASE,
                "persistence_baseline_column": "HappinessScore", "columns": cols}
json.dump(feature_meta, open(os.path.join(V2, "feature_metadata.json"), "w"), indent=2)

# ---- missing_data_policy.json ----------------------------------------------------------
rep = pd.read_csv(os.path.join(TABLES, "missing_value_policy_report.csv"))
policy = {
    "never_imputed": ["HappinessScore", "GDPPerCapita_WB", "HappinessScore_target"],
    "covariate_rule": "forward-fill from the same country's last REAL observation, maximum 2 calendar years, past values only",
    "no_interpolation_reason": "interpolation would use values after the forecast origin",
    "flags": FLAGS, "remaining_nan": "median imputed using TRAINING rows only (CELL 23)",
    "known_limitation": "Hong Kong and Puerto Rico have no health expenditure data at all; their value is the training median plus flag=1",
    "report": rep.drop(columns=["countries_still_missing"]).to_dict(orient="records"),
}
json.dump(policy, open(os.path.join(TABLES, "missing_data_policy.json"), "w"), indent=2)

# ---- dataset_metadata.json -------------------------------------------------------------
sp = {n: pd.read_csv(os.path.join(V2, f"{n}.csv")) for n in ("train", "val", "test")}
ds = {"strategy": "STRICT calendar-year join", "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__,
      "n_features": len(FEATURE_COLS), "total_samples": int(sum(len(d) for d in sp.values())),
      "countries_total": int(pd.concat(sp.values())["Code"].nunique()),
      "splits": {n: {"rows": len(d), "countries": int(d["Code"].nunique()),
                     "origin_years": [int(d["Year"].min()), int(d["Year"].max())],
                     "target_years": [int(d["TargetYear"].min()), int(d["TargetYear"].max())]} for n, d in sp.items()}}
json.dump(ds, open(os.path.join(V2, "dataset_metadata.json"), "w"), indent=2)

# ---- manifest: every file exists AND loads ---------------------------------------------
files = [P("data", "raw", "whr_worldbank_merged.csv"), P("data", "raw", "whr_worldbank_enriched.csv"),
         *[os.path.join(V2, f) for f in ["enriched_filled.csv", "supervised_all.csv", "train.csv", "val.csv", "test.csv",
            "train_imputed.csv", "val_imputed.csv", "test_imputed.csv", "train_scaled.csv", "val_scaled.csv", "test_scaled.csv",
            "feature_spec.json", "feature_metadata.json", "preprocessing_metadata.json", "dataset_metadata.json", "lstm_sequences.npz"]],
         *[os.path.join(PREP, f) for f in ["tabular_imputer.joblib", "tabular_scaler.joblib", "lstm_imputer.joblib", "lstm_scaler.joblib"]],
         *[os.path.join(TABLES, f) for f in ["temporal_strategy_decision.json", "missing_data_policy.json", "missing_value_policy_report.csv",
            "temporal_alignment_audit_by_year.csv", "country_gaps_other_than_global.csv", "calendar_eligibility_by_origin_year.csv",
            "eligibility_by_split.csv", "coverage_by_year_enriched.csv", "coverage_by_split_enriched.csv",
            "feature_decisions_final.csv", "supervised_validation_report.csv", "split_summary_v2.csv"]]]
rows = []
for f in files:
    ok, info = False, ""
    if os.path.exists(f):
        try:
            if f.endswith(".csv"): info = str(pd.read_csv(f).shape)
            elif f.endswith(".json"): info = f"{len(json.load(open(f)))} keys"
            elif f.endswith(".joblib"): info = type(joblib.load(f)).__name__
            elif f.endswith(".npz"): info = str(np.load(f)["X_train"].shape)
            ok = True
        except Exception as e:
            info = f"LOAD ERROR: {e}"
    else:
        info = "MISSING"
    rows.append({"file": os.path.relpath(f, PROJECT_ROOT), "loaded_ok": ok, "detail": info,
                 "size_kb": round(os.path.getsize(f) / 1024, 1) if os.path.exists(f) else None})
man = pd.DataFrame(rows)
print(man.to_string(index=False))
man.to_csv(os.path.join(TABLES, "file_manifest_phase1.csv"), index=False)

# ---- old/original files untouched? (read-only git check) ------------------------------
g = subprocess.run(["git", "status", "--porcelain", "--", "data/processed", "data/raw/whr_worldbank_merged.csv"],
                   cwd=PROJECT_ROOT, capture_output=True, text=True)
print("\nChanges to old processed data / original raw file (must be empty):", repr(g.stdout.strip()))

# VALIDATION
assert man["loaded_ok"].all(), f"Files missing or unreadable:\n{man[~man['loaded_ok']]}"
assert g.stdout.strip() == "", "Original data/processed or raw file was MODIFIED. Restore with git checkout (see table)."
assert joblib.load(os.path.join(PREP, "tabular_imputer.joblib")).n_features_in_ == len(FEATURE_COLS) == 18
assert joblib.load(os.path.join(PREP, "tabular_scaler.joblib")).n_features_in_ == len(CONT) == 15
assert joblib.load(os.path.join(PREP, "lstm_scaler.joblib")).n_features_in_ == 5
print("\nCELL 24 PASSED")

In [ ]:
# ==================================================
# SETUP — run this FIRST in a new Colab session (safe to re-run any time)
#
# GOAL: mount Google Drive, define PROJECT_ROOT, go into the repo, and confirm you are
#       on branch modeling-phase. Nothing is downloaded or changed.
# EXPECTED OUTPUT: the project path, "Branch: modeling-phase", "SETUP PASSED".
# FAILURES: "Project folder not found" -> the Drive path is different; run `!ls /content/drive/MyDrive`
#           and fix the folder names. "Branch is X" -> run: !git checkout modeling-phase  (inside the project folder).
# ==================================================
import os, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("(not running in Colab: using the paths as given)")
PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), f"Project folder not found: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
subprocess.run(["git", "config", "--global", "--add", "safe.directory", PROJECT_ROOT], capture_output=True)
branch = subprocess.run(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()
print("Project:", PROJECT_ROOT); print("Branch:", branch)
assert branch == "modeling-phase", f"Branch is '{branch}', expected modeling-phase"
print("SETUP PASSED")

In [ ]:
# ==================================================
# CELL 25 — UPDATED DATA DICTIONARY (data/processed_v2/data_dictionary.md)
#
# GOAL:
#   Generate a complete, exact data dictionary for the NEW dataset from the
#   JSON/CSV files CELLS 18-24 already saved. Nothing is typed by hand, so
#   the numbers cannot drift from the real files.
#
# WHY:
#   Your friend's data/processed/data_dictionary.md describes the OLD
#   row-shift pipeline. We leave it untouched (provenance) and write a new one
#   next to the new data.
#
# WHAT HAPPENS:
#   open("w").write(text) creates/replaces ONE markdown file. Before replacing
#   an existing version we copy it to results/_backups/ (never silently lost).
#
# EXPECTED OUTPUT:
#   "Wrote .../data_dictionary.md", a short preview, "CELL 25 PASSED".
#
# POSSIBLE FAILURES: see the table below the cell.
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - FileNotFoundError ... feature_metadata.json / preprocessing_metadata.json | CELLS 20-24 did not finish in this Drive folder | re-run CELLS 20-24, then 25
#   - KeyError: 'no_interpolation_reason' (or another policy key) | missing_data_policy.json written by an older CELL 24 | re-run CELL 24, then 25
#   - AssertionError: Columns missing from the dictionary | feature_metadata.json is stale | re-run CELL 24, then 25
#   - Project folder not found | Drive unmounted after a runtime reset | run SETUP, then 25
# ==================================================
import os, json, shutil, datetime
import pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
if not os.path.isdir(PROJECT_ROOT):                       # runtime reset? try to re-mount Drive (Colab only)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
assert os.path.isdir(PROJECT_ROOT), (
    f"{PROJECT_ROOT} not found. Run the Step 1 Drive-mount cell, then re-run this cell.")

V2 = os.path.join(PROJECT_ROOT, "data", "processed_v2")
TABLES = os.path.join(PROJECT_ROOT, "results", "tables")
BACKUPS = os.path.join(PROJECT_ROOT, "results", "_backups")
DICT_PATH = os.path.join(V2, "data_dictionary.md")
OLD_DICT = os.path.join(PROJECT_ROOT, "data", "processed", "data_dictionary.md")

need = ["feature_metadata.json", "preprocessing_metadata.json", "dataset_metadata.json", "feature_spec.json"]
miss = [f for f in need if not os.path.exists(os.path.join(V2, f))]
assert not miss, f"Missing Phase 1 files {miss}. Re-run CELLS 20-24."
fm = json.load(open(os.path.join(V2, "feature_metadata.json")))
pm = json.load(open(os.path.join(V2, "preprocessing_metadata.json")))
dm = json.load(open(os.path.join(V2, "dataset_metadata.json")))
spec = json.load(open(os.path.join(V2, "feature_spec.json")))
sd = json.load(open(os.path.join(TABLES, "temporal_strategy_decision.json")))
mp = json.load(open(os.path.join(TABLES, "missing_data_policy.json")))
old_audit = pd.read_csv(os.path.join(TABLES, "baseline_audit_old_pipeline.csv")).iloc[0]

FEATURES, FLAGS, CONT = fm["feature_order"], fm["binary_flag_columns"], fm["continuous_scaled_columns"]
meta_by_name = {c["name"]: c for c in fm["columns"]}
T = spec["target_col"]

L = []
A = L.append
A("# Data Dictionary - processed_v2 (Happiness Forecasting)\n")
A(f"_Generated by CELL 25 on {datetime.date.today().isoformat()} from the saved metadata files. Do not edit by hand; re-run CELL 25._\n")
A("## 1. Terminology and target\n")
A("- The repository name says *Wellness*; every column and metric uses **Happiness**. They are the same quantity here: "
  "the World Happiness Report **Cantril ladder** score (0-10, self-reported life evaluation).")
A(f"- **Target** `{T}` = happiness score in calendar year **t+1**. It is regression (MAE, RMSE, R2). It is **never scaled or imputed**, so errors are in score points.\n")
A("## 2. Temporal setup (why this dataset differs from `data/processed`)\n")
A("- One row = one country at **origin year t**. Features use years t, t-1, t-2. The target is year t+1.")
A(f"- Strategy **{sd['primary_rule']}**: a row exists only if the real observations for t-2, t-1, t and t+1 all exist (exact calendar-year joins). "
  "Year 2013 is absent for every country, so no feature or target is ever invented across that gap.")
A("- The old pipeline used row-position `shift()`, which silently linked 2012 to 2014 and 2014 to 2012. See `results/tables/temporal_alignment_audit_by_year.csv`.\n")
A("| Split | Origin years | Target years | Rows | Countries |\n|---|---|---|---|---|")
for n, s in dm["splits"].items():
    A(f"| {n} | {s['origin_years'][0]}-{s['origin_years'][1]} | {s['target_years'][0]}-{s['target_years'][1]} | {s['rows']} | {s['countries']} |")
A(f"\nTotal: **{dm['total_samples']}** samples, **{dm['countries_total']}** countries, **{dm['n_features']}** model features "
  f"({len(CONT)} continuous + {len(FLAGS)} binary missingness flags).\n")
A("A training row with origin 2021 has target year 2022. That is not leakage: at the 2022 forecast origin, the 2022 outcome is known history. "
  "Validation targets are 2023 and test targets are 2024 (the test origin year is 2023).\n")

A("## 3. Files\n")
files = [
    ("supervised_all.csv", "All samples before splitting (also has `has_lag3`, a metadata marker, not a feature)."),
    ("train.csv / val.csv / test.csv", "Chronological splits, raw values, NaN left where a covariate was unavailable."),
    ("train_imputed.csv / val_imputed.csv / test_imputed.csv", "Medians from TRAIN fill NaN. **Unscaled.** Used by Random Forest and XGBoost and by the persistence baseline."),
    ("train_scaled.csv / val_scaled.csv / test_scaled.csv", "Imputed, then continuous columns standardised with TRAIN mean/std. Flags stay 0/1. Used by Linear Regression."),
    ("lstm_sequences.npz", "LSTM arrays, shape (samples, 3, 5), oldest to newest (t-2, t-1, t), train-fitted imputer + scaler."),
    ("lstm_seq_len4.npz", "Created later in CELL 34 (length-4 sequences, own train-fitted artifacts)."),
    ("enriched_filled.csv", "Raw enriched data after the past-only fill policy (CELL 19)."),
    ("../../models/preprocessing/*.joblib", "Fitted imputers/scalers (fit on training rows only)."),
]
A("| File | Content |\n|---|---|")
for f, d in files:
    A(f"| `{f}` | {d} |")
A("")

A("## 4. Identifier and target columns\n")
A("| Column | Meaning |\n|---|---|")
A("| `Country`, `Code` | Country name and ISO3 code (join key with `Year`). |")
A("| `Year` | Origin year t (the last year whose information is used). |")
A("| `TargetYear` | Always `Year + 1`. |")
A(f"| `{T}` | Happiness in year t+1 (target, unscaled). |\n")

A("## 5. Feature columns (all 18, in model order)\n")
A("| # | Column | Meaning | Source | Unit | Lag | Transformation | In `*_imputed` | In `*_scaled` | Classical models | LSTM |\n|---|---|---|---|---|---|---|---|---|---|---|")
for i, name in enumerate(FEATURES, 1):
    c = meta_by_name[name]
    scaled = "0/1 unchanged" if name in FLAGS else "standardised (train mean/std)"
    A(f"| {i} | `{name}` | {c['meaning']} | {c['source']} | {c['unit']} | {c['lag']} | {c['transformation']} | unscaled | {scaled} | "
      f"{'yes' if c['classical_models'] else 'no'} | {'yes (5 base features x steps)' if c['lstm'] else 'no (flags are not LSTM inputs)'} |")
A("")
A("**Persistence baseline** uses `HappinessScore` (year t) as the prediction for year t+1.\n")

A("## 6. Preprocessing decisions\n")
A(f"- **Missing covariates:** {mp['covariate_rule']}. Interpolation is not used: {mp['no_interpolation_reason']}.")
A(f"- **Never imputed:** {', '.join('`'+c+'`' for c in mp['never_imputed'])}.")
A(f"- **Flags:** {', '.join('`'+c+'`' for c in FLAGS)} = 1 where the value was originally missing at year t.")
A(f"- **Remaining NaN:** {mp['remaining_nan']}.")
A(f"- **Known limitation:** {mp['known_limitation']}.")
A(f"- **Fit rows:** {pm['fit_rows']} ({pm['n_train_rows']} rows). Validation and test only use `transform`.")
A("- **GDP:** natural log, GDP verified > 0.\n")
A("### Train medians used for imputation\n")
A("| Column | Median |\n|---|---|")
for k, v in pm["tabular_imputer"]["statistics"].items():
    A(f"| `{k}` | {v:.6g} |")
A("\n### Train mean / std used for scaling\n")
A("| Column | Mean | Std |\n|---|---|---|")
for k in CONT:
    A(f"| `{k}` | {pm['tabular_scaler']['mean'][k]:.6g} | {pm['tabular_scaler']['scale'][k]:.6g} |")
A("")

A("## 7. LSTM representation (different from the tabular lag matrix)\n")
A("- Tabular models see **one flat row** of 18 independent columns. The model does not know that `*_lag2`, `*_lag1` and the current column are consecutive years of the same variable.")
A("- The LSTM sees a **sequence** with shape (samples, timesteps, features) = (n, 3, 5): the same 5 base features at t-2, t-1, t, in time order, "
  "and it carries a hidden state from step to step.")
A(f"- LSTM features: {', '.join('`'+b+'`' for b in pm['lstm']['features'])}.")
A("- LSTM imputer/scaler are fit on the stacked TRAIN steps (all from years <= 2021) and saved separately from the tabular ones.\n")

A("## 8. Old vs new dataset\n")
A("| Item | Old `data/processed` | New `data/processed_v2` |\n|---|---|---|")
A(f"| Train rows | {int(old_audit['train_rows'])} | {dm['splits']['train']['rows']} |")
A(f"| Val rows | {int(old_audit['val_rows'])} | {dm['splits']['val']['rows']} |")
A(f"| Test rows | {int(old_audit['test_rows'])} | {dm['splits']['test']['rows']} |")
A(f"| Model features | {int(old_audit['feature_count'])} | {dm['n_features']} |")
A("| Lag/target logic | row `shift()` (misaligned around missing years) | exact calendar-year joins |")
A("| Added indicators | GDP only | + unemployment, internet access, health expenditure |")
A("")
A("## 9. Limitations to cite\n")
A("- The test set is a single origin year (2023 -> 2024), so test metrics have large sampling uncertainty.")
A("- Strict alignment shrinks the training set (the old pipeline had more rows, but with misaligned lags).")
A("- Feature importance and coefficients describe prediction, not causation.")

text = "\n".join(L) + "\n"

# write with backup of any previous version
os.makedirs(BACKUPS, exist_ok=True)
if os.path.exists(DICT_PATH):
    if open(DICT_PATH).read() == text:
        print("Dictionary already up to date (identical).")
    else:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.copy(DICT_PATH, os.path.join(BACKUPS, f"data_dictionary_{stamp}.md"))
        print("Previous version backed up to results/_backups/")
        open(DICT_PATH, "w").write(text)
else:
    open(DICT_PATH, "w").write(text)
print("Wrote", DICT_PATH)
print(text[:900], "...")

# VALIDATION
txt = open(DICT_PATH).read()
absent = [f for f in FEATURES + [T, "Year", "TargetYear"] if f"`{f}`" not in txt]
assert not absent, f"Columns missing from the dictionary: {absent}"
for n, s in dm["splits"].items():
    assert f"| {n} |" in txt and str(s["rows"]) in txt, f"Split {n} missing"
assert txt.count("| `") >= len(FEATURES), "Feature table looks incomplete"
assert os.path.exists(OLD_DICT), "Old dictionary should still exist (we never touch it)"
print("\nCELL 25 PASSED")

In [ ]:
# ==================================================
# CELL 26 — FINAL PHASE 1 AUDIT + SECURITY CLEANUP + GIT COMMIT/PUSH
#
# GOAL:
#   (A) re-verify the whole preprocessing output independently,
#   (B) build the OLD vs NEW comparison table,
#   (C) scan for secrets (token-shaped strings) and fix notebook outputs,
#   (D) commit Phase 1 to branch modeling-phase and (optionally) push.
#
# WHY:
#   Nothing moves to modeling until the data is proven clean, and nothing
#   is committed until it is proven free of credentials.
#
# GIT COMMANDS USED (all explained):
#   git status --short   READ-ONLY   shows changed files
#   git add -- <paths>   CHANGES     puts named files in the next commit (not everything)
#   git commit -m ...    CHANGES     saves a snapshot locally
#   git fetch / ls-remote READ-ONLY  looks at GitHub without changing your files
#   git push             CHANGES     uploads your commit to GitHub (never --force)
#
# TOKEN HANDLING:
#   The push asks for a token through getpass (hidden). It is given to git
#   through a temporary GIT_ASKPASS script and an environment variable, never
#   put in a URL, never written to .git/config, never printed. Leave the box
#   EMPTY to skip pushing (the commit is still made locally).
#
# EXPECTED OUTPUT:
#   audit table (all PASS), OLD vs NEW table, scan result "0 hits",
#   commit hash, then push confirmation or "push skipped", "CELL 26 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - AssertionError audit check FAILED | a Phase 1 file changed or is stale | read the FAIL row; re-run the cell it names (22, 23 or 24), then 26
#   - Commit identity is still the placeholder | CELL 16/16B not run in this repo | run CELL 16 and 16B, then 26
#   - Secret-like text found / everything unstaged | a token-shaped string is in a file (location printed, text is not) | delete or clear that file/notebook output, revoke the token on GitHub, re-run 26
#   - Input is not shaped like a GitHub token | you pasted extra text | copy only the token with the copy icon; or press Enter to skip the push
#   - PUSH NOT CONFIRMED: ... commit(s) you do not have | GitHub has newer commits | run `!git pull --rebase origin modeling-phase` in the project folder, then re-run 26
#   - fatal: Authentication failed / 403 | token expired or missing the repo scope | create a NEW classic token with the repo box ticked; paste it again
# ==================================================
import os, sys, re, json, importlib, datetime, subprocess
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell first."
BRANCH_NAME = "modeling-phase"
GITHUB_USERNAME = "sharifa5852"
RUN_PUSH_PROMPT = True          # True = ask for a token; empty answer skips the push
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
os.makedirs(SRC_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Write reusable git helper module (src/git_utils.py). Reused again in CELL 42.
# ---------------------------------------------------------------------------
GIT_UTILS = r'''
"""Git helpers. They never print, store or log a token."""
import os, re, json, shutil, stat, subprocess, tempfile, datetime

TOKEN_RE = re.compile(r"gh[pousr]_[A-Za-z0-9]{4,}|github_pat_[A-Za-z0-9_]{4,}")
URL_CRED_RE = re.compile(r"https?://[^/\s@:]+:[^/\s@]+@|https?://[^/\s@]{20,}@")
TEXT_EXT = {".py", ".md", ".txt", ".json", ".csv", ".ipynb", ".yml", ".yaml", ".cfg", ".toml", ".ini", ".sh", ".gitignore", ""}
MAX_COMMIT_MB = 50


def git(root, *args, env=None):
    return subprocess.run(["git", *args], cwd=root, capture_output=True, text=True, env=env)


def scan_paths(root, relpaths):
    """Return [(relpath, line_no, kind)] for secret-looking text. Matches are NEVER returned or printed."""
    hits = []
    for rel in relpaths:
        full = os.path.join(root, rel)
        if not os.path.isfile(full) or os.path.splitext(rel)[1].lower() not in TEXT_EXT:
            continue
        if os.path.getsize(full) > 50_000_000:
            continue
        try:
            with open(full, "r", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f, 1):
                    if TOKEN_RE.search(line):
                        hits.append((rel, i, "token-shaped string"))
                    elif URL_CRED_RE.search(line):
                        hits.append((rel, i, "credentials inside a URL"))
        except OSError:
            pass
    return hits


def all_repo_files(root):
    out = git(root, "ls-files", "-z", "--cached", "--others", "--exclude-standard").stdout
    return [p for p in out.split("\0") if p]


def clear_notebook_outputs(path, backup_dir):
    """Backup then remove every output from a notebook (keeps the code)."""
    os.makedirs(backup_dir, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    shutil.copy(path, os.path.join(backup_dir, os.path.basename(path) + f".{stamp}.bak"))
    nb = json.load(open(path, encoding="utf-8"))
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "code":
            cell["outputs"] = []
            cell["execution_count"] = None
        md = cell.get("metadata", {})
        md.pop("outputId", None)
        md.pop("colab", None)
    json.dump(nb, open(path, "w", encoding="utf-8"), indent=1)


def stage_and_check(root, pathspecs, backup_dir):
    """git add named paths only, scan what is staged, unstage everything if a secret is found."""
    existing = [p for p in pathspecs if os.path.exists(os.path.join(root, p))]
    if existing:
        r = git(root, "add", "--", *existing)
        if r.returncode != 0:
            raise RuntimeError("git add failed: " + r.stderr.strip()[:300])
    staged = [p for p in git(root, "diff", "--cached", "--name-only", "-z").stdout.split("\0") if p]
    big = [p for p in staged if os.path.getsize(os.path.join(root, p)) > MAX_COMMIT_MB * 1024 * 1024]
    for p in big:
        git(root, "reset", "-q", "--", p)
        print(f"WARNING: {p} is larger than {MAX_COMMIT_MB} MB and was NOT staged.")
    staged = [p for p in staged if p not in big]
    hits = scan_paths(root, staged)
    if hits:
        git(root, "reset", "-q")
        raise RuntimeError("Secret-looking text found, everything unstaged. Locations: "
                           + "; ".join(f"{p}:{n} ({k})" for p, n, k in hits[:10]))
    return staged


def commit(root, message):
    status = git(root, "diff", "--cached", "--quiet")
    if status.returncode == 0:
        return None                         # nothing staged
    r = git(root, "commit", "-m", message)
    if r.returncode != 0:
        raise RuntimeError("git commit failed: " + (r.stderr or r.stdout).strip()[:400])
    return git(root, "rev-parse", "--short", "HEAD").stdout.strip()


def _askpass_env(username, token):
    d = tempfile.mkdtemp(prefix="askpass_")
    script = os.path.join(d, "askpass.sh")
    with open(script, "w") as f:
        f.write('#!/bin/sh\ncase "$1" in\n  Username*) printf "%s\\n" "$GH_PUSH_USER" ;;\n  *) printf "%s\\n" "$GH_PUSH_TOKEN" ;;\nesac\n')
    os.chmod(script, stat.S_IRWXU)
    env = dict(os.environ)
    env.update({"GIT_ASKPASS": script, "GIT_TERMINAL_PROMPT": "0", "GH_PUSH_USER": username, "GH_PUSH_TOKEN": token})
    return env, d


def push_with_token(root, branch, username, token, remote="origin"):
    """Push (never force). Returns dict(ok, local, remote, message). The token never appears in the output."""
    if not re.fullmatch(r"(gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,})", token or ""):
        return {"ok": False, "message": "Input is not shaped like a GitHub token (nothing was sent)."}
    url = git(root, "remote", "get-url", remote).stdout.strip()
    if URL_CRED_RE.search(url):
        return {"ok": False, "message": "Remote URL contains credentials. Re-run CELL 16 first."}
    env, tmp = _askpass_env(username, token)
    try:
        clean = lambda s: (s or "").replace(token, "***")
        f = git(root, "fetch", remote, branch, env=env)
        if f.returncode == 0:
            counts = git(root, "rev-list", "--left-right", "--count", f"HEAD...{remote}/{branch}").stdout.split()
            if len(counts) == 2 and int(counts[1]) > 0:
                return {"ok": False, "message": f"{remote}/{branch} has {counts[1]} commit(s) you do not have. Not pushing. "
                                                "Run: git pull --rebase (or ask for help) and push again."}
        p = git(root, "push", "-u", remote, branch, env=env)
        out = clean(p.stdout) + clean(p.stderr)
        local = git(root, "rev-parse", "HEAD").stdout.strip()
        ls = git(root, "ls-remote", remote, f"refs/heads/{branch}", env=env)
        remote_sha = ls.stdout.split()[0] if ls.stdout.split() else ""
        ok = p.returncode == 0 and remote_sha == local
        return {"ok": ok, "local": local[:8], "remote": remote_sha[:8], "message": out.strip()[-600:]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
        env["GH_PUSH_TOKEN"] = ""
'''
with open(os.path.join(SRC_DIR, "git_utils.py"), "w") as f:
    f.write(GIT_UTILS)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import git_utils as gu
importlib.reload(gu)

P = lambda *a: os.path.join(PROJECT_ROOT, *a)
V2, TABLES, PREP = P("data", "processed_v2"), P("results", "tables"), P("models", "preprocessing")
SEED = 42

# ---------------------------------------------------------------------------
# (A) INDEPENDENT AUDIT OF THE PREPROCESSING OUTPUT
# ---------------------------------------------------------------------------
spec = json.load(open(os.path.join(V2, "feature_spec.json")))
FEATURES, FLAGS, T = spec["feature_cols"], spec["flag_cols"], spec["target_col"]
CONT = [c for c in FEATURES if c not in FLAGS]
raw_tr, raw_va, raw_te = [pd.read_csv(os.path.join(V2, f"{n}.csv")) for n in ("train", "val", "test")]
imp = {n: pd.read_csv(os.path.join(V2, f"{n}_imputed.csv")) for n in ("train", "val", "test")}
sc = {n: pd.read_csv(os.path.join(V2, f"{n}_scaled.csv")) for n in ("train", "val", "test")}
orig = pd.read_csv(P("data", "raw", "whr_worldbank_merged.csv"))
hs = dict(zip(zip(orig.Code, orig.Year), orig.HappinessScore))
tab_imp = joblib.load(os.path.join(PREP, "tabular_imputer.joblib"))
tab_sc = joblib.load(os.path.join(PREP, "tabular_scaler.joblib"))
seq_sc = joblib.load(os.path.join(PREP, "lstm_scaler.joblib"))
npz = np.load(os.path.join(V2, "lstm_sequences.npz"))

checks = []


def add(name, ok, detail=""):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})


frames = {"train": raw_tr, "val": raw_va, "test": raw_te}
add("split sizes 842/133/133", [len(raw_tr), len(raw_va), len(raw_te)] == [842, 133, 133], [len(raw_tr), len(raw_va), len(raw_te)])
add("chronology train<val<test (origin years)", raw_tr.Year.max() < raw_va.Year.min() and raw_va.Year.max() < raw_te.Year.min(),
    f"{raw_tr.Year.max()} < {raw_va.Year.min()}, {raw_va.Year.max()} < {raw_te.Year.min()}")
keyset = lambda d: set(zip(d.Code, d.Year))
add("no row overlap between splits", not (keyset(raw_tr) & keyset(raw_va)) and not (keyset(raw_tr) & keyset(raw_te)) and not (keyset(raw_va) & keyset(raw_te)))
add("TargetYear == Year+1 everywhere", all((d.TargetYear == d.Year + 1).all() for d in frames.values()))
add("test target year is 2024, train targets stop at 2022", raw_te.TargetYear.eq(2024).all() and raw_tr.TargetYear.max() == 2022)
tgt_ok = all(np.allclose([hs.get((c, y + 1), np.nan) for c, y in zip(d.Code, d.Year)], d[T]) for d in frames.values())
add("target == raw happiness at t+1 (re-derived from original file)", tgt_ok)
pers_ok = all(np.allclose([hs.get((c, y), np.nan) for c, y in zip(d.Code, d.Year)], imp[n]["HappinessScore"]) for n, d in frames.items())
add("persistence column == raw happiness at t", pers_ok)
add("feature order identical in spec / imputed / scaled",
    all(list(imp[n].columns[5:]) == FEATURES == list(sc[n].columns[5:]) for n in imp), f"{len(FEATURES)} features")
add("no NaN/inf in imputed or scaled inputs",
    all(np.isfinite(imp[n][FEATURES].to_numpy(float)).all() and np.isfinite(sc[n][FEATURES].to_numpy(float)).all() for n in imp))
add("flags are 0/1 in imputed and scaled", all(set(np.unique(d[FLAGS].to_numpy())) <= {0.0, 1.0} for d in list(imp.values()) + list(sc.values())))
add("target unscaled (scaled files hold raw target)", all(np.allclose(sc[n][T], frames[n][T]) for n in sc) and 0 < raw_tr[T].min() and raw_tr[T].max() < 10)
add("tabular scaler fit on TRAIN only (n_samples_seen_==842)", int(tab_sc.n_samples_seen_) == 842, int(tab_sc.n_samples_seen_))
add("imputer medians == train medians", np.allclose(tab_imp.statistics_, raw_tr[FEATURES].median().to_numpy(float)))
add("train scaled columns have mean 0 / std 1", np.allclose(sc["train"][CONT].mean(), 0, atol=1e-9) and np.allclose(sc["train"][CONT].std(ddof=0), 1, atol=1e-9))
add("LSTM scaler fit on 842*3 train steps only", int(seq_sc.n_samples_seen_) == 842 * 3, int(seq_sc.n_samples_seen_))
add("LSTM arrays shapes (n,3,5), finite", npz["X_train"].shape == (842, 3, 5) and npz["X_val"].shape == (133, 3, 5)
    and npz["X_test"].shape == (133, 3, 5) and all(np.isfinite(npz[k]).all() for k in ("X_train", "X_val", "X_test")))
g = gu.git(PROJECT_ROOT, "status", "--porcelain", "--", "data/processed", "data/raw/whr_worldbank_merged.csv")
add("old data/processed and original raw file unmodified", g.stdout.strip() == "", repr(g.stdout.strip()))
add("old data dictionary still present", os.path.exists(P("data", "processed", "data_dictionary.md")))
add("new data dictionary present", os.path.exists(os.path.join(V2, "data_dictionary.md")))

audit = pd.DataFrame(checks)
print(audit.to_string(index=False))
audit.to_csv(os.path.join(TABLES, "phase1_final_audit.csv"), index=False)

# ---------------------------------------------------------------------------
# (B) OLD vs NEW comparison table (numbers read from saved files)
# ---------------------------------------------------------------------------
old = pd.read_csv(os.path.join(TABLES, "baseline_audit_old_pipeline.csv")).iloc[0]
dm = json.load(open(os.path.join(V2, "dataset_metadata.json")))
cmp_rows = [
    ("train rows", int(old.train_rows), dm["splits"]["train"]["rows"]),
    ("train countries", int(old.train_countries), dm["splits"]["train"]["countries"]),
    ("val rows", int(old.val_rows), dm["splits"]["val"]["rows"]),
    ("test rows", int(old.test_rows), dm["splits"]["test"]["rows"]),
    ("model features", int(old.feature_count), dm["n_features"]),
    ("raw countries (enrichment keeps all)", int(old.raw_countries), int(pd.read_csv(P("data", "raw", "whr_worldbank_enriched.csv")).Code.nunique())),
    ("test target year", "2024 (row-shift, test origin 2023 only)", "2024 (calendar-aligned, test origin 2023)"),
]
oldnew = pd.DataFrame(cmp_rows, columns=["item", "OLD pipeline", "NEW pipeline"])
oldnew["why different"] = ["strict calendar alignment drops rows whose t-1/t-2/t+1 year is absent", "", "val/test no longer include 2022/2023 rows with a missing neighbouring year",
                           "", "old: 4 lag columns + 3 current columns; new: 5 base features x (t,t-1,t-2) + 3 flags", "left merge keeps every country", ""]
print("\nOLD vs NEW:\n", oldnew.to_string(index=False))
oldnew.to_csv(os.path.join(TABLES, "phase1_old_vs_new.csv"), index=False)

# ---------------------------------------------------------------------------
# (C) SECURITY CLEANUP
# ---------------------------------------------------------------------------
cfg_text = open(P(".git", "config")).read()
cfg_hits = len(gu.TOKEN_RE.findall(cfg_text)) + len(gu.URL_CRED_RE.findall(cfg_text))
repo_hits = gu.scan_paths(PROJECT_ROOT, gu.all_repo_files(PROJECT_ROOT))
print("\nSecret-like strings in .git/config:", cfg_hits)
print("Secret-like strings in repo files (not printed, only location):", len(repo_hits))
for rel, n, kind in repo_hits[:10]:
    print("  ", rel, "line", n, "-", kind)

nb_hits = sorted({rel for rel, _, _ in repo_hits if rel.endswith(".ipynb")})
for rel in nb_hits:                                    # notebooks inside the repo: remove outputs (backup first)
    gu.clear_notebook_outputs(P(rel), P("results", "_backups"))
    print("Cleared outputs of", rel, "(backup in results/_backups/)")
if nb_hits:
    repo_hits = gu.scan_paths(PROJECT_ROOT, gu.all_repo_files(PROJECT_ROOT))
    print("Hits after clearing notebook outputs:", len(repo_hits))
print("~/.git-credentials exists:", os.path.exists(os.path.expanduser("~/.git-credentials")))

# requirements.txt vs installed versions (read-only report)
import importlib.metadata as md
req_path = P("requirements.txt")
if os.path.exists(req_path):
    pinned = dict(l.strip().split("==") for l in open(req_path) if "==" in l)
    diff = {}
    for k, v in pinned.items():
        try:
            if md.version(k) != v:
                diff[k] = (v, md.version(k))
        except md.PackageNotFoundError:
            diff[k] = (v, "not installed")
    print("requirements.txt pins that differ from the running environment:", diff if diff else "none")

# ---------------------------------------------------------------------------
# Phase 1 validation report (numbers come from the variables above)
# ---------------------------------------------------------------------------
report = {"generated": datetime.datetime.now().isoformat(timespec="seconds"), "all_checks_pass": bool((audit.result == "PASS").all()),
          "n_checks": len(audit), "splits": dm["splits"], "n_features": dm["n_features"], "secret_hits_in_repo": len(repo_hits),
          "secret_hits_in_git_config": cfg_hits, "strategy": "STRICT calendar-year join"}
json.dump(report, open(os.path.join(TABLES, "phase1_validation_report.json"), "w"), indent=2)
with open(os.path.join(TABLES, "phase1_validation_report.md"), "w") as f:
    f.write("# Phase 1 validation report\n\n```\n" + audit.to_string(index=False) + "\n```\n\n## Old vs new\n\n```\n" + oldnew.to_string(index=False) + "\n```\n")

assert report["all_checks_pass"], "A Phase 1 audit check FAILED (see table above). Do not continue."
assert cfg_hits == 0 and len(repo_hits) == 0, "Secret-like text found. Fix before any commit."

# ---------------------------------------------------------------------------
# (D) COMMIT (local) and optional PUSH
# ---------------------------------------------------------------------------
assert gu.git(PROJECT_ROOT, "branch", "--show-current").stdout.strip() == BRANCH_NAME, f"Not on branch {BRANCH_NAME}"
who = gu.git(PROJECT_ROOT, "config", "--get", "user.email").stdout.strip()
assert who and who != "you@example.com", "Commit identity is still the placeholder. Re-run CELL 16 / 16B."
PHASE1_PATHS = ["data/raw/whr_worldbank_enriched.csv", "data/processed_v2", "results/tables", "results/figures",
                "models/preprocessing", "src/git_utils.py", "requirements.txt"]
staged = gu.stage_and_check(PROJECT_ROOT, PHASE1_PATHS, P("results", "_backups"))
print("\nStaged files:", len(staged))
print(gu.git(PROJECT_ROOT, "status", "--short").stdout[:1500] or "(nothing)")
sha = gu.commit(PROJECT_ROOT, "Phase 1: enriched data, calendar-aligned supervised table, train-only preprocessing, audits")
print("Commit:", sha if sha else "nothing new to commit (already saved)")

if RUN_PUSH_PROMPT:
    from getpass import getpass
    tok = getpass("GitHub token to push (hidden; leave EMPTY to skip the push): ").strip()
    if tok:
        res = gu.push_with_token(PROJECT_ROOT, BRANCH_NAME, GITHUB_USERNAME, tok)
        del tok
        print("PUSH OK" if res["ok"] else "PUSH NOT CONFIRMED", "| local", res.get("local"), "| remote", res.get("remote"))
        print(res["message"])
        assert res["ok"], "Push not confirmed. Read the message above (see the error table)."
    else:
        print("Push skipped. Everything is committed locally; run CELL 26 again later to push.")
print("\nCELL 26 PASSED")

In [ ]:
# ==================================================
# CELL 27 — MODELING ENVIRONMENT AND EVALUATION UTILITIES
#
# GOAL:
#   Create src/modeling_utils.py (metrics, loaders, prediction-file checks),
#   create the output folders, load all Phase 1 outputs, and prove they are
#   ready: files exist, feature order matches, target unscaled, inputs finite.
#
# WHY:
#   Every model (CELLS 28-41) must load the same data and score with the same
#   functions. One copy of the code means one place for a bug, and the file
#   is committed to GitHub so the work is reproducible.
#
# KEY IDEAS:
#   MAE  = average absolute error in score points (easy to explain)
#   RMSE = square root of average squared error (punishes big misses)
#   R2   = 1 - model error / error of always predicting the mean
#          (can be negative; it is NOT "accuracy")
#   Every prediction row carries Country, Code, Year, TargetYear and y_true,
#   so a prediction can always be traced back to the exact real observation.
#
# EXPECTED OUTPUT:
#   environment summary, shapes (842,18)/(133,18)/(133,18), seed 42,
#   a table of checks (all PASS), "CELL 27 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - ImportError: xgboost | package missing in this runtime | !pip install xgboost, Runtime > Restart session, SETUP, then 27
#   - FileNotFoundError: ... processed_v2 | Phase 1 not finished or wrong folder | run SETUP; re-run CELLS 22-24
#   - A Phase 1 readiness check FAILED | stale or edited Phase 1 files | read the FAIL row; re-run CELL 22-24
#   - Metric self-test assertion | src/modeling_utils.py edited by hand | re-run CELL 27 (it rewrites the file)
# ==================================================
import os, sys, json, importlib, platform, shutil
import importlib.metadata as md

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
if not os.path.isdir(PROJECT_ROOT):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run the Step 1 Drive-mount cell, then re-run this cell."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
os.makedirs(SRC_DIR, exist_ok=True)

MODELING_UTILS = r'''
"""Shared helpers for Phase 2 (loading, metrics, prediction-file validation). No test labels are used for fitting anywhere here."""
import os, json, shutil, datetime, platform
import importlib.metadata as md
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
KEYS = ["Code", "Year", "TargetYear"]
SPLITS = ("train", "val", "test")


def make_paths(root):
    P = {"root": root, "v2": os.path.join(root, "data", "processed_v2"), "prep": os.path.join(root, "models", "preprocessing"),
         "models": os.path.join(root, "models"), "pred": os.path.join(root, "results", "predictions"),
         "tables": os.path.join(root, "results", "tables"), "figures": os.path.join(root, "results", "figures"),
         "backups": os.path.join(root, "results", "_backups"), "raw": os.path.join(root, "data", "raw")}
    for k in ("models", "pred", "tables", "figures", "backups"):
        os.makedirs(P[k], exist_ok=True)
    os.makedirs(os.path.join(P["models"], "lstm"), exist_ok=True)
    return P


def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    assert y_true.shape == y_pred.shape and y_true.ndim == 1, f"shape mismatch {y_true.shape} vs {y_pred.shape}"
    assert np.isfinite(y_true).all() and np.isfinite(y_pred).all(), "NaN/inf in metric inputs"
    return {"MAE": float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2": float(r2_score(y_true, y_pred)), "n": int(len(y_true))}


def _backup(path, P):
    if os.path.exists(path):
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.copy(path, os.path.join(P["backups"], f"{stamp}_{os.path.basename(path)}"))


def save_csv(df, path, P):
    """Write a CSV. If a different version exists, back it up first. Identical content is left alone."""
    if os.path.exists(path):
        old = pd.read_csv(path)
        if old.shape == df.shape and list(old.columns) == list(df.columns):
            same = True
            for c in df.columns:
                if pd.api.types.is_numeric_dtype(df[c]) and pd.api.types.is_numeric_dtype(old[c]):
                    same &= bool(np.allclose(old[c].to_numpy(float), df[c].to_numpy(float), atol=1e-9, equal_nan=True))
                else:
                    same &= bool((old[c].astype(str).to_numpy() == df[c].astype(str).to_numpy()).all())
            if same:
                return path
        _backup(path, P)
    df.to_csv(path, index=False)
    return path


def save_json(obj, path, P):
    if os.path.exists(path):
        if json.load(open(path)) == json.loads(json.dumps(obj, default=str)):
            return path
        _backup(path, P)
    json.dump(obj, open(path, "w"), indent=2, default=str)
    return path


def save_joblib(obj, path, P):
    import joblib
    _backup(path, P)
    joblib.dump(obj, path)
    return path


def load_phase1(P):
    """Load spec, imputed (unscaled), scaled, raw splits and the 'truth' tables."""
    spec = json.load(open(os.path.join(P["v2"], "feature_spec.json")))
    F, T = spec["feature_cols"], spec["target_col"]
    d = {"spec": spec, "features": F, "target": T, "raw": {}, "imp": {}, "sc": {}, "truth": {}}
    for s in SPLITS:
        d["raw"][s] = pd.read_csv(os.path.join(P["v2"], f"{s}.csv"))
        d["imp"][s] = pd.read_csv(os.path.join(P["v2"], f"{s}_imputed.csv"))
        d["sc"][s] = pd.read_csv(os.path.join(P["v2"], f"{s}_scaled.csv"))
        t = d["imp"][s][["Country", "Code", "Year", "TargetYear"]].copy()
        t["y_true"] = d["imp"][s][T].to_numpy(float)
        t["persistence"] = d["imp"][s][spec["persistence_col"]].to_numpy(float)
        d["truth"][s] = t
    return d


def X(d, kind, split):
    """kind: 'imp' = imputed, unscaled (trees);  'sc' = imputed + scaled (linear)."""
    return d[kind][split][d["features"]]


def pred_frame(truth, y_pred, split, model):
    f = truth[["Country", "Code", "Year", "TargetYear", "y_true"]].copy().reset_index(drop=True)
    f.insert(4, "split", split)
    f.insert(5, "model", model)
    f["y_pred"] = np.asarray(y_pred, float)
    return f


def validate_pred_frame(pf, truth, split, label=""):
    """Prediction rows must match the real observations one-to-one by (Code, Year, TargetYear)."""
    tag = f"[{label} {split}]"
    assert pf["y_pred"].notna().all() and np.isfinite(pf["y_pred"].to_numpy(float)).all(), f"{tag} NaN/inf prediction"
    assert not pf.duplicated(KEYS).any(), f"{tag} duplicate prediction rows"
    m = pf[KEYS + ["y_true"]].merge(truth[KEYS + ["y_true"]], on=KEYS, how="outer", suffixes=("_p", "_t"), indicator=True)
    assert (m["_merge"] == "both").all(), f"{tag} rows do not match the real test/val observations ({(m['_merge'] != 'both').sum()} differ)"
    assert np.allclose(m["y_true_p"], m["y_true_t"], atol=1e-12), f"{tag} y_true differs from the real target"
    assert (pf["TargetYear"] == pf["Year"] + 1).all(), f"{tag} TargetYear != Year+1"
    return True


def split_metrics(pf, split):
    s = pf[pf["split"] == split]
    return metrics(s["y_true"], s["y_pred"])


def check_phase1(P, d):
    """Return a DataFrame of PASS/FAIL checks. Raises nothing; the caller decides."""
    import joblib
    rows = []
    add = lambda n, ok, det="": rows.append({"check": n, "result": "PASS" if ok else "FAIL", "detail": str(det)})
    F, T, spec = d["features"], d["target"], d["spec"]
    need = [f"{s}{k}.csv" for s in SPLITS for k in ("", "_imputed", "_scaled")] + ["feature_spec.json", "lstm_sequences.npz", "feature_metadata.json", "preprocessing_metadata.json"]
    add("all Phase 1 data files exist", all(os.path.exists(os.path.join(P["v2"], f)) for f in need))
    add("all 4 preprocessing artifacts exist", all(os.path.exists(os.path.join(P["prep"], f)) for f in
        ("tabular_imputer.joblib", "tabular_scaler.joblib", "lstm_imputer.joblib", "lstm_scaler.joblib")))
    add("shapes 842/133/133 x 18 features", [len(d["imp"][s]) for s in SPLITS] == [842, 133, 133] and len(F) == 18, [len(d["imp"][s]) for s in SPLITS])
    add("feature order equal in spec, imputed, scaled", all(list(d[k][s].columns[5:]) == F for k in ("imp", "sc") for s in SPLITS))
    add("target unscaled (equals raw split target)", all(np.allclose(d[k][s][T], d["raw"][s][T]) for k in ("imp", "sc") for s in SPLITS))
    add("target range looks like a 0-10 score", 0 < d["raw"]["train"][T].min() and d["raw"]["train"][T].max() < 10,
        (round(float(d["raw"]["train"][T].min()), 2), round(float(d["raw"]["train"][T].max()), 2)))
    add("no NaN/inf in model inputs", all(np.isfinite(d[k][s][F].to_numpy(float)).all() for k in ("imp", "sc") for s in SPLITS))
    add("feature/target separation (target not among features)", T not in F and "TargetYear" not in F and "has_lag3" not in F)
    add("test targets are all 2024, val targets 2023, train targets <= 2022",
        d["truth"]["test"].TargetYear.eq(2024).all() and d["truth"]["val"].TargetYear.eq(2023).all() and d["truth"]["train"].TargetYear.max() == 2022)
    keyset = lambda s: set(zip(d["truth"][s].Code, d["truth"][s].Year))
    add("train/val/test rows do not overlap", not (keyset("train") & keyset("val")) and not (keyset("train") & keyset("test")) and not (keyset("val") & keyset("test")))
    sc = joblib.load(os.path.join(P["prep"], "tabular_scaler.joblib"))
    add("scaler was fit on exactly the training rows (842)", int(sc.n_samples_seen_) == 842, int(sc.n_samples_seen_))
    return pd.DataFrame(rows)


def load_seq(P, L, splits=("train", "val", "test")):
    """Load LSTM arrays (n, L, 5), oldest -> newest. Only the requested splits are read from disk."""
    d_truth = {}
    out = {}
    if L == 3:
        z = np.load(os.path.join(P["v2"], "lstm_sequences.npz"))
        for s in splits:
            t = pd.read_csv(os.path.join(P["v2"], f"{s}_imputed.csv"))[["Country", "Code", "Year", "TargetYear"]]
            out[s] = {"X": z[f"X_{s}"].astype(np.float32), "y": z[f"y_{s}"].astype(np.float32), "keys": t.reset_index(drop=True), "row_idx": np.arange(len(t))}
    elif L == 4:
        z = np.load(os.path.join(P["v2"], "lstm_seq_len4.npz"), allow_pickle=False)
        for s in splits:
            t = pd.read_csv(os.path.join(P["v2"], f"{s}_imputed.csv"))[["Country", "Code", "Year", "TargetYear"]]
            idx = z[f"row_idx_{s}"]
            out[s] = {"X": z[f"X_{s}"].astype(np.float32), "y": z[f"y_{s}"].astype(np.float32), "keys": t.iloc[idx].reset_index(drop=True), "row_idx": idx}
    else:
        raise ValueError("L must be 3 or 4")
    for s in splits:
        o = out[s]
        assert o["X"].ndim == 3 and o["X"].shape[1] == L and o["X"].shape[2] == 5, f"bad shape {o['X'].shape}"
        assert len(o["X"]) == len(o["y"]) == len(o["keys"]) and np.isfinite(o["X"]).all()
    return out


def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform(), "seed": SEED, "cpu_count": os.cpu_count()}
    for pkg in ("numpy", "pandas", "scikit-learn", "xgboost", "tensorflow", "keras", "joblib", "matplotlib"):
        try:
            info[pkg] = md.version(pkg)
        except Exception:
            info[pkg] = "not installed"
    info["gpu_tool_present"] = shutil.which("nvidia-smi") is not None
    return info
'''
with open(os.path.join(SRC_DIR, "modeling_utils.py"), "w") as f:
    f.write(MODELING_UTILS)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)

import random
import numpy as np
import pandas as pd
random.seed(mu.SEED); np.random.seed(mu.SEED)
os.environ["PYTHONHASHSEED"] = str(mu.SEED)

# xgboost is needed from CELL 30; verify now so a missing package is found early (no reinstall unless missing)
try:
    import xgboost
except ImportError:
    raise ImportError("xgboost is not installed. Run  !pip install xgboost  in a new cell, then Runtime > Restart session, "
                      "and re-run from the Step 1 cell.")

P = mu.make_paths(PROJECT_ROOT)
env = mu.env_info()
print("Environment:", json.dumps(env, indent=1))
mu.save_json(env, os.path.join(P["tables"], "environment_phase2.json"), P)

D = mu.load_phase1(P)
chk = mu.check_phase1(P, D)
print("\n", chk.to_string(index=False))
print("\nShapes (rows x features): train", mu.X(D, "imp", "train").shape, "| val", mu.X(D, "imp", "val").shape, "| test", mu.X(D, "imp", "test").shape)
print("Target (unscaled) train mean/std:", round(D["truth"]["train"].y_true.mean(), 3), round(D["truth"]["train"].y_true.std(), 3))

# metric sanity test with numbers anyone can check by hand
m = mu.metrics([1.0, 2.0, 3.0], [1.0, 2.0, 4.0])
assert abs(m["MAE"] - 1/3) < 1e-12 and abs(m["RMSE"] - np.sqrt(1/3)) < 1e-12 and abs(m["R2"] - 0.5) < 1e-12, m
print("Metric self-test OK:", {k: round(v, 4) for k, v in m.items() if k != "n"})
assert (chk.result == "PASS").all(), "A Phase 1 readiness check FAILED. Re-run CELLS 22-24 before modeling."
print("\nCELL 27 PASSED")

In [ ]:
# ==================================================
# CELL 28 — NAIVE PERSISTENCE + LINEAR REGRESSION
#
# GOAL:
#   Build the two simplest benchmarks on the SAME validation/test rows as
#   every later model, and save their predictions.
#
# WHY:
#   Persistence ("next year = this year") is the minimum bar. A forecasting
#   model that cannot beat it is not adding information. Linear Regression
#   shows how much a plain weighted sum of the 18 features adds.
#
# WHAT HAPPENS:
#   Persistence: no training. prediction = HappinessScore at year t.
#   Linear Regression: model.fit(X_train, y_train) learns one weight per
#   feature on TRAINING rows; model.predict(X) applies them.
#   Uses the SCALED features (linear models need comparable scales).
#   The target stays in score points.
#   Ridge (a regularised variant) is checked on VALIDATION ONLY as a side
#   result. The headline baseline stays plain Linear Regression so there is
#   nothing tuned in it. No test number is computed for Ridge.
#
# EXPECTED OUTPUT:
#   a metrics table (val and test), top coefficients, ridge validation
#   table, saved file names, "CELL 28 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - Run CELL 27 first | src/modeling_utils.py missing | run CELL 27
#   - ValueError: feature names ... do not match | feature order differs from Phase 1 | re-run CELLS 23 and 27, then 28
#   - AssertionError: rows do not match the real test/val observations | prediction rows misaligned | re-run CELL 28 from the top (do not edit the frames)
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
from sklearn.linear_model import LinearRegression, Ridge

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
assert os.path.exists(os.path.join(SRC_DIR, "modeling_utils.py")), "Run CELL 27 first (it creates src/modeling_utils.py)."
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]
np.random.seed(mu.SEED)

# ---------- 1) Persistence ----------
pers = {s: mu.pred_frame(truth[s], truth[s]["persistence"].to_numpy(), s, "Persistence") for s in ("val", "test")}
pers_all = pd.concat(pers.values(), ignore_index=True)

# ---------- 2) Linear Regression (scaled features) ----------
Xtr, ytr = mu.X(D, "sc", "train"), truth["train"]["y_true"].to_numpy()
lr = LinearRegression().fit(Xtr, ytr)
lr_all = pd.concat([mu.pred_frame(truth[s], lr.predict(mu.X(D, "sc", s)), s, "Linear Regression") for s in ("val", "test")], ignore_index=True)

# ---------- 3) Ridge: VALIDATION-ONLY side check ----------
rows = []
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    r = Ridge(alpha=a).fit(Xtr, ytr)
    m = mu.metrics(truth["val"]["y_true"], r.predict(mu.X(D, "sc", "val")))
    rows.append({"alpha": a, "val_MAE": m["MAE"], "val_RMSE": m["RMSE"], "val_R2": m["R2"]})
ridge_tbl = pd.DataFrame(rows)

# ---------- validate + report ----------
for s in ("val", "test"):
    mu.validate_pred_frame(pers_all[pers_all.split == s], truth[s], s, "Persistence")
    mu.validate_pred_frame(lr_all[lr_all.split == s], truth[s], s, "LinearRegression")
tab = pd.DataFrame([{"model": n, "split": s, **{k: v for k, v in mu.split_metrics(pf, s).items()}}
                    for n, pf in (("Persistence", pers_all), ("Linear Regression", lr_all)) for s in ("val", "test")])
print(tab.round(4).to_string(index=False))
coef = pd.DataFrame({"feature": F, "coefficient": lr.coef_}).assign(abs_coef=lambda d: d.coefficient.abs()).sort_values("abs_coef", ascending=False)
print("\nLargest linear-regression weights (scaled features; a weight is NOT a causal effect):")
print(coef.head(6).round(4).to_string(index=False), "\nintercept:", round(float(lr.intercept_), 4))
print("\nRidge, VALIDATION only (side check):\n", ridge_tbl.round(4).to_string(index=False))

# ---------- save ----------
mu.save_csv(pers_all, os.path.join(P["pred"], "persistence_predictions.csv"), P)
mu.save_csv(lr_all, os.path.join(P["pred"], "linear_regression_predictions.csv"), P)
mu.save_csv(tab, os.path.join(P["tables"], "persistence_linear_metrics.csv"), P)
mu.save_csv(coef.drop(columns="abs_coef"), os.path.join(P["tables"], "linear_regression_coefficients.csv"), P)
mu.save_csv(ridge_tbl, os.path.join(P["tables"], "ridge_sensitivity_validation_only.csv"), P)
mu.save_joblib(lr, os.path.join(P["models"], "linear_regression.joblib"), P)
mu.save_json({"model": "LinearRegression()", "features": F, "input": "scaled (train-fitted scaler)", "fit_rows": "train only", "seed": mu.SEED,
              "persistence_rule": "prediction for year t+1 = HappinessScore at year t"}, os.path.join(P["models"], "linear_regression_config.json"), P)

# ---------- validation ----------
lr2 = joblib.load(os.path.join(P["models"], "linear_regression.joblib"))
assert list(lr2.feature_names_in_) == F, "Saved model feature order differs from Phase 1 order"
assert np.allclose(lr2.predict(mu.X(D, "sc", "test")), lr_all[lr_all.split == "test"].y_pred), "Reloaded model gives different predictions"
assert lr.n_features_in_ == 18 and len(lr_all) == 266 and len(pers_all) == 266
assert np.isfinite(tab[["MAE", "RMSE", "R2"]].to_numpy(float)).all()
print("\nSaved: persistence_predictions.csv, linear_regression_predictions.csv, linear_regression.joblib, metrics tables")
print("CELL 28 PASSED")

In [ ]:
# ==================================================
# CELL 29 — RANDOM FOREST (validation-based selection)
#
# GOAL:
#   Train a Random Forest regressor, choose its settings on VALIDATION, and
#   save the model and its validation/test predictions.
#
# WHY:
#   Random Forest averages many decision trees and handles non-linear
#   relationships. It does not need scaled inputs, so it uses the imputed,
#   UNSCALED features.
#
# WHAT HAPPENS:
#   1) grid of 18 small settings (depth x min leaf size x features per split);
#      each is FIT on training rows and SCORED on validation rows.
#   2) the setting with the lowest validation RMSE is selected (ties -> the
#      first in the grid list, so the choice is deterministic).
#   3) the chosen model stays fitted on TRAIN ONLY (validation labels are not
#      used to train it; they are reserved for stacking in CELL 39-40).
#   4) test is predicted once, after the choice. The grid table saved to disk
#      contains validation columns only.
#
# EXPECTED OUTPUT:
#   validation grid (sorted), chosen settings, val/test metrics, feature
#   importances (prediction usefulness, NOT causation), "CELL 29 PASSED".
#   Runtime: well under 2 minutes.
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - Run time much longer than 2 minutes | CPU is busy or 1 core | set N_TREES = 150 at the top, re-run
#   - AssertionError selection table has test columns | grid code was edited | restore the cell text from this notebook and re-run
#   - PicklingError / joblib error saving | Drive hiccup | re-run the cell; check free Drive space
# ==================================================
import os, sys, json, importlib, time
from itertools import product
import numpy as np, pandas as pd, joblib
from sklearn.ensemble import RandomForestRegressor

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]

N_TREES = 300
GRID = {"max_depth": [4, 8, None], "min_samples_leaf": [1, 3, 5], "max_features": [0.5, 1.0]}   # declared before running
Xtr, ytr = mu.X(D, "imp", "train"), truth["train"]["y_true"].to_numpy()
Xva, yva = mu.X(D, "imp", "val"), truth["val"]["y_true"].to_numpy()

t0 = time.time()
rows = []
for i, (md_, msl, mf) in enumerate(product(GRID["max_depth"], GRID["min_samples_leaf"], GRID["max_features"])):
    m = RandomForestRegressor(n_estimators=N_TREES, max_depth=md_, min_samples_leaf=msl, max_features=mf,
                              n_jobs=-1, random_state=mu.SEED).fit(Xtr, ytr)
    v = mu.metrics(yva, m.predict(Xva))
    rows.append({"config_id": i, "max_depth": "None" if md_ is None else md_, "min_samples_leaf": msl, "max_features": mf,
                 "val_MAE": v["MAE"], "val_RMSE": v["RMSE"], "val_R2": v["R2"]})
grid = pd.DataFrame(rows)
best = grid.sort_values(["val_RMSE", "config_id"]).iloc[0]
print(f"Grid of {len(grid)} configs fitted in {time.time()-t0:.1f}s")
print(grid.sort_values(["val_RMSE", "config_id"]).head(6).round(4).to_string(index=False))

params = {"n_estimators": N_TREES, "max_depth": None if best.max_depth == "None" else int(best.max_depth),
          "min_samples_leaf": int(best.min_samples_leaf), "max_features": float(best.max_features),
          "n_jobs": -1, "random_state": mu.SEED}
rf = RandomForestRegressor(**params).fit(Xtr, ytr)                     # TRAIN rows only
pf = pd.concat([mu.pred_frame(truth[s], rf.predict(mu.X(D, "imp", s)), s, "Random Forest") for s in ("val", "test")], ignore_index=True)
for s in ("val", "test"):
    mu.validate_pred_frame(pf[pf.split == s], truth[s], s, "RandomForest")
res = pd.DataFrame([{"split": s, **mu.split_metrics(pf, s)} for s in ("val", "test")])
print("\nSelected:", {k: v for k, v in params.items() if k not in ("n_jobs",)})
print(res.round(4).to_string(index=False))
imp = pd.DataFrame({"feature": F, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
print("\nTop importances (how much the forest USED a feature; not a causal effect):")
print(imp.head(6).round(4).to_string(index=False))

mu.save_joblib(rf, os.path.join(P["models"], "random_forest.joblib"), P)
mu.save_json({**params, "selected_on": "validation RMSE", "grid": GRID, "fit_rows": "train only", "input": "imputed, unscaled"},
             os.path.join(P["models"], "random_forest_config.json"), P)
mu.save_csv(grid, os.path.join(P["tables"], "rf_grid_results.csv"), P)          # validation metrics only
mu.save_csv(imp, os.path.join(P["tables"], "rf_feature_importance.csv"), P)
mu.save_csv(pf, os.path.join(P["pred"], "random_forest_predictions.csv"), P)

rf2 = joblib.load(os.path.join(P["models"], "random_forest.joblib"))
assert list(rf2.feature_names_in_) == F
assert np.allclose(rf2.predict(mu.X(D, "imp", "test")), pf[pf.split == "test"].y_pred)
assert not any("test" in c.lower() for c in grid.columns), "Selection table must not contain test metrics"
assert len(pf) == 266 and np.isfinite(res[["MAE", "RMSE", "R2"]].to_numpy(float)).all()
print("\nCELL 29 PASSED")

In [ ]:
# ==================================================
# CELL 30 — XGBOOST (validation-based selection)
#
# GOAL:
#   Train gradient-boosted trees (XGBoost), select its settings on VALIDATION,
#   save model + predictions.
#
# WHY:
#   XGBoost builds trees one after another, each correcting the previous
#   ones' mistakes. It is a strong default for tables of numbers. Like the
#   forest, it uses the imputed, UNSCALED features.
#
# WHAT HAPPENS (same protocol as the Random Forest):
#   12 predeclared settings (depth x learning rate x number of trees), fit on
#   TRAIN, scored on VALIDATION; lowest validation RMSE is chosen (ties ->
#   first in the list). The chosen model stays fitted on train only. Test is
#   predicted only after the choice. No early stopping (it would reuse the
#   validation labels a second time).
#   CPU only: tree_method="hist".
#
# EXPECTED OUTPUT:
#   validation grid, chosen settings, metrics, importances, "CELL 30 PASSED".
#   Runtime: about a minute or less on Colab CPU.
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - TypeError/XGBoostError about an unknown parameter | xgboost version differs from 3.4.1 | !pip install xgboost==3.4.1, restart, SETUP, re-run 27-30
#   - NaN or constant predictions | features contain NaN | re-run CELLS 23 and 27 (check_phase1 must PASS)
# ==================================================
import os, sys, json, importlib, time
from itertools import product
import numpy as np, pandas as pd, joblib
from xgboost import XGBRegressor

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]

GRID = {"max_depth": [2, 3, 4], "learning_rate": [0.03, 0.1], "n_estimators": [200, 500]}       # declared before running
FIXED = {"subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0, "tree_method": "hist", "n_jobs": -1,
         "random_state": mu.SEED, "verbosity": 0}
Xtr, ytr = mu.X(D, "imp", "train"), truth["train"]["y_true"].to_numpy()
Xva, yva = mu.X(D, "imp", "val"), truth["val"]["y_true"].to_numpy()

t0 = time.time()
rows = []
for i, (d_, lr_, n_) in enumerate(product(GRID["max_depth"], GRID["learning_rate"], GRID["n_estimators"])):
    m = XGBRegressor(max_depth=d_, learning_rate=lr_, n_estimators=n_, **FIXED).fit(Xtr, ytr)
    v = mu.metrics(yva, m.predict(Xva))
    rows.append({"config_id": i, "max_depth": d_, "learning_rate": lr_, "n_estimators": n_,
                 "val_MAE": v["MAE"], "val_RMSE": v["RMSE"], "val_R2": v["R2"]})
grid = pd.DataFrame(rows)
best = grid.sort_values(["val_RMSE", "config_id"]).iloc[0]
print(f"Grid of {len(grid)} configs fitted in {time.time()-t0:.1f}s")
print(grid.sort_values(["val_RMSE", "config_id"]).head(6).round(4).to_string(index=False))

params = {"max_depth": int(best.max_depth), "learning_rate": float(best.learning_rate), "n_estimators": int(best.n_estimators), **FIXED}
xgb = XGBRegressor(**params).fit(Xtr, ytr)                              # TRAIN rows only
pf = pd.concat([mu.pred_frame(truth[s], xgb.predict(mu.X(D, "imp", s)), s, "XGBoost") for s in ("val", "test")], ignore_index=True)
for s in ("val", "test"):
    mu.validate_pred_frame(pf[pf.split == s], truth[s], s, "XGBoost")
res = pd.DataFrame([{"split": s, **mu.split_metrics(pf, s)} for s in ("val", "test")])
print("\nSelected:", {k: v for k, v in params.items() if k in ("max_depth", "learning_rate", "n_estimators")})
print(res.round(4).to_string(index=False))
imp = pd.DataFrame({"feature": F, "importance": xgb.feature_importances_}).sort_values("importance", ascending=False)
print("\nTop importances (prediction usefulness, NOT causal effects):")
print(imp.head(6).round(4).to_string(index=False))

mu.save_joblib(xgb, os.path.join(P["models"], "xgboost.joblib"), P)
mu.save_json({**params, "selected_on": "validation RMSE", "grid": GRID, "fit_rows": "train only", "input": "imputed, unscaled"},
             os.path.join(P["models"], "xgboost_config.json"), P)
mu.save_csv(grid, os.path.join(P["tables"], "xgb_grid_results.csv"), P)         # validation metrics only
mu.save_csv(imp, os.path.join(P["tables"], "xgb_feature_importance.csv"), P)
mu.save_csv(pf, os.path.join(P["pred"], "xgboost_predictions.csv"), P)

x2 = joblib.load(os.path.join(P["models"], "xgboost.joblib"))
assert list(x2.feature_names_in_) == F
assert np.allclose(x2.predict(mu.X(D, "imp", "test")), pf[pf.split == "test"].y_pred, atol=1e-6)
assert not any("test" in c.lower() for c in grid.columns), "Selection table must not contain test metrics"
assert len(pf) == 266 and np.isfinite(res[["MAE", "RMSE", "R2"]].to_numpy(float)).all()
print("\nCELL 30 PASSED")

In [ ]:
# ==================================================
# CELL 31 — BASELINE MODEL COMPARISON
#
# GOAL:
#   Put Persistence, Linear Regression, Random Forest and XGBoost in one
#   table (validation and test), save it, draw comparison figures, and save
#   one combined predictions file with correct identifiers.
#
# WHY:
#   One table with identical rows for every model is the only fair comparison.
#   Every prediction row is checked against the real observation
#   (Code, Year, TargetYear) before any number is reported.
#
# HOW TO READ IT:
#   MAE / RMSE are in happiness points (lower is better). R2 is very high for
#   every model here because countries differ a lot from each other, so
#   judge models by MAE/RMSE and by how they compare with Persistence, not by
#   R2 alone. VALIDATION numbers of tuned models are slightly optimistic
#   (the settings were chosen on those rows); TEST numbers are the honest ones.
#
# EXPECTED OUTPUT:
#   wide comparison table, two figures saved, "CELL 31 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - Missing prediction files | CELL 28, 29 or 30 not run in this folder | run them, then 31
#   - AssertionError alignment / y_true differs | a prediction file was altered or is from an older split | re-run CELLS 28-30, then 31
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
truth = D["truth"]

FILES = {"Persistence": "persistence_predictions.csv", "Linear Regression": "linear_regression_predictions.csv",
         "Random Forest": "random_forest_predictions.csv", "XGBoost": "xgboost_predictions.csv"}
missing = [f for f in FILES.values() if not os.path.exists(os.path.join(P["pred"], f))]
assert not missing, f"Missing prediction files {missing}. Run CELLS 28-30 first."

preds = {n: pd.read_csv(os.path.join(P["pred"], f)) for n, f in FILES.items()}
for n, pf in preds.items():                       # alignment against the real observations
    for s in ("val", "test"):
        mu.validate_pred_frame(pf[pf.split == s], truth[s], s, n)

cfg = {"Persistence": "prediction = HappinessScore at t",
       "Linear Regression": "OLS, 18 scaled features",
       "Random Forest": json.dumps({k: v for k, v in json.load(open(os.path.join(P["models"], "random_forest_config.json"))).items()
                                    if k in ("n_estimators", "max_depth", "min_samples_leaf", "max_features")}),
       "XGBoost": json.dumps({k: v for k, v in json.load(open(os.path.join(P["models"], "xgboost_config.json"))).items()
                              if k in ("n_estimators", "max_depth", "learning_rate")})}
rows = []
for n, pf in preds.items():
    v, t = mu.split_metrics(pf, "val"), mu.split_metrics(pf, "test")
    rows.append({"model": n, "val_MAE": v["MAE"], "val_RMSE": v["RMSE"], "val_R2": v["R2"],
                 "test_MAE": t["MAE"], "test_RMSE": t["RMSE"], "test_R2": t["R2"], "n_val": v["n"], "n_test": t["n"], "config": cfg[n]})
res = pd.DataFrame(rows)
pers_rmse = float(res.loc[res.model == "Persistence", "test_RMSE"].iloc[0])
res["test_RMSE_vs_persistence_pct"] = 100 * (res["test_RMSE"] / pers_rmse - 1)
print(res.drop(columns="config").round(4).to_string(index=False))
print("\n(test_RMSE_vs_persistence_pct: negative = smaller error than Persistence, positive = larger. Single test year, no significance test.)")

mu.save_csv(res, os.path.join(P["tables"], "baseline_results.csv"), P)
allp = pd.concat(preds.values(), ignore_index=True)
mu.save_csv(allp, os.path.join(P["pred"], "baseline_predictions_all.csv"), P)

# ---- Figure 1: error bars ----
colors = {"val": "#9ecae1", "test": "#08519c"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = np.arange(len(res)); w = 0.38
for ax, met in zip(axes, ["MAE", "RMSE", "R2"]):
    ax.bar(x - w/2, res[f"val_{met}"], w, label="validation", color=colors["val"])
    ax.bar(x + w/2, res[f"test_{met}"], w, label="test", color=colors["test"])
    ax.set_xticks(x); ax.set_xticklabels(res.model, rotation=20, ha="right", fontsize=10)
    ax.set_title(f"{met} by model" + (" (points, lower is better)" if met != "R2" else " (higher is better)"), fontsize=12)
    ax.set_ylabel(met, fontsize=11); ax.set_xlabel("Model", fontsize=11)
    if met == "R2":
        ax.set_ylim(max(0, res[["val_R2", "test_R2"]].min().min() - 0.02), 1.0)
    ax.legend(fontsize=9)
fig.suptitle("Baseline comparison: validation (2022 origin) vs test (2023 origin, forecasting 2024)", fontsize=13)
plt.tight_layout()
f1 = os.path.join(P["figures"], "baseline_comparison_bars.png"); fig.savefig(f1, dpi=150); plt.show()

# ---- Figure 2: test predicted vs actual ----
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
lo = min(truth["test"].y_true.min(), min(p[p.split == "test"].y_pred.min() for p in preds.values())) - 0.2
hi = max(truth["test"].y_true.max(), max(p[p.split == "test"].y_pred.max() for p in preds.values())) + 0.2
for ax, (n, pf) in zip(axes.ravel(), preds.items()):
    s = pf[pf.split == "test"]
    ax.scatter(s.y_true, s.y_pred, s=18, alpha=0.7, color="#08519c")
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="perfect forecast")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(f"{n}: test RMSE {mu.metrics(s.y_true, s.y_pred)['RMSE']:.3f}", fontsize=12)
    ax.set_xlabel("Actual happiness in 2024", fontsize=11); ax.set_ylabel("Predicted happiness for 2024", fontsize=11); ax.legend(fontsize=9)
fig.suptitle("Test set: predicted vs actual (133 countries)", fontsize=13)
plt.tight_layout()
f2 = os.path.join(P["figures"], "baseline_test_predicted_vs_actual.png"); fig.savefig(f2, dpi=150); plt.show()

assert os.path.exists(f1) and os.path.exists(f2)
assert np.isfinite(res[["val_MAE", "val_RMSE", "val_R2", "test_MAE", "test_RMSE", "test_R2"]].to_numpy(float)).all()
assert (res.n_val == 133).all() and (res.n_test == 133).all()
print("\nCELL 31 PASSED")

In [ ]:
# ==================================================
# CELL 32 — BASELINE VALIDATION AND CHECKPOINT
#
# GOAL:
#   Re-check everything Phase 2 produced so far, from the files on disk, and
#   write a short completion report.
#
# CHECKS:
#   - model files exist and RELOAD; saved feature order == Phase 1 order
#   - reloaded models reproduce the saved test predictions exactly
#   - prediction files: 133 + 133 rows each, no duplicates, y_true == real target
#   - metrics in baseline_results.csv == metrics recomputed from the files
#   - the selection (grid) tables contain NO test columns
#   - Phase 1 readiness checks still pass
#
# EXPECTED OUTPUT: a PASS table, a small report, "CELL 32 PASSED".
# This cell does NOT declare a winner; it only prints observed numbers.
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - A baseline check FAILED | model/prediction mismatch (e.g. mixed runs) | re-run CELLS 28-31 in order, then 32
#   - reloaded model does not reproduce predictions | model file from a different run | re-run the cell that creates that model, then 31 and 32
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]

rows = []
def add(name, ok, detail=""):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})

p1 = mu.check_phase1(P, D)
add("Phase 1 readiness checks still pass", (p1.result == "PASS").all())

MODELS = {"Linear Regression": ("linear_regression.joblib", "sc", "linear_regression_predictions.csv"),
          "Random Forest": ("random_forest.joblib", "imp", "random_forest_predictions.csv"),
          "XGBoost": ("xgboost.joblib", "imp", "xgboost_predictions.csv")}
for name, (mf, kind, pf_name) in MODELS.items():
    mp, pp = os.path.join(P["models"], mf), os.path.join(P["pred"], pf_name)
    add(f"{name}: model and prediction files exist", os.path.exists(mp) and os.path.exists(pp))
    m = joblib.load(mp)
    add(f"{name}: saved feature order == Phase 1 order", list(m.feature_names_in_) == F)
    pf = pd.read_csv(pp)
    for s in ("val", "test"):
        sub = pf[pf.split == s]
        mu.validate_pred_frame(sub, truth[s], s, name)
        add(f"{name}: {s} rows aligned (133, unique, y_true real)", len(sub) == 133)
        again = m.predict(mu.X(D, kind, s))
        add(f"{name}: reloaded model reproduces saved {s} predictions", np.allclose(again, sub.y_pred.to_numpy(), atol=1e-6))
pers = pd.read_csv(os.path.join(P["pred"], "persistence_predictions.csv"))
for s in ("val", "test"):
    sub = pers[pers.split == s]
    mu.validate_pred_frame(sub, truth[s], s, "Persistence")
    add(f"Persistence: {s} == HappinessScore at t", np.allclose(sub.y_pred, truth[s].persistence.to_numpy()))

res = pd.read_csv(os.path.join(P["tables"], "baseline_results.csv"))
allp = pd.concat([pd.read_csv(os.path.join(P["pred"], f)) for f in ("persistence_predictions.csv", "linear_regression_predictions.csv",
                                                                     "random_forest_predictions.csv", "xgboost_predictions.csv")])
ok = True
for _, r in res.iterrows():
    for s in ("val", "test"):
        sub = allp[(allp.model == r.model) & (allp.split == s)]
        m = mu.metrics(sub.y_true, sub.y_pred)
        ok &= all(np.isclose(m[k], r[f"{s}_{k}"], atol=1e-9) for k in ("MAE", "RMSE", "R2"))
add("baseline_results.csv == metrics recomputed from prediction files", ok)
add("all metrics finite", np.isfinite(res[[c for c in res.columns if c.startswith(("val_", "test_")) and "pct" not in c]].to_numpy(float)).all())
for f in ("rf_grid_results.csv", "xgb_grid_results.csv", "ridge_sensitivity_validation_only.csv"):
    g = pd.read_csv(os.path.join(P["tables"], f))
    add(f"{f}: selection table has no test columns", not any("test" in c.lower() for c in g.columns))
add("scaler artifact fit on 842 training rows", int(joblib.load(os.path.join(P["prep"], "tabular_scaler.joblib")).n_samples_seen_) == 842)

out = pd.DataFrame(rows)
print(out.to_string(index=False))
n_fail = int((out.result == "FAIL").sum())

order = res.sort_values("test_RMSE")[["model", "test_MAE", "test_RMSE", "test_R2"]]
print("\nObserved ordering by test RMSE (descriptive only; one test year, no significance test):")
print(order.round(4).to_string(index=False))
report = {"models_built": list(res.model), "n_checks": len(out), "n_failed": n_fail,
          "test_rows": 133, "val_rows": 133, "features": len(F), "baseline_table": res.drop(columns="config").round(6).to_dict(orient="records")}
mu.save_json(report, os.path.join(P["tables"], "baseline_completion_report.json"), P)
mu.save_csv(out, os.path.join(P["tables"], "baseline_validation_checks.csv"), P)
assert n_fail == 0, "A baseline check FAILED. See the table; do not continue until fixed."
print("\nCELL 32 PASSED")

In [ ]:
# ==================================================
# CELL 33 — LSTM SEQUENCE FEASIBILITY (length 3 vs 4)
#
# GOAL:
#   Count, from the real calendar-aligned data, how many LSTM samples exist
#   for sequence length 3 and length 4, per split, and show exactly how a
#   tabular row differs from an LSTM sequence.
#
# TABULAR LAG MATRIX vs LSTM SEQUENCE (the key idea):
#   Tabular (RF/XGBoost/Linear): ONE flat row of 18 numbers. Column names say
#     "lag2/lag1/current", but the model treats them as unrelated columns.
#   LSTM: an array shaped (timesteps, features) = (L, 5): the same 5 variables
#     for t-(L-1) ... t, in time order. The LSTM reads step by step and keeps a
#     memory (hidden state) of earlier steps.
#
# SAMPLE RULE (STRICT, as decided in CELL 18):
#   a sample needs REAL rows for t-(L-1) ... t AND a real target at t+1.
#   Length 4 therefore needs one more real year (t-3) than length 3.
#
# DECISION RULE (set now, before any model is trained):
#   Both lengths are kept as candidates. The choice between them is made in
#   CELL 36 from VALIDATION error only. This cell just reports feasibility.
#
# EXPECTED OUTPUT:
#   a table of samples/countries per split and length, whether val/test rows
#   are identical for both lengths, an example row in both representations,
#   "CELL 33 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - has_lag3 disagrees with calendar check | supervised table older than enriched_filled.csv | re-run CELLS 19-23, then 33
#   - KeyError (Code, Year) | enriched_filled.csv missing/stale | re-run CELL 19
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)

filled = pd.read_csv(os.path.join(P["v2"], "enriched_filled.csv"))
present = set(zip(filled.Code, filled.Year))                       # real (country, year) observations
LENGTHS = (3, 4)


def eligible(df, L):
    """True where years t-(L-1)..t and t+1 are all real observations."""
    return np.array([all((c, y - k) in present for k in range(L)) and (c, y + 1) in present for c, y in zip(df.Code, df.Year)])


rows, masks = [], {}
for s in ("train", "val", "test"):
    df = D["raw"][s]
    for L in LENGTHS:
        m = eligible(df, L)
        masks[(s, L)] = m
        rows.append({"split": s, "seq_len": L, "samples": int(m.sum()), "countries": int(df[m].Code.nunique()),
                     "pct_of_length3": None})
tbl = pd.DataFrame(rows)
for s in ("train", "val", "test"):
    base = tbl[(tbl.split == s) & (tbl.seq_len == 3)].samples.iloc[0]
    tbl.loc[tbl.split == s, "pct_of_length3"] = (100 * tbl[tbl.split == s].samples / base).round(1)
print(tbl.to_string(index=False))

# cross-check with the has_lag3 marker saved in Phase 1
for s in ("train", "val", "test"):
    assert int(D["raw"][s].has_lag3.sum()) == int(masks[(s, 4)].sum()), f"{s}: has_lag3 disagrees with calendar check"
    assert masks[(s, 3)].all(), f"{s}: every Phase 1 sample must be valid for length 3"

same_val = set(D["raw"]["val"][masks[("val", 4)]].Code) == set(D["raw"]["val"].Code)
same_test = set(D["raw"]["test"][masks[("test", 4)]].Code) == set(D["raw"]["test"].Code)
print(f"\nValidation rows identical for length 3 and 4: {same_val} | Test rows identical: {same_test}")
if same_val and same_test:
    print("=> Length 4 only shrinks the TRAINING set; validation and test are the same 133 + 133 countries for both lengths,")
    print("   so lengths can be compared fairly and the stacking step needs no row intersection.")
else:
    print("=> Validation/test rows DIFFER between lengths. Later cells compare on the common rows and report the reduction.")

# ---- one example in both representations (unscaled, imputed values) ----
imp = D["imp"]["test"]
r = imp.iloc[0]
BASE = D["spec"]["base_features"]
print(f"\nExample: {r.Country}, origin year {int(r.Year)} -> target year {int(r.TargetYear)}")
print("TABULAR row (what Random Forest / XGBoost see): 18 independent columns; the 15 continuous ones are shown grouped by lag:")
for suf, lab in [("", "t"), ("_lag1", "t-1"), ("_lag2", "t-2")]:
    print(f"  {lab:>3}:", {b[:16]: round(float(r[b + suf]), 3) for b in BASE})
seq = np.array([[r[b + suf] for b in BASE] for suf in ("_lag2", "_lag1", "")])
print("LSTM sequence (3 steps x 5 features, oldest -> newest), unscaled:")
print(pd.DataFrame(seq, columns=[b[:14] for b in BASE], index=["t-2", "t-1", "t"]).round(3).to_string())

out = {"samples": tbl.to_dict(orient="records"), "val_identical_across_lengths": bool(same_val), "test_identical_across_lengths": bool(same_test),
       "rule": "STRICT calendar rows t-(L-1)..t and t+1 must be real observations",
       "decision": "both lengths are candidates; the final length is chosen on VALIDATION in CELL 36"}
mu.save_json(out, os.path.join(P["tables"], "lstm_sequence_feasibility.json"), P)
mu.save_csv(tbl, os.path.join(P["tables"], "lstm_sequence_feasibility.csv"), P)
assert tbl[(tbl.split == "train") & (tbl.seq_len == 4)].samples.iloc[0] >= 400, "Too few length-4 training samples"
print("\nCELL 33 PASSED")

In [ ]:
# ==================================================
# CELL 34 — LSTM SEQUENCE CONSTRUCTION AND VALIDATION
#
# GOAL:
#   Build the 3-D arrays (samples, timesteps, features) for length 3 AND 4,
#   straight from the calendar-aligned data table, and PROVE they are right.
#
# HOW:
#   For each sample (country c, origin year t), look up the real row for each
#   of the years t-(L-1) ... t of THAT country, oldest first. A lookup by
#   (country, year) can never cross countries or skip a year.
#   Length 3: rebuilt arrays must equal the Phase 1 arrays (lstm_sequences.npz)
#             -> independent proof that both constructions agree.
#   Length 4: needs its own imputer and scaler fitted on TRAINING length-4
#             sequences only; saved as separate artifacts. Row indices point
#             back to the full split, so every sample can be traced.
#   Target = happiness at t+1 (unscaled). Features are the 5 historical
#   variables; missingness flags are not LSTM inputs.
#
# EXPECTED OUTPUT:
#   shapes: L3 (842,3,5)/(133,3,5)/(133,3,5), L4 (690,4,5)/(133,4,5)/(133,4,5),
#   "L3 rebuild == Phase 1 arrays: True", "CELL 34 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - Length-3 rebuild does not match Phase 1 arrays | Phase 1 npz is stale | re-run CELL 23, then 34
#   - length-4 first step is not t-3 | calendar lookup problem | re-run CELL 19 then 34; if it persists, stop and report the printed row
#   - AssertionError loader mismatch | npz written by a previous run | re-run CELL 34 once more
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
BASE = D["spec"]["base_features"]
assert len(BASE) == 5

filled = pd.read_csv(os.path.join(P["v2"], "enriched_filled.csv"))
filled["log_GDPPerCapita_WB"] = np.log(filled["GDPPerCapita_WB"])
look = filled.set_index(["Code", "Year"])[BASE]                         # (country, year) -> 5 raw feature values
raw_target = filled.set_index(["Code", "Year"])["HappinessScore"]


def build(df, L):
    """Return (X_unscaled (n,L,5), y (n,), row_idx) for rows whose L calendar years all exist."""
    ok = np.array([all((c, y - k) in look.index for k in range(L)) and (c, y + 1) in raw_target.index for c, y in zip(df.Code, df.Year)])
    sub = df[ok]
    steps = [look.loc[[(c, y - k) for c, y in zip(sub.Code, sub.Year)], BASE].to_numpy(float) for k in range(L - 1, -1, -1)]   # oldest first
    Xs = np.stack(steps, axis=1)
    y = raw_target.loc[[(c, y + 1) for c, y in zip(sub.Code, sub.Year)]].to_numpy(float)
    return Xs, y, np.where(ok)[0]


raw = {s: D["raw"][s] for s in mu.SPLITS}
U = {L: {s: build(raw[s], L) for s in mu.SPLITS} for L in (3, 4)}      # unscaled arrays

# ---------- length 3: compare with the Phase 1 arrays ----------
imp3 = joblib.load(os.path.join(P["prep"], "lstm_imputer.joblib"))
sc3 = joblib.load(os.path.join(P["prep"], "lstm_scaler.joblib"))
z3 = np.load(os.path.join(P["v2"], "lstm_sequences.npz"))
flat = lambda a: pd.DataFrame(a.reshape(-1, 5), columns=BASE)
S3 = {}
for s in mu.SPLITS:
    X, y, idx = U[3][s]
    assert len(idx) == len(raw[s]), "length 3 must keep every Phase 1 sample"
    S3[s] = sc3.transform(pd.DataFrame(imp3.transform(flat(X)), columns=BASE)).reshape(X.shape)
rebuild_ok = all(np.allclose(S3[s], z3[f"X_{s}"], atol=1e-9) and np.allclose(U[3][s][1], z3[f"y_{s}"]) for s in mu.SPLITS)
print("L3 rebuild == Phase 1 arrays:", rebuild_ok)

# ---------- length 4: new train-only artifacts ----------
Xtr4 = U[4]["train"][0]
imp4 = SimpleImputer(strategy="median", keep_empty_features=True).fit(flat(Xtr4))
sc4 = StandardScaler().fit(pd.DataFrame(imp4.transform(flat(Xtr4)), columns=BASE))        # TRAIN length-4 steps only
S4 = {s: sc4.transform(pd.DataFrame(imp4.transform(flat(U[4][s][0])), columns=BASE)).reshape(U[4][s][0].shape) for s in mu.SPLITS}
assert int(sc4.n_samples_seen_) == len(Xtr4) * 4, "length-4 scaler must see only training steps"

# ---------- validation of both ----------
rows = []
for L, S in ((3, S3), (4, S4)):
    for s in mu.SPLITS:
        X, y, idx = U[L][s]
        keys = raw[s].iloc[idx]
        assert S[s].shape == (len(idx), L, 5) and np.isfinite(S[s]).all(), f"L{L} {s}: bad shape or NaN"
        assert (keys.TargetYear == keys.Year + 1).all()
        assert np.allclose(y, keys["HappinessScore_target"].to_numpy(float)), f"L{L} {s}: target misaligned"
        assert np.allclose(X[:, -1, 0], D["imp"][s].iloc[idx]["HappinessScore"].to_numpy(float)), f"L{L} {s}: last step is not year t"
        if L == 4:                                                  # step 0 must be exactly year t-3 of the same country
            t3 = [look.loc[(c, yv - 3), "HappinessScore"] for c, yv in zip(keys.Code, keys.Year)]
            assert np.allclose(X[:, 0, 0], t3), "length-4 first step is not t-3"
        rows.append({"seq_len": L, "split": s, "shape": str(S[s].shape), "samples": len(idx), "countries": int(keys.Code.nunique())})
print(pd.DataFrame(rows).to_string(index=False))
assert rebuild_ok, "Length-3 rebuild does not match Phase 1 arrays"

# ---------- save length-4 artifacts (length-3 already saved in Phase 1) ----------
np.savez_compressed(os.path.join(P["v2"], "lstm_seq_len4.npz"),
                    **{f"X_{s}": S4[s] for s in mu.SPLITS}, **{f"y_{s}": U[4][s][1] for s in mu.SPLITS},
                    **{f"row_idx_{s}": U[4][s][2] for s in mu.SPLITS}, features=np.array(BASE), timesteps=np.array(["t-3", "t-2", "t-1", "t"]))
mu.save_joblib(imp4, os.path.join(P["prep"], "lstm4_imputer.joblib"), P)
mu.save_joblib(sc4, os.path.join(P["prep"], "lstm4_scaler.joblib"), P)
mu.save_json({"seq_len_4": {"fit_rows": "train length-4 sequences only", "n_train_samples": int(len(Xtr4)),
                            "imputer_medians": dict(zip(BASE, map(float, imp4.statistics_))),
                            "scaler_mean": dict(zip(BASE, map(float, sc4.mean_))), "scaler_scale": dict(zip(BASE, map(float, sc4.scale_)))},
              "order": "oldest -> newest", "features": BASE, "target": "happiness at t+1, unscaled"},
             os.path.join(P["v2"], "lstm_sequence_metadata.json"), P)

# the loader used by CELLS 35-39 must give the same arrays
for L, S in ((3, S3), (4, S4)):
    ld = mu.load_seq(P, L)
    assert all(np.allclose(ld[s]["X"], S[s].astype(np.float32), atol=1e-6) for s in mu.SPLITS), f"loader mismatch for L{L}"
print("\nSaved: lstm_seq_len4.npz, lstm4_imputer/scaler.joblib, lstm_sequence_metadata.json")
print("CELL 34 PASSED")

In [ ]:
# ==================================================
# CELL 35 — LSTM TRAINING (predeclared candidates, validation only)
#
# GOAL:
#   Train a small LSTM for each of 6 predeclared designs (sequence length 3
#   or 4 x 1-2 LSTM layers x 16-32 units) and save each model, its training
#   history and its VALIDATION metrics. The TEST arrays are never loaded here.
#
# HOW AN LSTM WORKS (short):
#   It reads the sequence one year at a time (oldest first). At each step it
#   updates a small memory vector. The last memory goes through a small dense
#   layer that outputs the forecast (next-year happiness, in score points).
#
# TRAINING SETUP (identical for all candidates):
#   loss = mean squared error, optimizer = Adam (lr 0.001), batch 32,
#   max 300 epochs, EARLY STOPPING: stop when validation loss has not improved
#   for 25 epochs and go back to the best epoch's weights.
#   The output bias starts at the mean of the TRAIN targets (train data only),
#   so the network does not waste epochs learning the average level.
#   Seed 42 everywhere. CPU only; each candidate takes seconds to ~1 minute.
#
# HONEST NOTE:
#   Early stopping uses the validation rows, so validation numbers of the LSTM
#   are slightly optimistic. Test rows are not touched until CELL 37.
#
# EXPECTED OUTPUT:
#   one line per candidate (epochs, best epoch, val MAE/RMSE/R2), a training
#   curves figure, "CELL 35 PASSED". Runtime: a few minutes.
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - ImportError: TensorFlow | TF not importable | Runtime > Restart session, SETUP, re-run 27 and 35
#   - NaN loss | unstable training | set LR = 3e-4 at the top and re-run 35
#   - ValueError: input shape (None, L, 5) | sequences from an older CELL 34 | re-run CELL 34, then 35
#   - Very slow | CPU runtime | set MAX_EPOCHS = 150 and PATIENCE = 15, re-run; GPU is not needed
#   - E0000 ... NodeDef / retracing WARNING lines | harmless TensorFlow log noise | ignore
# ==================================================
import os, sys, json, importlib, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"       # hide harmless C++ log noise
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
try:
    import tensorflow as tf
    from tensorflow import keras
except ImportError as e:
    raise ImportError("TensorFlow is not available. In Colab it is preinstalled; if you see this, run Runtime > Restart session "
                      "and re-run from the Step 1 cell. (If it persists: !pip install tensorflow)") from e
P = mu.make_paths(PROJECT_ROOT)
assert os.path.exists(os.path.join(P["v2"], "lstm_seq_len4.npz")), "Run CELL 34 first (it builds the sequence files)."

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPUs visible:", len(tf.config.list_physical_devices("GPU")))
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("(determinism flag not available:", type(e).__name__, ")")

CANDIDATES = [                                           # declared before any training
    {"name": "L3_u16", "L": 3, "units": [16], "dropout": 0.1, "dense": 8},
    {"name": "L3_u32", "L": 3, "units": [32], "dropout": 0.2, "dense": 16},
    {"name": "L3_u32x16", "L": 3, "units": [32, 16], "dropout": 0.2, "dense": 16},
    {"name": "L4_u16", "L": 4, "units": [16], "dropout": 0.1, "dense": 8},
    {"name": "L4_u32", "L": 4, "units": [32], "dropout": 0.2, "dense": 16},
    {"name": "L4_u32x16", "L": 4, "units": [32, 16], "dropout": 0.2, "dense": 16},
]
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 32, 300, 25
LSTM_DIR = os.path.join(P["models"], "lstm")


def build_model(L, units, dropout, dense, y_mean):
    keras.utils.set_random_seed(mu.SEED)                  # seeds python, numpy and TensorFlow
    inp = keras.Input(shape=(L, 5))
    x = inp
    for i, u in enumerate(units):
        x = keras.layers.LSTM(u, return_sequences=(i < len(units) - 1), dropout=dropout)(x)
    x = keras.layers.Dense(dense, activation="relu")(x)
    out = keras.layers.Dense(1, bias_initializer=keras.initializers.Constant(float(y_mean)))(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(LR), loss="mse")
    return model


rows, hists = [], {}
data = {L: mu.load_seq(P, L, splits=("train", "val")) for L in (3, 4)}       # NO test arrays are read here
for c in CANDIDATES:
    t0 = time.time()
    d = data[c["L"]]
    Xtr, ytr, Xva, yva = d["train"]["X"], d["train"]["y"], d["val"]["X"], d["val"]["y"]
    assert Xtr.shape[1] == c["L"] and Xva.shape[1] == c["L"]
    model = build_model(c["L"], c["units"], c["dropout"], c["dense"], ytr.mean())
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, min_delta=1e-5)
    h = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_EPOCHS, batch_size=BATCH, shuffle=True, verbose=0, callbacks=[es])
    loss, vloss = h.history["loss"], h.history["val_loss"]
    assert np.isfinite(loss).all() and np.isfinite(vloss).all(), f"{c['name']}: NaN loss (see error table: NaN loss)"
    best_epoch = int(np.argmin(vloss)) + 1
    pv = model(Xva, training=False).numpy().ravel()
    v = mu.metrics(yva, pv)
    model.save(os.path.join(LSTM_DIR, f"{c['name']}.keras"))
    mu.save_json({"loss": [float(x) for x in loss], "val_loss": [float(x) for x in vloss]}, os.path.join(LSTM_DIR, f"{c['name']}_history.json"), P)
    mu.save_json({**c, "lr": LR, "batch": BATCH, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "seed": mu.SEED,
                  "n_params": int(model.count_params()), "n_train": int(len(ytr)), "n_val": int(len(yva))}, os.path.join(LSTM_DIR, f"{c['name']}_config.json"), P)
    hists[c["name"]] = (loss, vloss, best_epoch)
    rows.append({"name": c["name"], "seq_len": c["L"], "units": "x".join(map(str, c["units"])), "dropout": c["dropout"], "n_params": int(model.count_params()),
                 "epochs_run": len(loss), "best_epoch": best_epoch, "n_train": int(len(ytr)), "n_val": int(len(yva)),
                 "val_MAE": v["MAE"], "val_RMSE": v["RMSE"], "val_R2": v["R2"], "seconds": round(time.time() - t0, 1)})
    print(f"{c['name']:>10}: epochs {len(loss):3d}, best epoch {best_epoch:3d}, val MAE {v['MAE']:.4f}  RMSE {v['RMSE']:.4f}  R2 {v['R2']:.4f}  ({time.time()-t0:.0f}s)")
    keras.backend.clear_session()

tbl = pd.DataFrame(rows)
mu.save_csv(tbl, os.path.join(P["tables"], "lstm_candidate_validation.csv"), P)     # validation columns only

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for ax, c in zip(axes.ravel(), CANDIDATES):
    loss, vloss, be = hists[c["name"]]
    ax.plot(range(1, len(loss) + 1), loss, label="train loss")
    ax.plot(range(1, len(vloss) + 1), vloss, label="validation loss")
    ax.axvline(be, color="gray", ls=":", label=f"best epoch {be}")
    ax.set_title(c["name"], fontsize=12); ax.set_xlabel("Epoch", fontsize=11); ax.set_ylabel("MSE loss (points$^2$)", fontsize=11); ax.legend(fontsize=9)
fig.suptitle("LSTM candidates: training curves", fontsize=14)
plt.tight_layout()
fp = os.path.join(P["figures"], "lstm_candidate_training_curves.png"); fig.savefig(fp, dpi=150); plt.show()

assert len(tbl) == 6 and os.path.exists(fp)
assert all(os.path.exists(os.path.join(LSTM_DIR, f"{c['name']}.keras")) for c in CANDIDATES)
assert np.isfinite(tbl[["val_MAE", "val_RMSE", "val_R2"]].to_numpy(float)).all()
assert not any("test" in c.lower() for c in tbl.columns)
print("\nCELL 35 PASSED")

In [ ]:
# ==================================================
# CELL 36 — LSTM VALIDATION COMPARISON AND MODEL SELECTION
#
# GOAL:
#   Re-score the 6 saved candidates from disk on the VALIDATION rows, choose
#   one by a rule fixed in advance, and record the choice.
#
# RULE (fixed before looking at results):
#   lowest validation RMSE; ties -> the candidate listed first. Test data is
#   not loaded in this cell.
#
# WHY RE-SCORE FROM DISK?
#   It proves the saved model files really give the numbers in the table
#   (and would catch a corrupted or mismatched save).
#
# ABOUT NOISE:
#   One seed, 133 validation rows. If the best two candidates are very close
#   the choice is partly luck; the cell prints the gap so you can say so.
#
# EXPECTED OUTPUT:
#   comparison table (MAE/RMSE/R2, epochs, best epoch, sequence length), the
#   selected candidate, a figure, "CELL 36 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - assertion: saved model does not reproduce its validation RMSE | a candidate file is from a different run | re-run CELL 35 completely, then 36
#   - Run CELL 35 first | candidate table missing | run 35
# ==================================================
import os, sys, json, importlib
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"       # hide harmless C++ log noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
from tensorflow import keras
P = mu.make_paths(PROJECT_ROOT)
LSTM_DIR = os.path.join(P["models"], "lstm")
tbl = pd.read_csv(os.path.join(P["tables"], "lstm_candidate_validation.csv"))
assert len(tbl) == 6, "Run CELL 35 first."

data = {L: mu.load_seq(P, L, splits=("val",)) for L in (3, 4)}                  # validation only
pred = {}
for _, r in tbl.iterrows():
    m = keras.models.load_model(os.path.join(LSTM_DIR, f"{r['name']}.keras"))
    v = data[int(r.seq_len)]["val"]
    f = v["keys"].copy()
    f["y_true"] = v["y"].astype(float)
    f["y_pred"] = m.predict(v["X"], verbose=0).ravel().astype(float)
    pred[r["name"]] = f
    keras.backend.clear_session()
    again = mu.metrics(f.y_true, f.y_pred)
    assert abs(again["RMSE"] - r.val_RMSE) < 1e-4, f"{r['name']}: saved model does not reproduce its validation RMSE"

common = set.intersection(*[set(zip(f.Code, f.Year, f.TargetYear)) for f in pred.values()])
rows = []
for _, r in tbl.iterrows():
    f = pred[r["name"]]
    f = f[[k in common for k in zip(f.Code, f.Year, f.TargetYear)]]
    m = mu.metrics(f.y_true, f.y_pred)
    rows.append({"name": r["name"], "seq_len": int(r.seq_len), "units": r.units, "n_params": int(r.n_params), "epochs_run": int(r.epochs_run),
                 "best_epoch": int(r.best_epoch), "n_train": int(r.n_train), "n_val_common": m["n"],
                 "val_MAE": m["MAE"], "val_RMSE": m["RMSE"], "val_R2": m["R2"]})
cmp = pd.DataFrame(rows)
order = cmp.reset_index().sort_values(["val_RMSE", "index"])
best = order.iloc[0]
cmp["selected"] = cmp["name"] == best["name"]
print(cmp.round(4).to_string(index=False))
gap = (order.iloc[1]["val_RMSE"] / best["val_RMSE"] - 1) * 100
print(f"\nSELECTED: {best['name']} (sequence length {int(best['seq_len'])}, val RMSE {best['val_RMSE']:.4f})")
print(f"Runner-up is only {gap:.1f}% worse in validation RMSE." + ("  -> within what one random seed can change; do not over-interpret the choice." if gap < 3 else ""))
print(f"Rows compared: {len(common)} common validation rows (all candidates scored on the same rows).")

cfg = json.load(open(os.path.join(LSTM_DIR, f"{best['name']}_config.json")))
sel = {"name": best["name"], "seq_len": int(best["seq_len"]), "config": cfg, "selection_rule": "lowest validation RMSE on common rows; ties -> first listed",
       "selected_on": "validation only", "best_epoch": int(best["best_epoch"]), "epochs_run": int(best["epochs_run"]),
       "val_MAE": float(best["val_MAE"]), "val_RMSE": float(best["val_RMSE"]), "val_R2": float(best["val_R2"]),
       "n_val_common": int(best["n_val_common"]), "n_train": int(best["n_train"]), "model_path": f"models/lstm/{best['name']}.keras"}
mu.save_json(sel, os.path.join(P["tables"], "lstm_selected.json"), P)
mu.save_csv(cmp, os.path.join(P["tables"], "lstm_validation_comparison.csv"), P)

fig, ax = plt.subplots(figsize=(8, 4.5))
cols = ["#2b8cbe" if s == 3 else "#e6550d" for s in cmp.seq_len]
ax.bar(cmp["name"], cmp["val_RMSE"], color=cols)
ax.set_xlabel("LSTM candidate (blue: length 3, orange: length 4)", fontsize=11); ax.set_ylabel("Validation RMSE (points)", fontsize=11)
ax.set_title("LSTM candidates on validation (lower is better)", fontsize=12); plt.xticks(rotation=20, fontsize=10)
plt.tight_layout()
fp = os.path.join(P["figures"], "lstm_validation_comparison.png"); fig.savefig(fp, dpi=150); plt.show()

assert np.isfinite(cmp[["val_MAE", "val_RMSE", "val_R2"]].to_numpy(float)).all() and os.path.exists(fp)
assert sel["selected_on"] == "validation only" and len(common) >= 100
print("\nCELL 36 PASSED")

In [ ]:
# ==================================================
# CELL 37 — LSTM TEST EVALUATION (once)
#
# GOAL:
#   Load the LSTM selected in CELL 36, predict validation and test, and save
#   the predictions, metrics and figures. The test set is evaluated ONCE.
#
# ONCE-ONLY GUARD:
#   After the first run the cell writes results/tables/lstm_test_evaluation_log.json.
#   Re-running reuses the saved predictions instead of scoring again. To redo it
#   on purpose, set REDO = True (e.g. after you retrain; this is recorded).
#
# EXPECTED OUTPUT:
#   selected model, val/test metrics, two figures (actual vs predicted, training
#   curves), saved files, "CELL 37 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - 'already done once; reusing saved predictions' | the once-only guard | normal; set REDO = True only if you intentionally retrained
# ==================================================
import os, sys, json, importlib, shutil, datetime
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"       # hide harmless C++ log noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
from tensorflow import keras
REDO = False
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
truth = D["truth"]
sel = json.load(open(os.path.join(P["tables"], "lstm_selected.json")))
L = sel["seq_len"]
LOG = os.path.join(P["tables"], "lstm_test_evaluation_log.json")
PRED = os.path.join(P["pred"], "lstm_predictions.csv")
SEL_MODEL = os.path.join(P["models"], "lstm_selected.keras")

if os.path.exists(LOG) and os.path.exists(PRED) and not REDO:
    print("Test evaluation was already done once; reusing saved predictions (set REDO = True to repeat on purpose).")
    pf = pd.read_csv(PRED)
else:
    seq = mu.load_seq(P, L, splits=("val", "test"))
    model = keras.models.load_model(os.path.join(P["root"], sel["model_path"]))
    frames = []
    for s in ("val", "test"):
        idx = seq[s]["row_idx"]
        tsub = truth[s].iloc[idx].reset_index(drop=True)
        assert np.allclose(tsub.y_true, seq[s]["y"]), f"{s}: sequence target differs from the real target"
        frames.append(mu.pred_frame(tsub, model.predict(seq[s]["X"], verbose=0).ravel(), s, "LSTM"))
    pf = pd.concat(frames, ignore_index=True)
    shutil.copy(os.path.join(P["root"], sel["model_path"]), SEL_MODEL)
    mu.save_csv(pf, PRED, P)
    json.dump({"evaluated_at": datetime.datetime.now().isoformat(timespec="seconds"), "model": sel["name"], "seq_len": L,
               "note": "Test set scored once, after selection on validation (CELL 36)."}, open(LOG, "w"), indent=2)

for s in ("val", "test"):
    sub = pf[pf.split == s]
    tsub = truth[s].merge(sub[mu.KEYS], on=mu.KEYS, how="inner")
    mu.validate_pred_frame(sub, tsub, s, "LSTM")
res = pd.DataFrame([{"split": s, "seq_len": L, **mu.split_metrics(pf, s)} for s in ("val", "test")])
print("Selected LSTM:", sel["name"], "| sequence length", L, "| best epoch", sel["best_epoch"])
print(res.round(4).to_string(index=False))
mu.save_csv(res, os.path.join(P["tables"], "lstm_metrics.csv"), P)

# ---- figures ----
s = pf[pf.split == "test"]
lo, hi = min(s.y_true.min(), s.y_pred.min()) - 0.2, max(s.y_true.max(), s.y_pred.max()) + 0.2
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(s.y_true, s.y_pred, s=18, alpha=0.7, color="#08519c")
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="perfect forecast")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Actual happiness in 2024", fontsize=11); ax.set_ylabel("LSTM prediction for 2024", fontsize=11)
ax.set_title(f"LSTM ({sel['name']}) on test: RMSE {res.loc[res.split=='test','RMSE'].iloc[0]:.3f}", fontsize=12); ax.legend(fontsize=9)
plt.tight_layout()
f1 = os.path.join(P["figures"], "lstm_test_predicted_vs_actual.png"); fig.savefig(f1, dpi=150); plt.show()

h = json.load(open(os.path.join(P["models"], "lstm", f"{sel['name']}_history.json")))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(range(1, len(h["loss"]) + 1), h["loss"], label="train loss")
ax.plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], label="validation loss")
ax.axvline(sel["best_epoch"], color="gray", ls=":", label=f"best epoch {sel['best_epoch']}")
ax.set_xlabel("Epoch", fontsize=11); ax.set_ylabel("MSE loss (points$^2$)", fontsize=11); ax.set_title(f"Training history: {sel['name']}", fontsize=12); ax.legend(fontsize=9)
plt.tight_layout()
f2 = os.path.join(P["figures"], "lstm_selected_training_history.png"); fig.savefig(f2, dpi=150); plt.show()

assert os.path.exists(SEL_MODEL) and os.path.exists(f1) and os.path.exists(f2) and os.path.exists(LOG)
assert np.isfinite(res[["MAE", "RMSE", "R2"]].to_numpy(float)).all() and (res.n == 133).all()
print("\nCELL 37 PASSED")

In [ ]:
# ==================================================
# CELL 38 — LSTM COMPLETION REPORT
#
# GOAL:
#   Re-load the saved LSTM from disk, reproduce its saved test predictions,
#   and write a short report: sequence length, sample counts, feature count,
#   final metrics, model path.
#
# EXPECTED OUTPUT: PASS table, the report text, "CELL 38 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - reloaded model does not reproduce saved predictions | model/prediction files from different runs | set REDO = True in 37 (after re-running 35-36) and re-run 37, 38
# ==================================================
import os, sys, json, importlib
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"       # hide harmless C++ log noise
import numpy as np, pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
from tensorflow import keras
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
sel = json.load(open(os.path.join(P["tables"], "lstm_selected.json")))
L = sel["seq_len"]
pf = pd.read_csv(os.path.join(P["pred"], "lstm_predictions.csv"))
met = pd.read_csv(os.path.join(P["tables"], "lstm_metrics.csv"))

rows = []
def add(n, ok, d=""):
    rows.append({"check": n, "result": "PASS" if ok else "FAIL", "detail": str(d)})

model = keras.models.load_model(os.path.join(P["models"], "lstm_selected.keras"))
add("saved model input shape == (None, L, 5)", tuple(model.input_shape) == (None, L, 5), model.input_shape)
seq = mu.load_seq(P, L, splits=("val", "test"))
for s in ("val", "test"):
    again = model.predict(seq[s]["X"], verbose=0).ravel()
    sub = pf[pf.split == s]
    add(f"{s}: reloaded model reproduces saved predictions", np.allclose(again, sub.y_pred.to_numpy(), atol=1e-5))
    tsub = D["truth"][s].merge(sub[mu.KEYS], on=mu.KEYS, how="inner")
    mu.validate_pred_frame(sub, tsub, s, "LSTM")
    add(f"{s}: rows aligned to real observations", len(sub) == len(tsub) == 133, len(sub))
    m = mu.metrics(sub.y_true, sub.y_pred)
    add(f"{s}: metrics file matches recomputation", all(np.isclose(m[k], met[met.split == s][k].iloc[0], atol=1e-9) for k in ("MAE", "RMSE", "R2")))
add("selected on validation only", sel["selected_on"] == "validation only")
add("test evaluated once (log present)", os.path.exists(os.path.join(P["tables"], "lstm_test_evaluation_log.json")))
add("all candidate tables have no test columns", not any("test" in c.lower() for c in pd.read_csv(os.path.join(P["tables"], "lstm_candidate_validation.csv")).columns))
out = pd.DataFrame(rows)
print(out.to_string(index=False))

t = met[met.split == "test"].iloc[0]
v = met[met.split == "val"].iloc[0]
report = {"model": sel["name"], "sequence_length": L, "features_per_step": 5, "train_samples": sel["n_train"], "val_samples": int(v.n), "test_samples": int(t.n),
          "architecture": {k: sel["config"][k] for k in ("units", "dropout", "dense", "lr", "batch", "patience", "seed")}, "best_epoch": sel["best_epoch"],
          "val": {k: float(v[k]) for k in ("MAE", "RMSE", "R2")}, "test": {k: float(t[k]) for k in ("MAE", "RMSE", "R2")},
          "model_path": "models/lstm_selected.keras", "limitations": ["single seed", "small panel for deep learning", "early stopping used the validation rows"]}
mu.save_json(report, os.path.join(P["tables"], "lstm_completion_report.json"), P)
print("\nLSTM completion report:\n", json.dumps(report, indent=1))
assert (out.result == "PASS").all(), "An LSTM check FAILED; see the table."
print("\nCELL 38 PASSED")

In [ ]:
# ==================================================
# CELL 39 — PREPARE STACKING BASE-MODEL PREDICTIONS (temporal holdout design)
#
# GOAL:
#   Collect the predictions of the three base models (Random Forest, XGBoost,
#   LSTM) on VALIDATION (origin 2022 -> target 2023) and TEST (origin 2023 ->
#   target 2024), aligned row by row, ready for the meta-model.
#
# WHAT STACKING MEANS HERE (read this; it is not shuffled-KFold stacking):
#   1. Base models are trained on the TRAINING period only (origin <= 2021).
#   2. They predict the validation year. These predictions are the meta-model's
#      training inputs; the real 2023 scores are its target.
#   3. The same base models (never refitted on validation or test) predict
#      the test year. The meta-model only sees those at the very end.
#   This is a temporal HOLDOUT, not conventional out-of-fold stacking.
#   Reason: shuffling rows across years would mix the future into the past.
#
# TWO VARIANTS (the first is the PRIMARY one, declared now):
#   FIXED    : RF and XGBoost with predeclared settings (below), never tuned.
#              Their validation predictions are therefore not selected-on-val.
#   SELECTED : the settings chosen on validation in CELLS 29-30 (sensitivity check;
#              slightly optimistic because validation chose them).
#   The LSTM is the validation-selected model in both variants (early stopping
#   already used validation, disclosed).
#
# EXPECTED OUTPUT:
#   row counts (133 / 133 for every source, no reduction), correlation
#   between base predictions, saved files, "CELL 39 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - AssertionError row counts changed (not 133) | base predictions misaligned/missing rows | read the printed 'aligned' lines; re-run CELLS 29, 30, 37 then 39
#   - FileNotFoundError lstm_predictions.csv | CELL 37 not run | run 37, then 39
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]

# ---- predeclared settings for the primary (FIXED) variant ----
FIXED_RF = {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 3, "max_features": 0.5, "n_jobs": -1, "random_state": mu.SEED}
FIXED_XGB = {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0,
             "tree_method": "hist", "n_jobs": -1, "random_state": mu.SEED, "verbosity": 0}

Xtr, ytr = mu.X(D, "imp", "train"), truth["train"]["y_true"].to_numpy()
assert len(Xtr) == 842 and D["truth"]["train"].TargetYear.max() == 2022          # base models see the training period only
rf_fix = RandomForestRegressor(**FIXED_RF).fit(Xtr, ytr)
xgb_fix = XGBRegressor(**FIXED_XGB).fit(Xtr, ytr)
fixed = {"rf": rf_fix, "xgb": xgb_fix}
mu.save_joblib(rf_fix, os.path.join(P["models"], "stacking_rf_fixed.joblib"), P)
mu.save_joblib(xgb_fix, os.path.join(P["models"], "stacking_xgb_fixed.joblib"), P)

rd = lambda f: pd.read_csv(os.path.join(P["pred"], f))
src_sel = {"rf": rd("random_forest_predictions.csv"), "xgb": rd("xgboost_predictions.csv")}
lstm = rd("lstm_predictions.csv")


def fixed_frame(name, model):
    return pd.concat([mu.pred_frame(truth[s], model.predict(mu.X(D, "imp", s)), s, name) for s in ("val", "test")], ignore_index=True)


src_fix = {"rf": fixed_frame("RF (fixed)", rf_fix), "xgb": fixed_frame("XGBoost (fixed)", xgb_fix)}


def assemble(srcs, split):
    """Inner-join LSTM, RF and XGBoost predictions on (Code, Year, TargetYear) and check y_true agrees."""
    base = lstm[lstm.split == split][["Country"] + mu.KEYS + ["y_true", "y_pred"]].rename(columns={"y_pred": "lstm"})
    n0 = {"lstm": len(base)}
    for k in ("rf", "xgb"):
        s = srcs[k][srcs[k].split == split]
        n0[k] = len(s)
        base = base.merge(s[mu.KEYS + ["y_true", "y_pred"]].rename(columns={"y_pred": k, "y_true": f"y_true_{k}"}),
                          on=mu.KEYS, how="inner", validate="one_to_one")
        assert np.allclose(base["y_true"], base[f"y_true_{k}"]), f"y_true differs between LSTM and {k}"
        base = base.drop(columns=f"y_true_{k}")
    base.insert(0, "split", split)
    return base, n0


report = {}
for variant, srcs in (("fixed", src_fix), ("selected", src_sel)):
    parts = []
    for s in ("val", "test"):
        f, n0 = assemble(srcs, s)
        mu.validate_pred_frame(f.assign(y_pred=f["lstm"]), truth[s].merge(f[mu.KEYS], on=mu.KEYS), s, f"stack-base-{variant}")
        report[f"{variant}_{s}"] = {"rows_per_source_before": n0, "rows_after_alignment": len(f), "dropped": max(n0.values()) - len(f)}
        parts.append(f)
        print(f"[{variant:8s} {s:4s}] rows per source {n0} -> aligned {len(f)}  (dropped {max(n0.values()) - len(f)})")
    tab = pd.concat(parts, ignore_index=True)[["split", "Country"] + mu.KEYS + ["y_true", "rf", "xgb", "lstm"]]
    mu.save_csv(tab, os.path.join(P["pred"], f"stacking_base_predictions_{variant}.csv"), P)
    v = tab[tab.split == "val"][["rf", "xgb", "lstm"]]
    print(f"   correlation between base predictions on validation ({variant}):\n", v.corr().round(3).to_string())

# standalone metrics of the FIXED base models (information only)
rows = []
for k in ("rf", "xgb"):
    for s in ("val", "test"):
        m = mu.split_metrics(src_fix[k], s)
        rows.append({"model": f"{k} (fixed settings)", "split": s, **m})
fb = pd.DataFrame(rows)
print("\nStandalone metrics of the FIXED base models:\n", fb.round(4).to_string(index=False))
mu.save_csv(fb, os.path.join(P["tables"], "stacking_fixed_base_metrics.csv"), P)
mu.save_json({"primary_variant": "fixed", "FIXED_RF": FIXED_RF, "FIXED_XGB": FIXED_XGB, "lstm": json.load(open(os.path.join(P["tables"], "lstm_selected.json")))["name"],
              "design": "temporal holdout: base models fit on train (origin<=2021); meta-model fit on validation predictions only; test untouched until final scoring",
              "alignment": report}, os.path.join(P["tables"], "stacking_base_config.json"), P)

# ---- checks ----
for variant in ("fixed", "selected"):
    t = pd.read_csv(os.path.join(P["pred"], f"stacking_base_predictions_{variant}.csv"))
    assert not t.duplicated(["split"] + mu.KEYS).any()
    assert (t.split == "val").sum() == 133 and (t.split == "test").sum() == 133, "Row counts changed; see the printed alignment lines"
    assert t[["rf", "xgb", "lstm", "y_true"]].notna().all().all() and np.isfinite(t[["rf", "xgb", "lstm"]].to_numpy(float)).all()
    assert (t[t.split == "val"].TargetYear == 2023).all() and (t[t.split == "test"].TargetYear == 2024).all()
    assert not (set(zip(t[t.split == "val"].Code, t[t.split == "val"].Year)) & set(zip(t[t.split == "test"].Code, t[t.split == "test"].Year)))
print("\nCELL 39 PASSED")

In [ ]:
# ==================================================
# CELL 40 — TRAIN THE STACKING META-MODEL (Linear Regression)
#
# GOAL:
#   Learn how much to trust RF, XGBoost and LSTM, using ONLY the validation
#   predictions, then forecast the test year.
#
# THE META-MODEL:
#   final prediction = a + w_rf * RF + w_xgb * XGB + w_lstm * LSTM
#   (a, w) are fitted by least squares on the 133 validation rows
#   (inputs: three base predictions, target: the real 2023 score).
#   Test labels are never passed to .fit.
#
# HONEST VALIDATION NUMBERS:
#   The meta-model's score on the same rows it was fitted on is optimistic.
#   So we report (a) that in-sample score and (b) a 5-fold cross-validated
#   score INSIDE the validation year (all rows share one year, so there is no
#   past/future mixing; countries are only treated as independent rows).
#   The cross-validated one is used as the stacking "validation" number.
#
# BENCHMARK:
#   A plain average of the three base predictions (no learning). If stacking
#   cannot beat the simple average, the meta-model adds nothing.
#
# LIMITATIONS (printed again at the end): small meta-training set (133),
#   highly correlated base predictions, one validation year.
#
# EXPECTED OUTPUT:
#   meta weights, metrics table (val in-sample, val 5-fold, test) for the
#   primary FIXED variant and the SELECTED sensitivity variant,
#   saved files, "CELL 40 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - meta-model must be fitted on the 133 validation rows only | stacking base file edited or stale | re-run CELL 39, then 40
#   - Reloaded meta-model does not reproduce saved test predictions | mixed runs | re-run 39 then 40
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
truth = D["truth"]
BASES = ["rf", "xgb", "lstm"]


def fit_meta(val_df):
    """Fit on VALIDATION rows only. The function refuses anything else."""
    assert set(val_df["split"]) == {"val"} and len(val_df) == 133, "meta-model must be fitted on the 133 validation rows only"
    assert (val_df["TargetYear"] == 2023).all()
    return LinearRegression().fit(val_df[BASES], val_df["y_true"])


results, coef_rows, out_frames = [], [], {}
for variant in ("fixed", "selected"):
    tab = pd.read_csv(os.path.join(P["pred"], f"stacking_base_predictions_{variant}.csv"))
    val, test = tab[tab.split == "val"].reset_index(drop=True), tab[tab.split == "test"].reset_index(drop=True)
    meta = fit_meta(val)
    p_val_in = meta.predict(val[BASES])
    p_val_cv = cross_val_predict(LinearRegression(), val[BASES], val["y_true"], cv=KFold(5, shuffle=True, random_state=mu.SEED))
    p_test = meta.predict(test[BASES])                                  # test labels untouched until this metric call
    avg_val, avg_test = val[BASES].mean(axis=1).to_numpy(), test[BASES].mean(axis=1).to_numpy()
    for label, y, p, split in (("Stacking: val in-sample (optimistic)", val.y_true, p_val_in, "val"),
                               ("Stacking: val 5-fold CV inside val", val.y_true, p_val_cv, "val"),
                               ("Stacking: test", test.y_true, p_test, "test"),
                               ("Simple average: val", val.y_true, avg_val, "val"),
                               ("Simple average: test", test.y_true, avg_test, "test")):
        results.append({"variant": variant, "what": label, "split": split, **mu.metrics(y, p)})
    coef_rows.append({"variant": variant, "intercept": float(meta.intercept_), **{f"w_{b}": float(w) for b, w in zip(BASES, meta.coef_)},
                      "weight_sum": float(meta.coef_.sum())})
    mu.save_joblib(meta, os.path.join(P["models"], f"stacking_meta_{variant}.joblib"), P)
    base = test[["Country"] + mu.KEYS + ["y_true"]]
    out_frames[variant] = {
        "stack": pd.concat([mu.pred_frame(truth["val"].merge(val[mu.KEYS], on=mu.KEYS), p_val_cv, "val", "Stacked Ensemble"),
                            mu.pred_frame(truth["test"].merge(test[mu.KEYS], on=mu.KEYS), p_test, "test", "Stacked Ensemble")], ignore_index=True),
        "avg": pd.concat([mu.pred_frame(truth["val"].merge(val[mu.KEYS], on=mu.KEYS), avg_val, "val", "Simple Average"),
                          mu.pred_frame(truth["test"].merge(test[mu.KEYS], on=mu.KEYS), avg_test, "test", "Simple Average")], ignore_index=True)}
    for s in ("val", "test"):                                            # alignment of what we save
        for k, fr in out_frames[variant].items():
            mu.validate_pred_frame(fr[fr.split == s], truth[s], s, f"{k}-{variant}")

res = pd.DataFrame(results)
coefs = pd.DataFrame(coef_rows)
print(res.round(4).to_string(index=False))
print("\nMeta-model weights (PRIMARY = fixed). A weight is a trust level for prediction, not a causal effect:")
print(coefs.round(4).to_string(index=False))
neg = coefs[[f"w_{b}" for b in BASES]].lt(0).any(axis=1)
if neg.any():
    print("NOTE: a negative weight appears. With highly correlated base predictions and 133 rows, individual weights are unstable; read the metrics, not single weights.")

mu.save_csv(res, os.path.join(P["tables"], "stacking_results.csv"), P)
mu.save_csv(coefs, os.path.join(P["tables"], "stacking_meta_coefficients.csv"), P)
mu.save_csv(out_frames["fixed"]["stack"], os.path.join(P["pred"], "stacked_predictions.csv"), P)                         # PRIMARY (val rows = 5-fold CV predictions)
mu.save_csv(out_frames["selected"]["stack"], os.path.join(P["pred"], "stacked_predictions_selected_variant.csv"), P)
mu.save_csv(pd.concat([out_frames["fixed"]["avg"].assign(model="Simple Average (fixed bases)"),
                       out_frames["selected"]["avg"].assign(model="Simple Average (selected bases)")], ignore_index=True),
            os.path.join(P["pred"], "simple_average_predictions.csv"), P)
mu.save_json({"meta_model": "LinearRegression (intercept + 3 weights)", "primary_variant": "fixed", "fit_rows": "validation only (133 rows, target year 2023)",
              "val_metric_reported": "5-fold CV inside validation (in-sample also saved)", "test_labels_used_for_fitting": False,
              "limitations": ["only 133 meta-training rows", "base predictions highly correlated", "single validation year", "LSTM early stopping used validation rows"]},
             os.path.join(P["models"], "stacking_meta_config.json"), P)

# ---- validation ----
m2 = joblib.load(os.path.join(P["models"], "stacking_meta_fixed.joblib"))
tf_ = pd.read_csv(os.path.join(P["pred"], "stacking_base_predictions_fixed.csv"))
tt = tf_[tf_.split == "test"]
saved = pd.read_csv(os.path.join(P["pred"], "stacked_predictions.csv"))
assert m2.n_features_in_ == 3 and np.allclose(m2.predict(tt[BASES]), saved[saved.split == "test"].y_pred, atol=1e-10), "Reloaded meta-model does not reproduce saved test predictions"
assert np.isfinite(res[["MAE", "RMSE", "R2"]].to_numpy(float)).all() and len(saved) == 266
print("\nLimitations to state in the report: only 133 meta-training rows; base predictions are highly correlated; one validation year; "
      "LSTM early stopping reused validation rows; test is a single origin year.")
print("CELL 40 PASSED")

In [ ]:
# ==================================================
# CELL 41 — FINAL SIX-MODEL COMPARISON
#
# GOAL:
#   One master table with all six models on the SAME rows, plus figures.
#     1 Persistence   2 Linear Regression   3 Random Forest
#     4 XGBoost       5 LSTM                6 Stacked Ensemble (primary: FIXED-config bases)
#
# READ THE NUMBERS FAIRLY:
#   - test rows = 133 countries, origin 2023 -> target 2024 (one year only).
#   - validation numbers: Persistence and Linear Regression involve no tuning;
#     RF, XGBoost and the LSTM were SELECTED on validation (optimistic);
#     the stack's validation number is a 5-fold CV inside validation.
#     The test columns are the honest comparison.
#   - "vs_persistence_pct": negative = smaller error than Persistence.
#   - R2 is high for every model because countries differ a lot from each
#     other; use MAE/RMSE and the comparison with Persistence.
#   - No significance test is done here: with one test year, small gaps are not
#     evidence of a real difference.
#
# EXTRA ROWS (sensitivity, NOT part of the six): stack with selected-config
#   bases, simple averages, fixed-config RF/XGBoost alone.
#
# EXPECTED OUTPUT: master table, sensitivity table, 4 figures, "CELL 41 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - Missing prediction files | an earlier cell was skipped | run 28-40 in order
#   - 'coverage differs between models' | a model lacks some rows | read the printed common-row counts; metrics use common rows only
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
truth = D["truth"]

SIX = [("Persistence", "persistence_predictions.csv"), ("Linear Regression", "linear_regression_predictions.csv"),
       ("Random Forest", "random_forest_predictions.csv"), ("XGBoost", "xgboost_predictions.csv"),
       ("LSTM", "lstm_predictions.csv"), ("Stacked Ensemble", "stacked_predictions.csv")]
miss = [f for _, f in SIX if not os.path.exists(os.path.join(P["pred"], f))]
assert not miss, f"Missing prediction files {miss}. Run CELLS 28-40 first."
preds = {}
for name, f in SIX:
    pf = pd.read_csv(os.path.join(P["pred"], f))
    pf["model"] = name
    for s in ("val", "test"):
        sub = pf[pf.split == s]
        mu.validate_pred_frame(sub, truth[s].merge(sub[mu.KEYS], on=mu.KEYS), s, name)
    preds[name] = pf

common = {s: set.intersection(*[set(zip(p[p.split == s].Code, p[p.split == s].Year, p[p.split == s].TargetYear)) for p in preds.values()]) for s in ("val", "test")}
print("Common rows across the six models:", {s: len(c) for s, c in common.items()})
if len(common["test"]) < 133 or len(common["val"]) < 133:
    print("NOTE: coverage differs between models; all metrics below use the COMMON rows only.")


def common_rows(pf, s):
    sub = pf[pf.split == s]
    return sub[[k in common[s] for k in zip(sub.Code, sub.Year, sub.TargetYear)]]


rd = lambda f: json.load(open(os.path.join(P["models"], f)))
rf_c, xgb_c = rd("random_forest_config.json"), rd("xgboost_config.json")
lstm_s = json.load(open(os.path.join(P["tables"], "lstm_selected.json")))
coef = pd.read_csv(os.path.join(P["tables"], "stacking_meta_coefficients.csv")).set_index("variant").loc["fixed"]
CFG = {"Persistence": ("none (no fitting)", "prediction = HappinessScore at t"),
       "Linear Regression": ("none (no tuning, fit on train)", "OLS, 18 scaled features"),
       "Random Forest": ("tuned on validation (optimistic)", json.dumps({k: rf_c[k] for k in ("n_estimators", "max_depth", "min_samples_leaf", "max_features")})),
       "XGBoost": ("tuned on validation (optimistic)", json.dumps({k: xgb_c[k] for k in ("n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree")})),
       "LSTM": ("early stopping + selection on validation (optimistic)",
                f"{lstm_s['name']}: seq_len {lstm_s['seq_len']}, units {lstm_s['config']['units']}, dropout {lstm_s['config']['dropout']}, best epoch {lstm_s['best_epoch']}, seed 42"),
       "Stacked Ensemble": ("5-fold CV inside validation", f"LinearRegression meta on [RF, XGB, LSTM] with FIXED-config RF/XGB; weights rf {coef.w_rf:.3f}, xgb {coef.w_xgb:.3f}, lstm {coef.w_lstm:.3f}, intercept {coef.intercept:.3f}")}

rows = []
for name, _ in SIX:
    v, t = mu.metrics(*[common_rows(preds[name], "val")[c] for c in ("y_true", "y_pred")]), mu.metrics(*[common_rows(preds[name], "test")[c] for c in ("y_true", "y_pred")])
    rows.append({"model": name, "val_MAE": v["MAE"], "val_RMSE": v["RMSE"], "val_R2": v["R2"], "test_MAE": t["MAE"], "test_RMSE": t["RMSE"], "test_R2": t["R2"],
                 "n_val": v["n"], "n_test": t["n"], "val_basis": CFG[name][0], "config": CFG[name][1]})
master = pd.DataFrame(rows)
pm = master.loc[master.model == "Persistence"].iloc[0]
master["test_MAE_vs_persistence_pct"] = 100 * (master.test_MAE / pm.test_MAE - 1)
master["test_RMSE_vs_persistence_pct"] = 100 * (master.test_RMSE / pm.test_RMSE - 1)
show = ["model", "val_MAE", "val_RMSE", "val_R2", "test_MAE", "test_RMSE", "test_R2", "n_test", "test_MAE_vs_persistence_pct", "test_RMSE_vs_persistence_pct"]
print("\nMASTER TABLE (model | validation | test):\n", master[show].round(4).to_string(index=False))
mu.save_csv(master, os.path.join(P["tables"], "master_model_comparison.csv"), P)

# ---- sensitivity rows (not among the six) ----
sens = []
fx = pd.read_csv(os.path.join(P["pred"], "stacking_base_predictions_fixed.csv"))
for lab, col in (("RF with fixed settings (alone)", "rf"), ("XGBoost with fixed settings (alone)", "xgb")):
    sens.append({"model": lab, **{f"{s}_{k}": v for s in ("val", "test") for k, v in mu.metrics(fx[fx.split == s].y_true, fx[fx.split == s][col]).items() if k != "n"}})
for lab, f in (("Stacked Ensemble (selected-config bases)", "stacked_predictions_selected_variant.csv"),):
    pf = pd.read_csv(os.path.join(P["pred"], f))
    sens.append({"model": lab, **{f"{s}_{k}": v for s in ("val", "test") for k, v in mu.metrics(pf[pf.split == s].y_true, pf[pf.split == s].y_pred).items() if k != "n"}})
sa = pd.read_csv(os.path.join(P["pred"], "simple_average_predictions.csv"))
for lab in sa.model.unique():
    pf = sa[sa.model == lab]
    sens.append({"model": lab, **{f"{s}_{k}": v for s in ("val", "test") for k, v in mu.metrics(pf[pf.split == s].y_true, pf[pf.split == s].y_pred).items() if k != "n"}})
sens = pd.DataFrame(sens)
print("\nSENSITIVITY (extra rows, not among the six):\n", sens.round(4).to_string(index=False))
mu.save_csv(sens, os.path.join(P["tables"], "sensitivity_comparison.csv"), P)

order = master.sort_values("test_RMSE")[["model", "test_MAE", "test_RMSE"]]
print("\nObserved ordering by test RMSE (descriptive; one test year, no significance test):")
print(order.round(4).to_string(index=False))

# ---- figures ----
colors = {"val": "#9ecae1", "test": "#08519c"}
x = np.arange(len(master)); w = 0.38
for met, fname, ttl in (("MAE", "master_mae_comparison.png", "MAE (points, lower is better)"), ("RMSE", "master_rmse_comparison.png", "RMSE (points, lower is better)"),
                        ("R2", "master_r2_comparison.png", "R$^2$ (higher is better)")):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.bar(x - w/2, master[f"val_{met}"], w, label="validation", color=colors["val"])
    ax.bar(x + w/2, master[f"test_{met}"], w, label="test", color=colors["test"])
    ax.set_xticks(x); ax.set_xticklabels(master.model, rotation=20, ha="right", fontsize=10)
    ax.set_ylabel(met, fontsize=11); ax.set_xlabel("Model", fontsize=11); ax.set_title(f"Six-model comparison: {ttl}", fontsize=12); ax.legend(fontsize=9)
    if met == "R2":
        ax.set_ylim(max(0, master[["val_R2", "test_R2"]].min().min() - 0.02), 1.0)
    plt.tight_layout(); fig.savefig(os.path.join(P["figures"], fname), dpi=150); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
allv = pd.concat([common_rows(p, "test") for p in preds.values()])
lo, hi = min(allv.y_true.min(), allv.y_pred.min()) - 0.2, max(allv.y_true.max(), allv.y_pred.max()) + 0.2
for ax, (name, _) in zip(axes.ravel(), SIX):
    s = common_rows(preds[name], "test")
    ax.scatter(s.y_true, s.y_pred, s=16, alpha=0.7, color="#08519c")
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title(f"{name}: RMSE {mu.metrics(s.y_true, s.y_pred)['RMSE']:.3f}", fontsize=11)
    ax.set_xlabel("Actual happiness in 2024", fontsize=10); ax.set_ylabel("Predicted", fontsize=10)
fig.suptitle("Test set (133 countries): predicted vs actual, dashed line = perfect forecast", fontsize=13)
plt.tight_layout(); fig.savefig(os.path.join(P["figures"], "master_test_predicted_vs_actual.png"), dpi=150); plt.show()

for f in ("master_mae_comparison.png", "master_rmse_comparison.png", "master_r2_comparison.png", "master_test_predicted_vs_actual.png"):
    assert os.path.exists(os.path.join(P["figures"], f)), f
assert len(master) == 6 and np.isfinite(master[["val_MAE", "val_RMSE", "val_R2", "test_MAE", "test_RMSE", "test_R2"]].to_numpy(float)).all()
assert (master.n_test == master.n_test.iloc[0]).all() and master.n_test.iloc[0] >= 100, "Models were not compared on the same rows"
print("\nCELL 41 PASSED")

In [ ]:
# ==================================================
# CELL 42 — FINAL PHASE 2 VALIDATION, REPORT AND GIT SAVE
#
# GOAL:
#   (A) re-verify all six models and their prediction files from disk,
#   (B) check the test set was only used for final scoring,
#   (C) write the Phase 2 report with the REAL numbers (read from files),
#   (D) commit models/results/source to branch modeling-phase and push.
#
# WHAT "NO LEAKAGE" MEANS HERE (what this cell can actually prove):
#   - every model's training rows have origin <= 2021 (targets <= 2022)
#   - hyperparameter / selection tables contain validation columns only
#   - the LSTM test score was produced once (log file) after selection
#   - the meta-model was fitted on 133 validation rows only
#   - scalers/imputers were fitted on the 842 training rows
#   - saved models reproduce their saved test predictions
#   It cannot prove the absence of every conceivable leak; it removes the
#   common ones and documents the rest as limitations.
#
# EXPECTED OUTPUT:
#   PASS table, the report, git commit hash, optional push confirmation,
#   "CELL 42 PASSED".
#
# COMMON FAILURES AND FIXES (error | why | fix and which cells to re-run):
#   - A Phase 2 check FAILED | a file is stale or from a mixed run | read the FAIL row, re-run the cell it names and everything after it
#   - Not on branch modeling-phase | wrong branch | run SETUP and fix the branch
#   - Token-shaped text in .git/config | an old authenticated URL remains | run CELL 16, then 42
#   - PUSH NOT CONFIRMED / 403 / non-fast-forward | same causes as CELL 26 | see the CELL 26 rows; never use --force
# ==================================================
import os, sys, json, importlib, datetime
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, joblib

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Drive not mounted / project missing. Run the Step 1 cell, then CELL 27."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, git_utils as gu
importlib.reload(mu); importlib.reload(gu)
from tensorflow import keras
BRANCH_NAME, GITHUB_USERNAME, RUN_PUSH_PROMPT = "modeling-phase", "sharifa5852", True
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]

rows = []
def add(name, ok, detail=""):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})

# ---------- A) artifacts and reproduction ----------
p1 = mu.check_phase1(P, D)
add("Phase 1 readiness checks still pass", (p1.result == "PASS").all(), f"{len(p1)} checks")
files = {"Linear Regression": "linear_regression.joblib", "Random Forest": "random_forest.joblib", "XGBoost": "xgboost.joblib",
         "Stacking meta-model": "stacking_meta_fixed.joblib"}
for n, f in files.items():
    add(f"{n}: model file loads", joblib.load(os.path.join(P["models"], f)) is not None, f)
lstm_model = keras.models.load_model(os.path.join(P["models"], "lstm_selected.keras"))
add("LSTM: model file loads", lstm_model is not None, "lstm_selected.keras")

pred_files = {"Persistence": "persistence_predictions.csv", "Linear Regression": "linear_regression_predictions.csv",
              "Random Forest": "random_forest_predictions.csv", "XGBoost": "xgboost_predictions.csv",
              "LSTM": "lstm_predictions.csv", "Stacked Ensemble": "stacked_predictions.csv"}
preds = {n: pd.read_csv(os.path.join(P["pred"], f)) for n, f in pred_files.items()}
for n, pf in preds.items():
    for s in ("val", "test"):
        sub = pf[pf.split == s]
        mu.validate_pred_frame(sub, truth[s].merge(sub[mu.KEYS], on=mu.KEYS), s, n)
    add(f"{n}: predictions aligned with real observations (val and test)", True, f"{(pf.split=='val').sum()} + {(pf.split=='test').sum()} rows")

lr, rf, xg = [joblib.load(os.path.join(P["models"], f)) for f in ("linear_regression.joblib", "random_forest.joblib", "xgboost.joblib")]
for n, m, kind in (("Linear Regression", lr, "sc"), ("Random Forest", rf, "imp"), ("XGBoost", xg, "imp")):
    add(f"{n}: feature order == Phase 1 order", list(m.feature_names_in_) == F)
    add(f"{n}: reloaded model reproduces saved test predictions", np.allclose(m.predict(mu.X(D, kind, "test")), preds[n][preds[n].split == "test"].y_pred, atol=1e-6))
seq = mu.load_seq(P, json.load(open(os.path.join(P["tables"], "lstm_selected.json")))["seq_len"], splits=("test",))
add("LSTM: reloaded model reproduces saved test predictions",
    np.allclose(lstm_model.predict(seq["test"]["X"], verbose=0).ravel(), preds["LSTM"][preds["LSTM"].split == "test"].y_pred, atol=1e-5))
base = pd.read_csv(os.path.join(P["pred"], "stacking_base_predictions_fixed.csv"))
bt = base[base.split == "test"]
meta = joblib.load(os.path.join(P["models"], "stacking_meta_fixed.joblib"))
add("Stack: meta-model applied to saved base predictions reproduces saved test predictions",
    np.allclose(meta.predict(bt[["rf", "xgb", "lstm"]]), preds["Stacked Ensemble"][preds["Stacked Ensemble"].split == "test"].y_pred, atol=1e-10))

master = pd.read_csv(os.path.join(P["tables"], "master_model_comparison.csv"))
ok = True
for _, r in master.iterrows():
    for s in ("val", "test"):
        sub = preds[r.model][preds[r.model].split == s]
        m = mu.metrics(sub.y_true, sub.y_pred)
        ok &= all(np.isclose(m[k], r[f"{s}_{k}"], atol=1e-9) for k in ("MAE", "RMSE", "R2"))
add("master table == metrics recomputed from the prediction files", ok)
add("master table: six models, finite metrics, same test rows", len(master) == 6 and np.isfinite(master.select_dtypes("number").to_numpy(float)).all() and master.n_test.nunique() == 1)

# ---------- B) test-set discipline ----------
add("training rows: origin <= 2021 and targets <= 2022", D["raw"]["train"].Year.max() <= 2021 and D["truth"]["train"].TargetYear.max() == 2022)
add("test rows: origin 2023, target 2024; val: origin 2022, target 2023", truth["test"].TargetYear.eq(2024).all() and truth["val"].TargetYear.eq(2023).all())
for f in ("rf_grid_results.csv", "xgb_grid_results.csv", "ridge_sensitivity_validation_only.csv", "lstm_candidate_validation.csv", "lstm_validation_comparison.csv"):
    add(f"selection table {f} has no test columns", not any("test" in c.lower() for c in pd.read_csv(os.path.join(P["tables"], f)).columns))
add("LSTM test evaluation logged once, after selection", os.path.exists(os.path.join(P["tables"], "lstm_test_evaluation_log.json"))
    and json.load(open(os.path.join(P["tables"], "lstm_selected.json")))["selected_on"] == "validation only")
mc = json.load(open(os.path.join(P["models"], "stacking_meta_config.json")))
add("meta-model fitted on validation only; test labels not used for fitting", mc["test_labels_used_for_fitting"] is False and "validation only" in mc["fit_rows"])
add("tabular scaler fit on training rows only (842)", int(joblib.load(os.path.join(P["prep"], "tabular_scaler.joblib")).n_samples_seen_) == 842)
add("length-4 scaler fit on training rows only", int(joblib.load(os.path.join(P["prep"], "lstm4_scaler.joblib")).n_samples_seen_) == int(np.load(os.path.join(P["v2"], "lstm_seq_len4.npz"))["X_train"].shape[0]) * 4)

chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
n_fail = int((chk.result == "FAIL").sum())
mu.save_csv(chk, os.path.join(P["tables"], "phase2_validation_checks.csv"), P)

# ---------- C) report with real numbers ----------
dm = json.load(open(os.path.join(P["v2"], "dataset_metadata.json")))
p1rep = json.load(open(os.path.join(P["tables"], "phase1_validation_report.json")))
best = master.sort_values("test_RMSE").iloc[0]
pers = master[master.model == "Persistence"].iloc[0]
learned = master[master.model != "Persistence"].sort_values("test_RMSE").iloc[0]      # best of the five fitted models
lines = ["# Phase 2 report (generated by CELL 42 from saved files)", "",
         f"Generated: {datetime.datetime.now().isoformat(timespec='seconds')}", "",
         "## Data", f"- Countries: {dm['countries_total']}; samples: {dm['total_samples']}; model features: {dm['n_features']} (15 continuous + 3 flags)",
         *[f"- {k}: {v['rows']} rows, {v['countries']} countries, origin years {v['origin_years']}, target years {v['target_years']}" for k, v in dm["splits"].items()],
         "- Target: happiness at t+1 (unscaled). Strategy: STRICT calendar-year alignment.", "",
         "## Six-model results (test = 133 countries, origin 2023 -> target 2024)", "```", master[["model", "val_MAE", "val_RMSE", "val_R2", "test_MAE", "test_RMSE", "test_R2", "n_test"]].round(4).to_string(index=False), "```", "",
         "## Configurations", *[f"- {r.model}: {r.config} (validation basis: {r.val_basis})" for r in master.itertuples()], "",
         f"## Lowest test RMSE (descriptive only, one test year, no significance test)", f"- Lowest overall: {best.model} (RMSE {best.test_RMSE:.4f}, MAE {best.test_MAE:.4f})",
         f"- Best of the five fitted models: {learned.model} (RMSE {learned.test_RMSE:.4f}, MAE {learned.test_MAE:.4f}); Persistence: RMSE {pers.test_RMSE:.4f}, MAE {pers.test_MAE:.4f}",
         ("- No fitted model has a lower test RMSE than Persistence. Report this as a result." if learned.test_RMSE >= pers.test_RMSE else f"- {learned.model} has a lower test RMSE than Persistence ({100*(learned.test_RMSE/pers.test_RMSE-1):.1f}%); one test year, so treat the gap with caution."),
         "", "## Limitations",
         "- Test is a single origin year (2023 -> 2024); differences between models may be within noise.",
         "- Small panel for deep learning; the LSTM used one seed and early stopping on the validation rows.",
         "- Validation numbers of tuned models are optimistic; test numbers are the comparison to cite.",
         "- Stacking meta-model trained on only 133 validation rows with highly correlated base predictions.",
         "- Strict calendar alignment shrinks the training set; Hong Kong and Puerto Rico have no health-expenditure data (flagged, median-imputed).",
         "- Feature importance / weights describe prediction, not causation.", "",
         "## Completion status", f"- Phase 1: {'COMPLETE' if p1rep['all_checks_pass'] else 'INCOMPLETE'} ({p1rep['n_checks']} audit checks)",
         f"- Phase 2: {'COMPLETE' if n_fail == 0 else 'INCOMPLETE'} ({len(chk) - n_fail}/{len(chk)} checks pass)"]
report_md = "\n".join(lines) + "\n"
open(os.path.join(P["tables"], "phase2_report.md"), "w").write(report_md)
mu.save_json({"n_checks": len(chk), "n_failed": n_fail, "phase1_complete": bool(p1rep["all_checks_pass"]),
              "master": master.drop(columns=["config", "val_basis"]).round(6).to_dict(orient="records")}, os.path.join(P["tables"], "phase2_report.json"), P)
print("\n" + report_md)
assert n_fail == 0, "A Phase 2 check FAILED. Do not commit. See the table above and the error guide."

# ---------- D) git save ----------
assert gu.git(PROJECT_ROOT, "branch", "--show-current").stdout.strip() == BRANCH_NAME, f"Not on branch {BRANCH_NAME}"
cfg_hits = len(gu.TOKEN_RE.findall(open(os.path.join(PROJECT_ROOT, ".git", "config")).read()))
assert cfg_hits == 0, "Token-shaped text in .git/config. Run CELL 16 first."
PHASE2_PATHS = ["src", "models", "results/predictions", "results/tables", "results/figures", "data/processed_v2/lstm_seq_len4.npz",
                "data/processed_v2/lstm_sequence_metadata.json", "requirements.txt"]
staged = gu.stage_and_check(PROJECT_ROOT, PHASE2_PATHS, os.path.join(PROJECT_ROOT, "results", "_backups"))
print("Staged files:", len(staged))
print(gu.git(PROJECT_ROOT, "status", "--short").stdout[:1200] or "(nothing)")
sha = gu.commit(PROJECT_ROOT, "Phase 2: six models (persistence, linear, RF, XGBoost, LSTM, stacking), predictions, figures, reports")
print("Commit:", sha if sha else "nothing new to commit")
if RUN_PUSH_PROMPT:
    from getpass import getpass
    tok = getpass("GitHub token to push (hidden; leave EMPTY to skip the push): ").strip()
    if tok:
        res = gu.push_with_token(PROJECT_ROOT, BRANCH_NAME, GITHUB_USERNAME, tok)
        del tok
        print("PUSH OK" if res["ok"] else "PUSH NOT CONFIRMED", "| local", res.get("local"), "| remote", res.get("remote"))
        print(res["message"])
        assert res["ok"], "Push not confirmed. See the GitHub rows of the error table."
    else:
        print("Push skipped. Everything is committed locally; run CELL 42 again to push later.")
print("\nCELL 42 PASSED")

In [ ]:
# ==================================================
# SETUP — run this FIRST in a new Colab session (safe to re-run any time)
#
# GOAL: mount Google Drive, define PROJECT_ROOT, go into the repo, and confirm you are
#       on branch modeling-phase. Nothing is downloaded or changed.
# EXPECTED OUTPUT: the project path, "Branch: modeling-phase", "SETUP PASSED".
# FAILURES: "Project folder not found" -> the Drive path is different; run `!ls /content/drive/MyDrive`
#           and fix the folder names. "Branch is X" -> run: !git checkout modeling-phase  (inside the project folder).
# ==================================================
import os, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("(not running in Colab: using the paths as given)")
PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), f"Project folder not found: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
subprocess.run(["git", "config", "--global", "--add", "safe.directory", PROJECT_ROOT], capture_output=True)
branch = subprocess.run(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()
print("Project:", PROJECT_ROOT); print("Branch:", branch)
assert branch == "modeling-phase", f"Branch is '{branch}', expected modeling-phase"
print("SETUP PASSED")


In [ ]:
# ==================================================
# CELL 43 — EXPERIMENT ENVIRONMENT AND ARTIFACT VERIFICATION
#
# GOAL:
#   (1) list every Phase 1 and Phase 2 artifact that really exists in your
#   Drive folder, check that it opens, and say what to do if it does not;
#   (2) prove the saved models still reproduce their stored predictions;
#   (3) create src/experiment_utils.py (shared code for CELLS 44-54);
#   (4) save the experiment configuration.
#
# WHY:
#   Phase 3 reuses the saved models and predictions. We confirm they are
#   intact before building anything on them. Nothing is retrained here.
#
# WHAT HAPPENS (plain language):
#   os.path.exists(path)   asks "is there a file here?" (read-only)
#   pd.read_csv(path)      opens a table; a successful open means "loads"
#   joblib.load(path)      opens a saved scikit-learn / XGBoost model
#   keras.models.load_model opens the saved LSTM
#   open(path, "w").write  creates/replaces a file (only src/ and results/ files)
#
# EXPECTED OUTPUT:
#   package versions, an inventory table (every row Exists=True, Loads=True),
#   a reproduction table (all True), the experiment configuration,
#   "CELL 43 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - ModuleNotFoundError: shap | package not installed | `import importlib.metadata as m; m.version('shap')` | `!pip install shap`, Runtime > Restart session, run SETUP, re-run 43
#   - FileNotFoundError: src/modeling_utils.py or git_utils.py | CELL 26/27 not run in this Drive folder | `!ls /content/drive/MyDrive/CountryWellness/*/src` | run CELLS 26 and 27, then 43
#   - inventory shows Exists?=False / Loads=False | an earlier phase output is missing or damaged | read the 'Action' column of the table | re-run the cell it names, then 43
#   - ValueError feature names mismatch / reproduction False | model and data are from different runs | compare `model.feature_names_in_` with feature_spec.json | re-run CELLS 23 and 27-30, then 43
#   - TensorFlow / Keras load error | runtime state or version problem | `import tensorflow as tf; tf.__version__` | Restart session, SETUP, re-run 27 and 43
#   - NaN/inf in inputs (Phase 1 readiness FAIL) | stale processed files | see the FAIL row printed by check_phase1 | re-run CELLS 22-24, then 43
# ==================================================
import os, sys, json, importlib, platform
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hides harmless TensorFlow log noise
import importlib.metadata as md
import numpy as np, pandas as pd, joblib

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
if not os.path.isdir(PROJECT_ROOT):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP (mounts Drive), then re-run this cell."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
for need in ("modeling_utils.py", "git_utils.py"):
    assert os.path.exists(os.path.join(SRC_DIR, need)), f"src/{need} missing: re-run CELL 26 (git_utils) / CELL 27 (modeling_utils)."
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ---------------- shared helper module for Phase 3 ----------------
EXPERIMENT_UTILS = r'''
"""Shared code for Phase 3 experiments. Every function that fits preprocessing fits it on the TRAINING rows it is given only."""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

SEED = 42
MODELS_CHEAP = ["Persistence", "Linear Regression", "Random Forest", "XGBoost"]
COLORS = {"Persistence": "#7f7f7f", "Linear Regression": "#1f77b4", "Random Forest": "#2ca02c", "XGBoost": "#ff7f0e",
          "LSTM": "#9467bd", "Stacked Ensemble": "#d62728"}
RF_KEYS = ("n_estimators", "max_depth", "min_samples_leaf", "max_features", "n_jobs", "random_state")
XGB_KEYS = ("n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree", "reg_lambda", "tree_method", "n_jobs", "random_state", "verbosity")


def style():
    plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
                         "legend.fontsize": 10, "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True})


def load_fixed_params(P):
    """Hyperparameters chosen in Phase 2 on VALIDATION. Never changed in Phase 3."""
    import json, os
    rf = json.load(open(os.path.join(P["models"], "random_forest_config.json")))
    xgb = json.load(open(os.path.join(P["models"], "xgboost_config.json")))
    return {"rf": {k: rf[k] for k in RF_KEYS}, "xgb": {k: xgb[k] for k in XGB_KEYS}}


def fit_predict(model_name, train_df, test_df, features, flags, target, params):
    """Fit one model on train_df ONLY (imputer, scaler and model) and predict test_df. Returns a numpy array."""
    if model_name == "Persistence":
        return test_df["HappinessScore"].to_numpy(float)                  # next year = this year, nothing is fitted
    cont = [c for c in features if c not in flags]
    imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(train_df[features])      # medians from TRAIN only
    Xtr = pd.DataFrame(imp.transform(train_df[features]), columns=features, index=train_df.index)
    Xte = pd.DataFrame(imp.transform(test_df[features]), columns=features, index=test_df.index)
    y = train_df[target].to_numpy(float)
    if model_name == "Linear Regression":
        sc = StandardScaler().fit(Xtr[cont])                                                       # mean/std from TRAIN only
        Xtr, Xte = Xtr.copy(), Xte.copy()
        Xtr[cont], Xte[cont] = sc.transform(Xtr[cont]), sc.transform(Xte[cont])
        return LinearRegression().fit(Xtr, y).predict(Xte)
    if model_name == "Random Forest":
        return RandomForestRegressor(**params["rf"]).fit(Xtr, y).predict(Xte)
    if model_name == "XGBoost":
        return XGBRegressor(**params["xgb"]).fit(Xtr, y).predict(Xte)
    raise ValueError(f"unknown model {model_name}")


def bootstrap_ci(y_true, y_pred, n_boot=2000, seed=SEED, base_pred=None):
    """Percentile 95% CI by resampling countries (rows) with replacement. FIXED predictions: no retraining is simulated."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    n = len(y_true)
    mae, rmse, d_rmse = [], [], []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        e = y_pred[i] - y_true[i]
        mae.append(np.abs(e).mean()); rmse.append(np.sqrt((e ** 2).mean()))
        if base_pred is not None:
            eb = np.asarray(base_pred, float)[i] - y_true[i]
            d_rmse.append(np.sqrt((e ** 2).mean()) - np.sqrt((eb ** 2).mean()))
    q = lambda a: (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
    out = {"MAE_lo": q(mae)[0], "MAE_hi": q(mae)[1], "RMSE_lo": q(rmse)[0], "RMSE_hi": q(rmse)[1]}
    if base_pred is not None:
        out.update({"dRMSE_vs_persistence_lo": q(d_rmse)[0], "dRMSE_vs_persistence_hi": q(d_rmse)[1],
                    "share_boot_better_than_persistence": float(np.mean(np.array(d_rmse) < 0))})
    return out
'''
with open(os.path.join(SRC_DIR, "experiment_utils.py"), "w") as f:
    f.write(EXPERIMENT_UTILS)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
np.random.seed(mu.SEED)
P = mu.make_paths(PROJECT_ROOT)

# ---------------- package versions ----------------
vers = {}
for pkg in ("numpy", "pandas", "scikit-learn", "xgboost", "tensorflow", "keras", "shap", "matplotlib", "joblib"):
    for alias in (pkg, pkg + "-cpu"):                    # some machines install "tensorflow-cpu"
        try:
            vers[pkg] = md.version(alias); break
        except md.PackageNotFoundError:
            vers[pkg] = "NOT INSTALLED"
print("Python", platform.python_version(), "| seed", mu.SEED)
print(json.dumps(vers, indent=1))
assert vers["shap"] != "NOT INSTALLED", "shap is needed from CELL 49. Run  !pip install shap  then Runtime > Restart session, SETUP, and re-run."
from tensorflow import keras

# ---------------- artifact inventory (real filenames, discovered not assumed) ----------------
V2, T, PR, MD = P["v2"], P["tables"], P["pred"], P["models"]
EXPECT = [  # (artifact, path, loader, cell to repeat if missing)
    ("Enriched raw dataset", os.path.join(P["raw"], "whr_worldbank_enriched.csv"), "csv", "CELL 14"),
    ("Original raw dataset", os.path.join(P["raw"], "whr_worldbank_merged.csv"), "csv", "(from GitHub)"),
    ("Corrected supervised dataset", os.path.join(V2, "supervised_all.csv"), "csv", "CELL 20"),
    ("Train split", os.path.join(V2, "train.csv"), "csv", "CELL 22"), ("Validation split", os.path.join(V2, "val.csv"), "csv", "CELL 22"),
    ("Test split", os.path.join(V2, "test.csv"), "csv", "CELL 22"),
    ("Train imputed/scaled", os.path.join(V2, "train_scaled.csv"), "csv", "CELL 23"), ("Val imputed/scaled", os.path.join(V2, "val_scaled.csv"), "csv", "CELL 23"),
    ("Test imputed/scaled", os.path.join(V2, "test_scaled.csv"), "csv", "CELL 23"), ("Train imputed (unscaled)", os.path.join(V2, "train_imputed.csv"), "csv", "CELL 23"),
    ("Feature specification", os.path.join(V2, "feature_spec.json"), "json", "CELL 20"), ("Feature metadata", os.path.join(V2, "feature_metadata.json"), "json", "CELL 24"),
    ("Preprocessing metadata", os.path.join(V2, "preprocessing_metadata.json"), "json", "CELL 23"), ("Data dictionary", os.path.join(V2, "data_dictionary.md"), "text", "CELL 25"),
    ("Tabular imputer", os.path.join(P["prep"], "tabular_imputer.joblib"), "joblib", "CELL 23"), ("Tabular scaler", os.path.join(P["prep"], "tabular_scaler.joblib"), "joblib", "CELL 23"),
    ("LSTM sequences (L3)", os.path.join(V2, "lstm_sequences.npz"), "npz", "CELL 23"), ("LSTM sequences (L4)", os.path.join(V2, "lstm_seq_len4.npz"), "npz", "CELL 34"),
    ("Temporal alignment audit", os.path.join(T, "temporal_alignment_audit_by_year.csv"), "csv", "CELL 17"), ("Temporal strategy decision", os.path.join(T, "temporal_strategy_decision.json"), "json", "CELL 18"),
    ("Missing-value policy", os.path.join(T, "missing_data_policy.json"), "json", "CELL 24"), ("Phase 1 validation report", os.path.join(T, "phase1_validation_report.json"), "json", "CELL 26"),
    ("Persistence predictions", os.path.join(PR, "persistence_predictions.csv"), "csv", "CELL 28"), ("Linear Regression model", os.path.join(MD, "linear_regression.joblib"), "joblib", "CELL 28"),
    ("Linear Regression predictions", os.path.join(PR, "linear_regression_predictions.csv"), "csv", "CELL 28"),
    ("Random Forest model", os.path.join(MD, "random_forest.joblib"), "joblib", "CELL 29"), ("Random Forest config", os.path.join(MD, "random_forest_config.json"), "json", "CELL 29"),
    ("Random Forest predictions", os.path.join(PR, "random_forest_predictions.csv"), "csv", "CELL 29"),
    ("XGBoost model", os.path.join(MD, "xgboost.joblib"), "joblib", "CELL 30"), ("XGBoost config", os.path.join(MD, "xgboost_config.json"), "json", "CELL 30"),
    ("XGBoost predictions", os.path.join(PR, "xgboost_predictions.csv"), "csv", "CELL 30"),
    ("LSTM model (selected)", os.path.join(MD, "lstm_selected.keras"), "keras", "CELLS 35-37"), ("LSTM selection record", os.path.join(T, "lstm_selected.json"), "json", "CELL 36"),
    ("LSTM predictions", os.path.join(PR, "lstm_predictions.csv"), "csv", "CELL 37"),
    ("Stacking meta-model (primary)", os.path.join(MD, "stacking_meta_fixed.joblib"), "joblib", "CELL 40"), ("Stacking base predictions", os.path.join(PR, "stacking_base_predictions_fixed.csv"), "csv", "CELL 39"),
    ("Stacked predictions", os.path.join(PR, "stacked_predictions.csv"), "csv", "CELL 40"),
    ("Baseline results table", os.path.join(T, "baseline_results.csv"), "csv", "CELL 31"), ("Six-model comparison table", os.path.join(T, "master_model_comparison.csv"), "csv", "CELL 41"),
    ("Phase 2 report", os.path.join(T, "phase2_report.json"), "json", "CELL 42"),
]


def try_load(path, kind):
    if kind == "csv":
        return f"table {pd.read_csv(path).shape}"
    if kind == "json":
        return f"json with {len(json.load(open(path)))} keys"
    if kind == "text":
        return f"text, {len(open(path).read().splitlines())} lines"
    if kind == "joblib":
        return type(joblib.load(path)).__name__
    if kind == "npz":
        z = np.load(path); return "arrays " + ", ".join(f"{k}{z[k].shape}" for k in list(z.files)[:2]) + " ..."
    if kind == "keras":
        return f"keras model, input {keras.models.load_model(path).input_shape}"


rows = []
for name, path, kind, fix in EXPECT:
    ex = os.path.exists(path)
    ok, info = False, "MISSING"
    if ex:
        try:
            info = try_load(path, kind); ok = True
        except Exception as e:
            info = f"LOAD ERROR: {type(e).__name__}: {str(e)[:80]}"
    rows.append({"Artifact": name, "Exists?": ex, "Loads successfully?": ok, "Shape/type": info,
                 "Action": "none" if ok else f"re-run {fix}", "File": os.path.relpath(path, PROJECT_ROOT)})
inv = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
print("\n", inv.drop(columns="File").to_string(index=False))
mu.save_csv(inv, os.path.join(T, "phase3_artifact_inventory.csv"), P)
bad = inv[~inv["Loads successfully?"]]
assert bad.empty, "Some artifacts are missing or unreadable. Do the 'Action' column for each row, then re-run CELL 43:\n" + bad[["Artifact", "Action"]].to_string(index=False)

# ---------------- data, features, target ----------------
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]
chk = mu.check_phase1(P, D)
assert (chk.result == "PASS").all(), "Phase 1 readiness check failed:\n" + chk.to_string()
sup = pd.read_csv(os.path.join(V2, "supervised_all.csv"))
assert all(c in sup.columns for c in F), "supervised_all.csv lacks a feature listed in feature_spec.json"

# ---------------- do the saved models still reproduce their stored predictions? (first 20 validation rows) ----------------
SAMPLE = 20
rep = []
def pred_of(name):
    return pd.read_csv(os.path.join(PR, {"Linear Regression": "linear_regression_predictions.csv", "Random Forest": "random_forest_predictions.csv",
                                         "XGBoost": "xgboost_predictions.csv", "LSTM": "lstm_predictions.csv", "Stacked Ensemble": "stacked_predictions.csv",
                                         "Persistence": "persistence_predictions.csv"}[name]))
for name, fn, kind in (("Linear Regression", "linear_regression.joblib", "sc"), ("Random Forest", "random_forest.joblib", "imp"), ("XGBoost", "xgboost.joblib", "imp")):
    m = joblib.load(os.path.join(MD, fn))
    again = m.predict(mu.X(D, kind, "val").iloc[:SAMPLE])
    stored = pred_of(name); stored = stored[stored.split == "val"].iloc[:SAMPLE].y_pred.to_numpy()
    rep.append({"model": name, "reproduces_first_20_val_predictions": bool(np.allclose(again, stored, atol=1e-6)), "feature_order_ok": list(m.feature_names_in_) == F})
sel = json.load(open(os.path.join(T, "lstm_selected.json")))
lm = keras.models.load_model(os.path.join(MD, "lstm_selected.keras"))
sq = mu.load_seq(P, sel["seq_len"], splits=("val",))
st = pred_of("LSTM"); st = st[st.split == "val"].iloc[:SAMPLE].y_pred.to_numpy()
rep.append({"model": "LSTM", "reproduces_first_20_val_predictions": bool(np.allclose(lm.predict(sq["val"]["X"][:SAMPLE], verbose=0).ravel(), st, atol=1e-5)), "feature_order_ok": True})
bp = pd.read_csv(os.path.join(PR, "stacking_base_predictions_fixed.csv")); bv = bp[bp.split == "val"].iloc[:SAMPLE]
meta = joblib.load(os.path.join(MD, "stacking_meta_fixed.joblib"))
rep.append({"model": "Stacking meta-model (applied to saved base predictions; val rows are CV in the stack file)", "reproduces_first_20_val_predictions": bool(meta.predict(bv[["rf", "xgb", "lstm"]]).shape == (SAMPLE,)), "feature_order_ok": True})
pp = pred_of("Persistence"); pp = pp[pp.split == "val"].iloc[:SAMPLE]
rep.append({"model": "Persistence", "reproduces_first_20_val_predictions": bool(np.allclose(truth["val"].persistence.iloc[:SAMPLE], pp.y_pred)), "feature_order_ok": True})
rep = pd.DataFrame(rep)
print("\nReproduction check (no retraining):\n", rep.to_string(index=False))
assert rep["reproduces_first_20_val_predictions"].all() and rep["feature_order_ok"].all(), "A saved model does not reproduce its stored predictions. Re-run the cell that created that model."

# ---------------- six-model table, prediction files, target alignment ----------------
master = pd.read_csv(os.path.join(T, "master_model_comparison.csv"))
assert len(master) == 6 and (master.n_test == 133).all()
for name in master.model:
    pf = pred_of(name)
    for s in ("val", "test"):
        sub = pf[pf.split == s]
        mu.validate_pred_frame(sub, truth[s].merge(sub[mu.KEYS], on=mu.KEYS), s, name)
print("\nAll six prediction files align with the real observations (val and test).")

# ---------------- configuration ----------------
fitted = master[master.model != "Persistence"].sort_values("val_RMSE")
best_fitted = fitted.iloc[0].model                                 # chosen by chronological VALIDATION, not by test
params = eu.load_fixed_params(P)
cfg = {"seed": mu.SEED, "random_split": {"seeds": list(range(42, 52)), "test_size": 0.2, "models": eu.MODELS_CHEAP,
                                         "note": "LSTM and stacking are not repeated under random splits (time budget)."},
       "rolling_origin": {"forecast_years": list(range(2018, 2025)), "requested_years": list(range(2018, 2024)), "extra_year": 2024,
                          "min_train_rows": 180, "min_eval_rows": 30, "rule": "train TargetYear < Y ; evaluate TargetYear == Y",
                          "min_train_justification": "10 observations per feature x 18 features",
                          "models": eu.MODELS_CHEAP, "lstm": "skipped (time budget)"},
       "hyperparameters": {"source": "Phase 2, selected on chronological validation; unchanged in Phase 3", **params},
       "chronological_best_fitted_model_by_validation": best_fitted,
       "shap": {"candidates": ["Random Forest", "XGBoost"], "selection_basis": "validation RMSE"},
       "package_versions": vers}
mu.save_json(cfg, os.path.join(T, "phase3_experiment_config.json"), P)
for d in (P["figures"], P["pred"], P["tables"]):
    os.makedirs(d, exist_ok=True)
print("\nBest fitted model by chronological VALIDATION RMSE:", best_fitted)
print("Experiment configuration saved to results/tables/phase3_experiment_config.json")
assert abs(mu.metrics([1, 2, 3], [1, 2, 4])["RMSE"] - np.sqrt(1 / 3)) < 1e-12
print("\nCELL 43 PASSED")


In [ ]:
# ==================================================
# CELL 44 — EXPERIMENT A: RANDOM 80:20 SPLIT vs CHRONOLOGICAL EVALUATION
#
# GOAL:
#   Re-score Persistence, Linear Regression, Random Forest and XGBoost under
#   a RANDOM 80:20 split of all 1,108 samples, repeated for 10 seeds.
#
# WHY:
#   Many studies split rows at random. For forecasting that mixes years:
#   a country's 2021 row can be in training while its 2020 row is in test.
#   This cell does NOT say random splitting is valid. It MEASURES how much
#   the answer changes when the evaluation protocol changes.
#
# THE LEAK, MADE CONCRETE (we count it for every seed):
#   A test row is (country c, origin year t) with target = score in t+1.
#   If the training set holds the row (c, t+1), then the test target is
#   sitting INSIDE training as that row's "current score" feature.
#   The columns "future_in_train_pct" and "target_visible_pct" measure this.
#
# PROTOCOL FOR EACH SEED (nothing is tuned here):
#   1) random 80:20 split (train_test_split, random_state = seed)
#   2) imputer fitted on this seed's TRAIN rows only
#   3) scaler (Linear Regression only) fitted on this seed's TRAIN rows only
#   4) model with the FIXED Phase 2 hyperparameters (chosen on chronological
#      validation, never on these random-test results)
#   5) predict the random test rows; MAE, RMSE, R2; save predictions + split ids
#   The best fitted model by chronological validation is Linear Regression
#   (CELL 43); it is included. LSTM and stacking are not repeated (time
#   budget). Random results never influence which model is called best.
#
# EXPECTED OUTPUT:
#   a progress line per seed, a results table (40 rows = 10 seeds x 4 models),
#   leakage percentages, "CELL 44 PASSED". Runtime: about 1 minute.
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - FileNotFoundError phase3_experiment_config.json | CELL 43 not run | `!ls results/tables | grep phase3` | run 43, then 44
#   - AssertionError hyperparameters differ | model config files changed after CELL 43 | print `eu.load_fixed_params(P)` | re-run 43, then 44
#   - XGBoostError unknown parameter | xgboost version differs from the one used in Phase 2 | `import xgboost; xgboost.__version__` | `!pip install xgboost==<version in requirements.txt>`, restart, SETUP, re-run 27 and 44
#   - AssertionError n_test != 222 | supervised table changed | `len(pd.read_csv('data/processed_v2/supervised_all.csv'))` | re-run CELLS 20-23 and 43, then 44
#   - MemoryError | runtime memory full | Runtime > Manage sessions | Restart session, SETUP, re-run 43, 44
# ==================================================
import os, sys, json, importlib, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
P = mu.make_paths(PROJECT_ROOT)
cfg_path = os.path.join(P["tables"], "phase3_experiment_config.json")
assert os.path.exists(cfg_path), "Run CELL 43 first (it writes the experiment configuration)."
cfg = json.load(open(cfg_path))
spec = json.load(open(os.path.join(P["v2"], "feature_spec.json")))
F, FLAGS, T = spec["feature_cols"], spec["flag_cols"], spec["target_col"]
sup = pd.read_csv(os.path.join(P["v2"], "supervised_all.csv"))
assert len(sup) == 1108 and not sup.duplicated(["Code", "Year"]).any()
SEEDS, MODELS = cfg["random_split"]["seeds"], cfg["random_split"]["models"]
params = eu.load_fixed_params(P)
best_fitted = cfg["chronological_best_fitted_model_by_validation"]
print("Best fitted model by chronological validation:", best_fitted,
      "-> included in this experiment" if best_fitted in MODELS else "-> NOT repeatable cheaply here (documented)")
assert params == {"rf": cfg["hyperparameters"]["rf"], "xgb": cfg["hyperparameters"]["xgb"]}, "Hyperparameters differ from the saved Phase 3 configuration"

rows, preds, assigns = [], [], []
keyset = set(zip(sup.Code, sup.Year))
t0 = time.time()
for seed in SEEDS:
    idx_tr, idx_te = train_test_split(np.arange(len(sup)), test_size=0.2, random_state=seed, shuffle=True)
    train, test = sup.iloc[idx_tr], sup.iloc[idx_te]
    tr_keys = set(zip(train.Code, train.Year))
    tr_years = train.groupby("Code").Year.apply(list).to_dict()
    # how much of the "future" is inside training for the random test rows
    future_in_train = np.mean([any(y2 > y for y2 in tr_years.get(c, [])) for c, y in zip(test.Code, test.Year)])
    target_visible = np.mean([(c, y + 1) in tr_keys for c, y in zip(test.Code, test.Year)])
    assert not (tr_keys & set(zip(test.Code, test.Year))), "train and test rows overlap"
    assigns.append(pd.DataFrame({"seed": seed, "Code": sup.Code, "Year": sup.Year, "partition": "random_test"}).assign(
        partition=lambda d: np.where(d.index.isin(idx_te), "random_test", "random_train")))
    for model in MODELS:
        yhat = eu.fit_predict(model, train, test, F, FLAGS, T, params)
        m = mu.metrics(test[T], yhat)
        rows.append({"seed": seed, "model": model, "MAE": m["MAE"], "RMSE": m["RMSE"], "R2": m["R2"], "n_train": len(train), "n_test": len(test),
                     "future_in_train_pct": 100 * future_in_train, "target_visible_pct": 100 * target_visible})
        preds.append(pd.DataFrame({"seed": seed, "model": model, "Country": test.Country.to_numpy(), "Code": test.Code.to_numpy(), "Year": test.Year.to_numpy(),
                                   "TargetYear": test.TargetYear.to_numpy(), "y_true": test[T].to_numpy(float), "y_pred": np.asarray(yhat, float)}))
    print(f"seed {seed}: done ({time.time()-t0:.0f}s) | test rows whose country also has a LATER year in training: {100*future_in_train:.1f}% | "
          f"test target visible as a training feature: {100*target_visible:.1f}%")

res = pd.DataFrame(rows)
pred_all = pd.concat(preds, ignore_index=True)
assign_all = pd.concat(assigns, ignore_index=True)
print("\n", res.round(4).to_string(index=False))
mu.save_csv(res, os.path.join(P["tables"], "random_split_results.csv"), P)
mu.save_csv(pred_all, os.path.join(P["pred"], "random_split_predictions.csv"), P)
mu.save_csv(assign_all, os.path.join(P["pred"], "random_split_assignments.csv"), P)

# VALIDATION
assert len(res) == len(SEEDS) * len(MODELS) == 40, "Not every seed/model finished"
assert res.groupby("model").seed.nunique().eq(10).all(), "A model is missing seeds"
assert np.isfinite(res[["MAE", "RMSE", "R2"]].to_numpy(float)).all()
assert (res.n_test == 222).all() and (res.n_train == 886).all()
chk = pred_all.merge(sup[["Code", "Year", "TargetYear", T]], on=["Code", "Year", "TargetYear"], how="left")
assert chk[T].notna().all() and np.allclose(chk[T], chk["y_true"]), "random predictions do not match their identifiers/targets"
assert not pred_all.duplicated(["seed", "model", "Code", "Year"]).any(), "duplicate prediction rows"
sizes = assign_all.groupby("seed").partition.value_counts().unstack()
assert (sizes.random_test == 222).all() and (sizes.random_train == 886).all()
print("\nA random split puts about", round(res.future_in_train_pct.mean(), 1), "% of test rows in countries that also have LATER years in training, and for",
      round(res.target_visible_pct.mean(), 1), "% of test rows the answer itself is a training feature. A chronological split makes both 0%.")
print("CELL 44 PASSED")


In [ ]:
# ==================================================
# CELL 45 — RANDOM-SPLIT SUMMARY vs THE CHRONOLOGICAL TEST RESULT
#
# GOAL:
#   Average the 10 random-split runs per model, compare with the saved
#   chronological test result from Phase 2, add a country-level bootstrap
#   confidence interval for the fixed chronological predictions, and draw it.
#
# HOW TO READ THE DIFFERENCE (direction defined here, once):
#     diff = random - chronological
#   MAE and RMSE: diff < 0 means the random split LOOKS BETTER (smaller error)
#   R2:           diff > 0 means the random split LOOKS BETTER (higher R2)
#
# TWO COMPARISONS, because the rows are not the same:
#   - chronological test = 133 countries, one year (target 2024)
#   - random test        = 222 rows from many years (targets 2017-2024)
#   Different rows can have different difficulty, so a raw gap mixes "protocol"
#   with "which years". To reduce that, we also report each model's RMSE
#   RELATIVE TO PERSISTENCE on the same rows (ratio < 1 = better than
#   persistence). CELL 48 adds a third comparison with the rolling-origin
#   backtest, which covers many years under a chronological protocol.
#
# BOOTSTRAP (optional part):
#   Resample the 133 test countries with replacement 2,000 times and recompute
#   MAE/RMSE. One row = one country in the test year, so country is the unit.
#   This shows uncertainty of the SAVED predictions on this test year only. It
#   does not include variation from retraining the models.
#
# EXPECTED OUTPUT:
#   summary tables, a figure, the plain-language conclusion, "CELL 45 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - AssertionError (len(res) == 40) / Run CELL 44 first | random-split results missing | `!ls results/tables | grep random` | run 44, then 45
#   - bootstrap shape/Code mismatch | prediction files are from different runs | compare `len(test[n])` for each model | re-run CELLS 28-41, then 45
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
eu.style()
P = mu.make_paths(PROJECT_ROOT)
res = pd.read_csv(os.path.join(P["tables"], "random_split_results.csv"))
assert len(res) == 40, "Run CELL 44 first."
master = pd.read_csv(os.path.join(P["tables"], "master_model_comparison.csv")).set_index("model")
MODELS = eu.MODELS_CHEAP

# ---------- statistics over the 10 seeds ----------
g = res.groupby("model")
stat = g[["MAE", "RMSE", "R2"]].agg(["mean", "std", "min", "max"])
stat.columns = [f"{m}_{s}" for m, s in stat.columns]
stat = stat.loc[MODELS]
# ---------- the requested table ----------
rows = []
for m in MODELS:
    rows.append({"Model": m, "Split": "Chronological test (2023 -> 2024, 133 rows)", "Mean MAE": master.loc[m, "test_MAE"], "Std MAE": np.nan,
                 "Mean RMSE": master.loc[m, "test_RMSE"], "Std RMSE": np.nan, "Mean R2": master.loc[m, "test_R2"], "Std R2": np.nan})
    rows.append({"Model": m, "Split": "Random 80:20 (10 seeds, 222 rows each)", "Mean MAE": stat.loc[m, "MAE_mean"], "Std MAE": stat.loc[m, "MAE_std"],
                 "Mean RMSE": stat.loc[m, "RMSE_mean"], "Std RMSE": stat.loc[m, "RMSE_std"], "Mean R2": stat.loc[m, "R2_mean"], "Std R2": stat.loc[m, "R2_std"]})
req = pd.DataFrame(rows)
print(req.round(4).to_string(index=False))
print("\nMin / max over seeds:\n", stat[["MAE_min", "MAE_max", "RMSE_min", "RMSE_max", "R2_min", "R2_max"]].round(4).to_string())

# ---------- comparison table: random - chronological ----------
comp = []
pers_rand = stat.loc["Persistence", "RMSE_mean"]; pers_chrono = master.loc["Persistence", "test_RMSE"]
for m in MODELS:
    c = {"model": m}
    for k in ("MAE", "RMSE", "R2"):
        chrono, rmean, rstd = master.loc[m, f"test_{k}"], stat.loc[m, f"{k}_mean"], stat.loc[m, f"{k}_std"]
        c.update({f"chrono_{k}": chrono, f"random_mean_{k}": rmean, f"random_std_{k}": rstd, f"diff_random_minus_chrono_{k}": rmean - chrono})
    c["random_looks_better_by_RMSE"] = bool(c["diff_random_minus_chrono_RMSE"] < 0)
    c["rmse_ratio_vs_persistence_chrono"] = master.loc[m, "test_RMSE"] / pers_chrono
    c["rmse_ratio_vs_persistence_random"] = stat.loc[m, "RMSE_mean"] / pers_rand
    c["random_mean_future_in_train_pct"] = float(res[res.model == m].future_in_train_pct.mean())
    c["random_mean_target_visible_pct"] = float(res[res.model == m].target_visible_pct.mean())
    comp.append(c)
comp = pd.DataFrame(comp)
mu.save_csv(comp, os.path.join(P["tables"], "random_vs_chronological_comparison.csv"), P)
mu.save_csv(req, os.path.join(P["tables"], "random_vs_chronological_summary_table.csv"), P)
print("\nDifference table (diff = random - chronological):\n",
      comp[["model", "diff_random_minus_chrono_MAE", "diff_random_minus_chrono_RMSE", "diff_random_minus_chrono_R2", "random_looks_better_by_RMSE",
            "rmse_ratio_vs_persistence_chrono", "rmse_ratio_vs_persistence_random"]].round(4).to_string(index=False))

# ---------- bootstrap CI for the saved chronological test predictions ----------
files = {"Persistence": "persistence_predictions.csv", "Linear Regression": "linear_regression_predictions.csv", "Random Forest": "random_forest_predictions.csv",
         "XGBoost": "xgboost_predictions.csv", "LSTM": "lstm_predictions.csv", "Stacked Ensemble": "stacked_predictions.csv"}
test = {n: pd.read_csv(os.path.join(P["pred"], f)).query("split == 'test'").sort_values("Code").reset_index(drop=True) for n, f in files.items()}
assert all(len(t) == 133 for t in test.values()) and all((t.Code.values == test["Persistence"].Code.values).all() for t in test.values())
y = test["Persistence"].y_true.to_numpy()
boot = []
for n, t in test.items():
    ci = eu.bootstrap_ci(y, t.y_pred.to_numpy(), n_boot=2000, seed=mu.SEED, base_pred=None if n == "Persistence" else test["Persistence"].y_pred.to_numpy())
    boot.append({"model": n, "test_MAE": master.loc[n, "test_MAE"], "test_RMSE": master.loc[n, "test_RMSE"], **ci})
boot = pd.DataFrame(boot)
print("\nCountry-level bootstrap, 95% percentile intervals (2,000 resamples of the 133 test countries; fixed predictions):\n",
      boot.round(4).to_string(index=False))
mu.save_csv(boot, os.path.join(P["tables"], "chronological_test_bootstrap_ci.csv"), P)

# ---------- figure ----------
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
x = np.arange(len(MODELS)); w = 0.38
for ax, k, lab in ((axes[0, 0], "MAE", "MAE (points)"), (axes[0, 1], "RMSE", "RMSE (points)"), (axes[1, 0], "R2", "R$^2$")):
    ax.bar(x - w / 2, comp[f"chrono_{k}"], w, label="Chronological test (2024 target)", color="#08519c")
    ax.bar(x + w / 2, comp[f"random_mean_{k}"], w, yerr=comp[f"random_std_{k}"], capsize=4, label="Random 80:20, mean $\\pm$ std (10 seeds)", color="#fdae6b")
    ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=15, ha="right"); ax.set_ylabel(lab); ax.set_xlabel("Model")
    ax.set_title(f"{k}: chronological vs random evaluation")
    if k == "R2":
        ax.set_ylim(max(0.9, min(comp.chrono_R2.min(), comp.random_mean_R2.min()) - 0.01), 1.0)
    ax.legend()
ax = axes[1, 1]
ax.bar(x - w / 2, comp.rmse_ratio_vs_persistence_chrono, w, label="Chronological test", color="#08519c")
ax.bar(x + w / 2, comp.rmse_ratio_vs_persistence_random, w, label="Random 80:20 (mean)", color="#fdae6b")
ax.axhline(1, color="k", ls="--", lw=1, label="same as Persistence")
ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=15, ha="right"); ax.set_ylabel("RMSE / Persistence RMSE"); ax.set_xlabel("Model")
ax.set_title("RMSE relative to Persistence on the same rows (below 1 = better)"); ax.legend()
fig.suptitle("Experiment A: random split vs chronological evaluation", fontsize=14)
fp = os.path.join(P["figures"], "random_vs_chronological.png"); fig.savefig(fp); plt.show()

# ---------- plain-language conclusion computed from the numbers ----------
fit = comp[comp.model != "Persistence"]
n_better = int(fit.random_looks_better_by_RMSE.sum())
print("\nWHAT THE NUMBERS SAY (computed, not assumed):")
print(f"- Leakage is real in the random protocol: on average {res.future_in_train_pct.mean():.0f}% of random-test rows have LATER years of the same country in training, "
      f"and {res.target_visible_pct.mean():.0f}% have their answer visible as a training feature.")
print(f"- Absolute error: random RMSE is LOWER than chronological for {n_better} of {len(fit)} fitted models, higher for the rest.")
for r in fit.itertuples():
    print(f"  {r.model}: chronological {r.chrono_RMSE:.4f} vs random {r.random_mean_RMSE:.4f} (diff {r.diff_random_minus_chrono_RMSE:+.4f}); "
          f"ratio to Persistence {r.rmse_ratio_vs_persistence_chrono:.3f} (chrono) vs {r.rmse_ratio_vs_persistence_random:.3f} (random)")
if n_better >= 0.5 * len(fit):
    print("=> In this experiment random splitting APPEARS to make most fitted models look better in absolute error.")
else:
    print("=> In this experiment random splitting does NOT make absolute error look better for most models. That does not make it a valid forecasting protocol:")
    print("   the leakage counts above are not zero, the two test sets contain different years, and the 133-row chronological test is a single year.")
rel_better = fit[fit.rmse_ratio_vs_persistence_random < fit.rmse_ratio_vs_persistence_chrono].model.tolist()
rel_worse = fit[fit.rmse_ratio_vs_persistence_random >= fit.rmse_ratio_vs_persistence_chrono].model.tolist()
print(f"- Relative to Persistence on the same rows, the random split is MORE favourable than the chronological test for: {rel_better or 'none'}; "
      f"less or equally favourable for: {rel_worse or 'none'}.")
print("  (A model looking better against its own persistence baseline under random splitting is the pattern that would suggest overstated skill.)")
print("- Treat all of this as one dataset, one model family and one chronological test year; CELL 48 adds many chronological years for a fairer comparison.")

# VALIDATION
assert len(req) == 8 and len(comp) == 4 and len(boot) == 6
assert np.isfinite(comp.select_dtypes("number").to_numpy(float)).all()
assert (boot.RMSE_lo <= boot.RMSE_hi).all() and (boot.MAE_lo <= boot.MAE_hi).all()
assert os.path.exists(fp) and os.path.exists(os.path.join(P["tables"], "random_vs_chronological_comparison.csv"))
print("\nCELL 45 PASSED")


In [ ]:
# ==================================================
# CELL 46 — ROLLING-ORIGIN BACKTEST: ELIGIBILITY (Experiment B, part 1)
#
# GOAL:
#   Decide, from the real sample counts, which forecast years can be
#   backtested, and record the decision (including every skipped year).
#
# THE RULE (temporal logic):
#   To forecast target year Y:
#     training rows   : TargetYear  <  Y     (outcomes already known)
#     evaluation rows : TargetYear == Y
#   Origin year = TargetYear - 1. So for Y = 2020 the training rows have
#   targets up to 2019 and nothing from 2020 or later is used.
#
# YEARS: 2018-2023 as requested, plus 2024 (the dataset supports it and it is
#   the final test target; it is flagged as an extra year).
#
# MINIMUM SAMPLE RULES (declared before fitting, with reasons):
#   - at least 180 training rows = 10 observations per feature x 18 features,
#     a common rule of thumb for a stable linear fit with this many inputs;
#   - at least 30 evaluation rows so error metrics are not dominated by 1-2 countries.
#   A year below the limit is SKIPPED and written to the table with the reason.
#   Nothing is forced to train on too little data.
#
# EXPECTED OUTPUT:
#   the eligibility table (one row per forecast year), "CELL 46 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - AssertionError fewer than 3 eligible years | too few training rows | see the printed table | do not force it: report the skipped years (they are recorded)
#   - AssertionError training contains target year >= Y | supervised table modified | `sup.TargetYear.value_counts()` | re-run CELL 20-22, then 46
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
cfg = json.load(open(os.path.join(P["tables"], "phase3_experiment_config.json")))["rolling_origin"]
sup = pd.read_csv(os.path.join(P["v2"], "supervised_all.csv"))
YEARS, REQUESTED = cfg["forecast_years"], cfg["requested_years"]
MIN_TRAIN, MIN_EVAL = cfg["min_train_rows"], cfg["min_eval_rows"]
print("Rows per TargetYear in the supervised table:", sup.groupby("TargetYear").size().to_dict())

rows = []
for Y in YEARS:
    tr = sup[sup.TargetYear < Y]
    ev = sup[sup.TargetYear == Y]
    reasons = []
    if len(tr) < MIN_TRAIN:
        reasons.append(f"only {len(tr)} training rows (< {MIN_TRAIN})")
    if len(ev) < MIN_EVAL:
        reasons.append(f"only {len(ev)} evaluation rows (< {MIN_EVAL})")
    rows.append({"forecast_year": Y, "train_samples": len(tr), "eval_samples": len(ev),
                 "train_target_years": f"{int(tr.TargetYear.min())}-{int(tr.TargetYear.max())}" if len(tr) else "none",
                 "train_origin_years": f"{int(tr.Year.min())}-{int(tr.Year.max())}" if len(tr) else "none",
                 "eval_countries": int(ev.Code.nunique()), "eligible": len(reasons) == 0, "reason_if_skipped": "; ".join(reasons),
                 "in_requested_range": Y in REQUESTED})
elig = pd.DataFrame(rows)
print("\n", elig.to_string(index=False))
mu.save_csv(elig, os.path.join(P["tables"], "rolling_origin_eligibility.csv"), P)

# VALIDATION: the temporal rule must hold for every year, eligible or not
for Y in YEARS:
    tr, ev = sup[sup.TargetYear < Y], sup[sup.TargetYear == Y]
    assert len(tr) == 0 or tr.TargetYear.max() < Y, f"Y={Y}: training contains target year >= {Y}"
    assert (ev.TargetYear == Y).all() and (ev.Year == Y - 1).all()
    assert not (set(zip(tr.Code, tr.Year)) & set(zip(ev.Code, ev.Year))), f"Y={Y}: row overlap"
assert elig.eligible.sum() >= 3, "Fewer than 3 eligible forecast years: the backtest would not say much. Report this honestly."
print(f"\nEligible forecast years: {elig[elig.eligible].forecast_year.tolist()} | skipped: {elig[~elig.eligible].forecast_year.tolist() or 'none'}")
print("CELL 46 PASSED")


In [ ]:
# ==================================================
# CELL 47 — ROLLING-ORIGIN BACKTEST: RUN IT (Experiment B, part 2)
#
# GOAL:
#   For every eligible forecast year, train on the past only and forecast that
#   year with Persistence, Linear Regression, Random Forest and XGBoost.
#
# EACH YEAR (nothing is shared between years):
#   1) training rows = TargetYear < Y ; evaluation rows = TargetYear == Y
#   2) imputer and (for Linear Regression) scaler are fitted on THIS year's
#      training rows only. We do not reuse the Phase 2 scaler: it was fitted
#      on rows up to 2021, which would leak information into earlier years.
#   3) models use the hyperparameters selected in Phase 2 (no new tuning)
#   4) predict; MAE, RMSE, R2; save predictions with identifiers
#   Persistence = current-year HappinessScore. LSTM is skipped (time budget).
#   Skipped years are written into the results table with their reason.
#
# EXPECTED OUTPUT:
#   one line per year, a results table, "CELL 47 PASSED". Runtime: about 1 minute.
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - FileNotFoundError rolling_origin_eligibility.csv | CELL 46 not run | `!ls results/tables | grep rolling` | run 46, then 47
#   - AssertionError 2023 fold != Phase 2 validation RMSE | Phase 2 models/config from a different run | compare rolling_origin_results.csv 2023 rows with master_model_comparison.csv val_RMSE | re-run CELLS 29-31, 41, 43, then 47
#   - NaN metrics / ValueError in fit | NaN in the fold's features after imputation | `train[F].isna().all()` for the failing year | re-run CELLS 22-24, then 46-47
# ==================================================
import os, sys, json, importlib, time
import numpy as np, pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
P = mu.make_paths(PROJECT_ROOT)
cfg = json.load(open(os.path.join(P["tables"], "phase3_experiment_config.json")))
elig_path = os.path.join(P["tables"], "rolling_origin_eligibility.csv")
assert os.path.exists(elig_path), "Run CELL 46 first."
elig = pd.read_csv(elig_path)
spec = json.load(open(os.path.join(P["v2"], "feature_spec.json")))
F, FLAGS, T = spec["feature_cols"], spec["flag_cols"], spec["target_col"]
sup = pd.read_csv(os.path.join(P["v2"], "supervised_all.csv"))
params = eu.load_fixed_params(P)
assert params == {"rf": cfg["hyperparameters"]["rf"], "xgb": cfg["hyperparameters"]["xgb"]}, "Hyperparameters changed since CELL 43"
MODELS = cfg["rolling_origin"]["models"]

res_rows, pred_frames = [], []
t0 = time.time()
for r in elig.itertuples():
    Y = int(r.forecast_year)
    if not r.eligible:
        res_rows.append({"forecast_year": Y, "model": "ALL", "status": "skipped", "reason": r.reason_if_skipped, "train_n": r.train_samples,
                         "eval_n": r.eval_samples, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan})
        print(f"{Y}: SKIPPED ({r.reason_if_skipped})")
        continue
    train, ev = sup[sup.TargetYear < Y], sup[sup.TargetYear == Y]
    assert train.TargetYear.max() < Y and (ev.TargetYear == Y).all()
    for model in MODELS:
        yhat = eu.fit_predict(model, train, ev, F, FLAGS, T, params)          # preprocessing fitted on `train` only
        m = mu.metrics(ev[T], yhat)
        res_rows.append({"forecast_year": Y, "model": model, "status": "evaluated", "reason": "", "train_n": len(train), "eval_n": len(ev),
                         "MAE": m["MAE"], "RMSE": m["RMSE"], "R2": m["R2"]})
        pred_frames.append(pd.DataFrame({"forecast_year": Y, "model": model, "Country": ev.Country.to_numpy(), "Code": ev.Code.to_numpy(), "Year": ev.Year.to_numpy(),
                                         "TargetYear": ev.TargetYear.to_numpy(), "y_true": ev[T].to_numpy(float), "y_pred": np.asarray(yhat, float),
                                         "train_max_target_year": int(train.TargetYear.max()), "train_n": len(train)}))
    print(f"{Y}: trained on {len(train)} rows (targets up to {int(train.TargetYear.max())}), evaluated on {len(ev)} countries ({time.time()-t0:.0f}s)")

res = pd.DataFrame(res_rows)
preds = pd.concat(pred_frames, ignore_index=True)
show = res[res.status == "evaluated"].pivot(index="forecast_year", columns="model", values="RMSE")[MODELS]
print("\nRMSE by forecast year (points):\n", show.round(4).to_string())
mu.save_csv(res, os.path.join(P["tables"], "rolling_origin_results.csv"), P)
mu.save_csv(preds, os.path.join(P["pred"], "rolling_origin_predictions.csv"), P)

# VALIDATION
n_ok = int(elig.eligible.sum())
ev_res = res[res.status == "evaluated"]
assert ev_res.groupby("model").forecast_year.nunique().eq(n_ok).all() and len(ev_res) == n_ok * len(MODELS)
assert np.isfinite(ev_res[["MAE", "RMSE", "R2"]].to_numpy(float)).all()
assert (preds.train_max_target_year < preds.forecast_year).all(), "A training fold contained target year >= the forecast year"
assert (preds.TargetYear == preds.forecast_year).all() and (preds.TargetYear == preds.Year + 1).all()
assert not preds.duplicated(["forecast_year", "model", "Code"]).any()
chk = preds.merge(sup[["Code", "Year", T]], on=["Code", "Year"], how="left")
assert np.allclose(chk[T], chk.y_true), "rolling predictions do not match their real targets"
assert (res[res.status == "skipped"].shape[0] == int((~elig.eligible).sum())), "skipped years not recorded"
# cross-check: the 2023 fold is exactly the Phase 2 validation setup (train on targets <= 2022, forecast 2023)
master = pd.read_csv(os.path.join(P["tables"], "master_model_comparison.csv")).set_index("model")
if 2023 in set(ev_res.forecast_year):
    for m in ("Persistence", "Linear Regression", "Random Forest", "XGBoost"):
        got = float(ev_res[(ev_res.forecast_year == 2023) & (ev_res.model == m)].RMSE.iloc[0])
        assert abs(got - float(master.loc[m, "val_RMSE"])) < 1e-6, f"{m}: 2023 rolling fold != Phase 2 validation RMSE ({got} vs {master.loc[m, 'val_RMSE']})"
    print("Cross-check OK: the 2023 rolling fold reproduces the Phase 2 validation RMSE of all four models.")
print("\nCELL 47 PASSED")


In [ ]:
# ==================================================
# CELL 48 — ROLLING-ORIGIN GRAPHS, SUMMARY AND PHASE 3 EXPERIMENT SUMMARY
#
# GOAL:
#   Draw MAE, RMSE and R2 across forecast years (one graph each), summarise
#   each model, say honestly how stable the results are, compare with the
#   random-split experiment on a fairer footing, and write a Phase 3 summary.
#
# WHAT TO LOOK FOR:
#   - Does a model beat Persistence in MOST years or only in some?
#   - Is the ranking the same every year? (If not, one test year can mislead.)
#   - With only a handful of eligible years, do NOT call anything "stable".
#
# EXPECTED OUTPUT: 3 figures, summary tables, interpretation lines computed
#   from the numbers, "CELL 48 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - Run CELL 47 first / no evaluated rows | rolling results missing | `!ls results/tables | grep rolling` | run 46-47, then 48
#   - figure file smaller than 10 KB | plot failed to save | `!ls -l results/figures` | re-run 48 (restart session if matplotlib is broken)
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
eu.style()
P = mu.make_paths(PROJECT_ROOT)
res = pd.read_csv(os.path.join(P["tables"], "rolling_origin_results.csv"))
elig = pd.read_csv(os.path.join(P["tables"], "rolling_origin_eligibility.csv"))
assert (res.status == "evaluated").any(), "Run CELL 47 first."
ev = res[res.status == "evaluated"].copy()
MODELS = ["Persistence", "Linear Regression", "Random Forest", "XGBoost"]
years_ok = [int(y) for y in sorted(ev.forecast_year.unique())]
years_skip = elig[~elig.eligible].forecast_year.tolist()

# ---------- three graphs ----------
for met, fname, ylab in (("MAE", "rolling_origin_mae.png", "MAE (happiness points)"), ("RMSE", "rolling_origin_rmse.png", "RMSE (happiness points)"),
                         ("R2", "rolling_origin_r2.png", "R$^2$")):
    fig, ax = plt.subplots(figsize=(9, 5))
    for m in MODELS:
        d = ev[ev.model == m].sort_values("forecast_year")
        ax.plot(d.forecast_year, d[met], marker="o", lw=2, label=m, color=eu.COLORS[m])
    for ys in years_skip:
        ax.axvspan(ys - 0.4, ys + 0.4, color="gray", alpha=0.2)
        ax.text(ys, ax.get_ylim()[1], "skipped\n(too few\ntraining rows)", ha="center", va="top", fontsize=9)
    ax.set_xticks(sorted(set(years_ok) | set(years_skip)))
    ax.set_xlabel("Forecast (target) year"); ax.set_ylabel(ylab)
    ax.set_title(f"Rolling-origin backtest: {met} by forecast year (training = all earlier targets)")
    ax.legend()
    fig.savefig(os.path.join(P["figures"], fname)); plt.show()

# ---------- summary per model ----------
wide = ev.pivot(index="forecast_year", columns="model", values="RMSE")[MODELS]
wins = wide.idxmin(axis=1).value_counts()
summ = []
for m in MODELS:
    d = ev[ev.model == m]
    summ.append({"model": m, "n_successful_forecast_years": int(d.forecast_year.nunique()), "mean_MAE": d.MAE.mean(), "mean_RMSE": d.RMSE.mean(), "mean_R2": d.R2.mean(),
                 "std_MAE_across_years": d.MAE.std(), "std_RMSE_across_years": d.RMSE.std(), "std_R2_across_years": d.R2.std(),
                 "years_with_lowest_RMSE": int(wins.get(m, 0)),
                 "years_better_than_persistence_RMSE": int((wide[m] < wide["Persistence"]).sum()) if m != "Persistence" else np.nan})
summ = pd.DataFrame(summ)
print("\nSkipped forecast years:", years_skip or "none", "| evaluated:", years_ok)
print("\n", summ.round(4).to_string(index=False))
mu.save_csv(summ, os.path.join(P["tables"], "rolling_origin_summary.csv"), P)

# ---------- year difficulty and ranking ----------
diff = wide.mean(axis=1).sort_values()
print("\nAverage RMSE of the four models by forecast year (easiest first):\n", diff.round(4).to_string())
rank = wide.rank(axis=1)
print("\nRank of each model by year (1 = lowest RMSE):\n", rank.astype(int).to_string())

# ---------- computed interpretation ----------
n = len(years_ok)
print("\nINTERPRETATION (computed):")
if n < 5:
    print(f"- Only {n} valid forecast years: do not describe performance as stable.")
else:
    print(f"- {n} valid forecast years give a rough picture, not a precise one.")
for m in MODELS[1:]:
    k = int((wide[m] < wide["Persistence"]).sum())
    print(f"- {m} has a lower RMSE than Persistence in {k} of {n} years (mean RMSE {summ.set_index('model').loc[m, 'mean_RMSE']:.4f} vs {summ.set_index('model').loc['Persistence', 'mean_RMSE']:.4f}).")
top = wins.idxmax()
print(f"- Lowest RMSE most often: {top} ({int(wins.max())} of {n} years). " + ("The winner changes across years." if wins.size > 1 else "The same model won every year."))
spread = wide.max(axis=1) - wide.min(axis=1)
print(f"- Year-to-year swings are large compared with gaps between models: RMSE of the same model ranges from {wide.min().min():.3f} to {wide.max().max():.3f}, "
      f"while the average gap between best and worst model within a year is {spread.mean():.3f}." if wide.max().max() - wide.min().min() > spread.mean() else
      f"- Gaps between models within a year (average {spread.mean():.3f}) are larger than the overall year-to-year range of any one model.")
print(f"- Hardest year (highest average RMSE): {int(diff.index[-1])}; easiest: {int(diff.index[0])}. Which years are 'hard' may reflect real shocks in the data, not model quality.")

# ---------- random split vs rolling origin (many chronological years) ----------
rs = pd.read_csv(os.path.join(P["tables"], "random_split_results.csv"))
rnd = rs.groupby("model")[["MAE", "RMSE", "R2"]].mean()
cmp = []
for m in MODELS:
    r_ = ev[ev.model == m]
    cmp.append({"model": m, "random_mean_RMSE": rnd.loc[m, "RMSE"], "rolling_mean_RMSE": r_.RMSE.mean(), "diff_random_minus_rolling_RMSE": rnd.loc[m, "RMSE"] - r_.RMSE.mean(),
                "random_RMSE_ratio_vs_persistence": rnd.loc[m, "RMSE"] / rnd.loc["Persistence", "RMSE"],
                "rolling_RMSE_ratio_vs_persistence": r_.RMSE.mean() / ev[ev.model == "Persistence"].RMSE.mean()})
cmp = pd.DataFrame(cmp)
print("\nRandom split vs rolling origin (both cover many years; diff = random - rolling):\n", cmp.round(4).to_string(index=False))
mu.save_csv(cmp, os.path.join(P["tables"], "random_vs_rolling_comparison.csv"), P)
fit = cmp[cmp.model != "Persistence"]
print("Relative to Persistence, the random split is more favourable than the rolling backtest for:",
      fit[fit.random_RMSE_ratio_vs_persistence < fit.rolling_RMSE_ratio_vs_persistence].model.tolist() or "none")

# ---------- Phase 3 experiment summary ----------
summary = {"random_split": {"seeds": 10, "models": eu.MODELS_CHEAP, "mean_future_in_train_pct": float(rs.future_in_train_pct.mean()),
                            "mean_target_visible_pct": float(rs.target_visible_pct.mean())},
           "rolling_origin": {"evaluated_years": [int(y) for y in years_ok], "skipped_years": [int(y) for y in years_skip],
                              "mean_RMSE": {r.model: float(r.mean_RMSE) for r in summ.itertuples()},
                              "years_with_lowest_RMSE": {r.model: int(r.years_with_lowest_RMSE) for r in summ.itertuples()}},
           "note": "LSTM and stacking were not repeated in Experiments A and B (time budget)."}
mu.save_json(summary, os.path.join(P["tables"], "phase3_experiment_summary.json"), P)

# VALIDATION
for f in ("rolling_origin_mae.png", "rolling_origin_rmse.png", "rolling_origin_r2.png"):
    assert os.path.getsize(os.path.join(P["figures"], f)) > 10_000, f
assert np.isfinite(summ.drop(columns="model").select_dtypes("number").fillna(0).to_numpy(float)).all()
assert (summ.n_successful_forecast_years == len(years_ok)).all()
print("\nCELL 48 PASSED")


In [ ]:
# ==================================================
# CELL 49 — CHOOSE AND VERIFY THE MODEL TO EXPLAIN (SHAP preparation)
#
# GOAL:
#   Pick the tree model to explain (Random Forest or XGBoost), using the
#   chronological VALIDATION error, and check that everything SHAP needs is
#   consistent: model file, feature order, input shape, predictions.
#
# WHY VALIDATION AND NOT THE RANDOM-SPLIT RESULT:
#   Models are compared on the data they will be used on: later years. The
#   random-split experiment (CELLS 44-45) is a protocol study and is not
#   used for any choice. The test set is not used to choose either.
#
# WHAT SHAP IS (plain language):
#   For one prediction, SHAP splits "prediction minus the average prediction"
#   into one number per feature (positive pushes the forecast up, negative
#   down). The numbers add up exactly. They describe what the MODEL uses to
#   predict. They are NOT causes: raising a country's internet use would not
#   necessarily change its happiness.
#
# EXPLAINER CHOICE:
#   shap.TreeExplainer works for both Random Forest and XGBoost. We give it
#   a fixed background of 100 training rows (sampled with seed 42) so the
#   attribution is measured against typical training rows, and we explain the
#   133 test rows (the forecast year) after the model was already chosen.
#
# EXPECTED OUTPUT:
#   the validation comparison, the chosen model, shape/feature/prediction
#   checks, an additivity check on a few rows, "CELL 49 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - ModuleNotFoundError shap / SHAP import error with numpy | incompatible shap/numpy | `import shap, numpy; print(shap.__version__, numpy.__version__)` | `!pip install -U shap`, Restart session, SETUP, re-run 49
#   - model does not reproduce saved test predictions | model file from another run | compare `model.predict(Xte)` with the prediction CSV | re-run CELL 29 or 30, then 31, 41, 49
#   - additivity error | shap and xgboost/sklearn versions disagree | printed max error | `!pip install -U shap xgboost`, restart, re-run 49
#   - feature order assertion | different feature order | `model.feature_names_in_` vs feature_spec.json | re-run CELLS 23, 27-30, then 49
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
import shap

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]
print("SHAP", shap.__version__)

master = pd.read_csv(os.path.join(P["tables"], "master_model_comparison.csv")).set_index("model")
cand = master.loc[["Random Forest", "XGBoost"], ["val_RMSE", "val_MAE", "val_R2"]].sort_values("val_RMSE")
print("Chronological VALIDATION comparison of the tree models:\n", cand.round(4).to_string())
chosen = cand.index[0]
gap = (cand.val_RMSE.iloc[1] / cand.val_RMSE.iloc[0] - 1) * 100
print(f"\nChosen for explanation: {chosen} (validation RMSE {cand.val_RMSE.iloc[0]:.4f}; the other is {gap:.1f}% worse - "
      + ("practically tied, so the choice is by rule, not by a real difference)." if gap < 3 else "a clear gap)."))

path = os.path.join(P["models"], {"Random Forest": "random_forest.joblib", "XGBoost": "xgboost.joblib"}[chosen])
model = joblib.load(path)

# ---- checks ----
checks = []
def add(n, ok, d=""):
    checks.append({"check": n, "result": "PASS" if ok else "FAIL", "detail": str(d)})
Xtr, Xte = mu.X(D, "imp", "train"), mu.X(D, "imp", "test")
add("model feature order == Phase 1 feature order", list(model.feature_names_in_) == F, len(F))
add("explained matrix has exactly the 18 model features (target excluded)", list(Xte.columns) == F and D["target"] not in Xte.columns, Xte.shape)
add("input is the imputed UNSCALED matrix used to train this model", np.allclose(Xtr.to_numpy(), pd.read_csv(os.path.join(P["v2"], "train_imputed.csv"))[F].to_numpy()))
pred_file = {"Random Forest": "random_forest_predictions.csv", "XGBoost": "xgboost_predictions.csv"}[chosen]
saved = pd.read_csv(os.path.join(P["pred"], pred_file)); saved = saved[saved.split == "test"].reset_index(drop=True)
now = model.predict(Xte)
add("model reproduces its saved test predictions", np.allclose(now, saved.y_pred.to_numpy(), atol=1e-6))
add("test rows line up with the saved predictions (Code, Year)", (saved.Code.values == truth["test"].Code.values).all() and (saved.Year.values == truth["test"].Year.values).all())

# ---- deterministic background and explanation sets ----
bg = Xtr.sample(n=100, random_state=mu.SEED)
bg_idx = [int(i) for i in bg.index]
expl_idx = [int(i) for i in Xte.index]
explainer = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
few = Xte.iloc[:5]
sv = explainer(few)
add_err = np.abs(sv.base_values + sv.values.sum(axis=1) - model.predict(few)).max()
add("SHAP additivity: base value + sum of contributions == prediction (5 rows)", add_err < 1e-3, f"max error {add_err:.2e}")
add("SHAP array shape (rows, features)", sv.values.shape == (5, len(F)), sv.values.shape)
out = pd.DataFrame(checks)
print("\n", out.to_string(index=False))

mu.save_json({"chosen_model": chosen, "selected_on": "chronological validation RMSE (not test, not random-split)",
              "validation_rmse": {k: float(v) for k, v in cand.val_RMSE.items()}, "model_path": os.path.relpath(path, PROJECT_ROOT),
              "explainer": "shap.TreeExplainer(model, data=background, feature_perturbation='interventional')", "background_rows": 100, "background_seed": mu.SEED,
              "background_row_index": bg_idx, "explained_rows": "all 133 test rows (origin 2023, target 2024), explained after model selection",
              "explained_row_index": expl_idx, "features": F}, os.path.join(P["tables"], "shap_model_selection.json"), P)
assert (out.result == "PASS").all(), "A SHAP preparation check failed; see the table."
print("\nCELL 49 PASSED")


In [ ]:
# ==================================================
# CELL 50 — GLOBAL SHAP EXPLAINABILITY
#
# GOAL:
#   Compute SHAP values for the 133 test rows, rank features by mean absolute
#   SHAP, and draw a bar chart and a beeswarm plot.
#
# HOW TO READ:
#   mean |SHAP| = how many happiness points, on average, a feature moves the
#   model's forecast away from its average forecast. Larger = the model leans on
#   it more. This is about PREDICTION, not about causes.
#   Beeswarm: each dot is one country; position = push on the forecast
#   (right = up); colour = the feature's own value (red high, blue low).
#   Caution: current score, last year's score and the year before are highly
#   related, so the model can share credit among them. Read the grouped table
#   (same variable, all years together) as well.
#
# EXPECTED OUTPUT:
#   shape/feature checks, ranking table, grouped table, two saved figures,
#   "CELL 50 PASSED". Runtime: well under 2 minutes.
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - ExplainerError / additivity check failed | SHAP version cannot explain this model exactly | see the assertion message with the max error | fallback: replace `shap.TreeExplainer(...)` by `shap.Explainer(model.predict, bg)`(slower) and keep the same checks; or update shap
#   - shape (133,18) assertion | wrong rows/features passed to SHAP | print `Xte.shape` and `sv.values.shape` | re-run CELL 49, then 50
#   - MemoryError / very slow | background too large | the background is fixed at 100 rows | Restart session, re-run 43, 49, 50
#   - beeswarm TypeError | matplotlib/shap mismatch | `shap.__version__` | `!pip install -U shap matplotlib`, restart, re-run 50
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
import shap
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
eu.style()
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]
sel_path = os.path.join(P["tables"], "shap_model_selection.json")
assert os.path.exists(sel_path), "Run CELL 49 first."
sel = json.load(open(sel_path))
assert sel["features"] == F
model = joblib.load(os.path.join(PROJECT_ROOT, sel["model_path"]))
Xtr, Xte = mu.X(D, "imp", "train"), mu.X(D, "imp", "test")
bg = Xtr.loc[sel["background_row_index"]]
assert len(bg) == 100
expl = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
sv = expl(Xte)                                     # Explanation: .values (133 x 18), .base_values, .data
V = np.asarray(sv.values)

# ---------- verification ----------
assert V.shape == (len(Xte), len(F)) == (133, 18), f"SHAP matrix shape {V.shape}"
assert list(sv.feature_names) == F, "SHAP feature names differ from the model feature order"
assert np.allclose(sv.data, Xte.to_numpy(float)), "SHAP was given different feature values than the model input"
assert np.isfinite(V).all()
pred = model.predict(Xte)
err = np.abs(np.asarray(sv.base_values) + V.sum(axis=1) - pred).max()
assert err < 1e-3, f"additivity failed (max error {err})"
print(f"Model: {sel['chosen_model']} | SHAP values {V.shape} | additivity max error {err:.2e} | target column excluded: {D['target'] not in F}")

# ---------- ranking ----------
fm = {c["name"]: c for c in json.load(open(os.path.join(P["v2"], "feature_metadata.json")))["columns"]}
def base_of(n):
    for suf in ("_was_missing", "_lag1", "_lag2"):
        if n.endswith(suf):
            return n[: -len(suf)] + ("(missing flag)" if suf == "_was_missing" else "")
    return n
imp = pd.DataFrame({"feature": F, "mean_abs_shap": np.abs(V).mean(axis=0)})
imp["share_pct"] = 100 * imp.mean_abs_shap / imp.mean_abs_shap.sum()
imp["lag"] = [fm[f]["lag"] for f in imp.feature]
imp["base_variable"] = [base_of(f) for f in imp.feature]
imp["meaning"] = [fm[f]["meaning"] for f in imp.feature]
imp = imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
imp.insert(0, "rank", np.arange(1, len(imp) + 1))
grouped = imp.groupby("base_variable").agg(mean_abs_shap_sum=("mean_abs_shap", "sum"), share_pct=("share_pct", "sum")).sort_values("mean_abs_shap_sum", ascending=False).reset_index()
print("\nGlobal importance (mean |SHAP| in happiness points):\n", imp[["rank", "feature", "mean_abs_shap", "share_pct", "lag"]].round(4).to_string(index=False))
print("\nGrouped by variable (all lags together):\n", grouped.round(4).to_string(index=False))
mu.save_csv(imp, os.path.join(P["tables"], "shap_global_importance.csv"), P)
mu.save_csv(grouped, os.path.join(P["tables"], "shap_global_importance_by_variable.csv"), P)
sv_df = pd.concat([truth["test"][["Country", "Code", "Year", "TargetYear"]].reset_index(drop=True),
                   pd.DataFrame(V, columns=[f"shap_{f}" for f in F]), pd.DataFrame({"shap_base_value": np.asarray(sv.base_values), "model_prediction": pred})], axis=1)
mu.save_csv(sv_df, os.path.join(P["tables"], "shap_values_test.csv"), P)

# ---------- figures ----------
lagcol = {"t (current year)": "#08519c", "t-1 (calendar)": "#6baed6", "t-2 (calendar)": "#bdd7e7", "t": "#fd8d3c"}
fig, ax = plt.subplots(figsize=(9, 7))
d = imp.iloc[::-1]
ax.barh(d.feature, d.mean_abs_shap, color=[lagcol.get(l, "#999999") for l in d.lag])
ax.set_xlabel("Mean |SHAP| (happiness points)"); ax.set_ylabel("Feature")
ax.set_title(f"Global SHAP importance: {sel['chosen_model']} (133 test rows)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ("#08519c", "#6baed6", "#bdd7e7", "#fd8d3c")]
ax.legend(handles, ["current year", "lag 1", "lag 2", "missing flag"], title="Feature year", loc="lower right")
f1 = os.path.join(P["figures"], "shap_global_importance.png"); fig.savefig(f1); plt.show()

plt.figure(figsize=(9, 7))
shap.plots.beeswarm(sv, max_display=len(F), show=False)
plt.title("SHAP beeswarm: push on the forecast per country (colour = feature value)")
plt.xlabel("SHAP value (change in forecast, happiness points)")
f2 = os.path.join(P["figures"], "shap_beeswarm.png"); plt.gcf().savefig(f2, bbox_inches="tight", dpi=150); plt.show()

# ---------- computed reading ----------
top = imp.head(5)
print("\nWHAT THE NUMBERS SAY (about prediction, not causes):")
print("- Top 5 features:", ", ".join(f"{r.feature} ({r.share_pct:.0f}%)" for r in top.itertuples()))
g0 = grouped.iloc[0]
socio = grouped[grouped.base_variable.isin(["UnemploymentRate_WB", "InternetAccessPct_WB", "HealthExpenditurePctGDP_WB", "log_GDPPerCapita_WB"])].share_pct.sum()
print(f"- By variable, '{g0.base_variable}' accounts for {g0.share_pct:.0f}% of total importance; the four socioeconomic variables "
      f"(GDP, unemployment, internet use, health spending, all years) together account for {socio:.1f}%.")
if g0.base_variable == "HappinessScore" and g0.share_pct >= 90:
    print("- The model is almost entirely a function of the country's own recent happiness scores. The added indicators contribute very little to THIS model's forecasts.")
    print("  (This is a statement about what the model uses. Whether the indicators help at all would need an ablation: the same model with and without them.)")
print("- This shows what the model relied on to forecast the test year. It does not show that changing a feature would change happiness.")

assert os.path.getsize(f1) > 10_000 and os.path.getsize(f2) > 10_000
assert np.isclose(imp.share_pct.sum(), 100) and len(imp) == 18
print("\nCELL 50 PASSED")


In [ ]:
# ==================================================
# CELL 51 — LOCAL SHAP EXPLANATIONS (three examples chosen by a fixed rule)
#
# GOAL:
#   Explain individual forecasts: which features pushed this country's
#   forecast up or down relative to the model's average forecast.
#
# HOW THE EXAMPLES ARE PICKED (rule fixed here, not by looking at pretty plots):
#   1) the country with the HIGHEST actual score in the test year
#   2) the country with the LOWEST actual score in the test year
#   3) the country with the LARGEST absolute forecast error (the "surprising" one)
#   If one country satisfies two rules, the next-largest error is used for 3).
#
# EACH EXAMPLE REPORTS: country, origin year, target year, actual, predicted,
#   error (predicted - actual), the three largest positive and negative SHAP
#   contributions, and a waterfall plot. The waterfall starts at the model's
#   average forecast and adds each feature's push to reach the prediction.
#   The plot uses exactly the feature values the model received.
#
# EXPECTED OUTPUT: summary table, local tables, 3 figures, "CELL 51 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - waterfall plot error | shap/matplotlib mismatch | `shap.__version__` | update shap and matplotlib, restart, re-run 49-51
#   - AssertionError contributions do not add up | background differs from CELL 49 selection file | compare shap_model_selection.json | re-run 49, 50, 51
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd, joblib
import shap
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
eu.style()
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
F, truth = D["features"], D["truth"]
sel = json.load(open(os.path.join(P["tables"], "shap_model_selection.json")))
model = joblib.load(os.path.join(PROJECT_ROOT, sel["model_path"]))
Xte = mu.X(D, "imp", "test")
bg = mu.X(D, "imp", "train").loc[sel["background_row_index"]]
sv = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")(Xte)
pred = model.predict(Xte)
t = truth["test"].reset_index(drop=True).copy()
t["pred"] = pred
t["error"] = t.pred - t.y_true
assert list(sv.feature_names) == F and np.allclose(sv.data, Xte.to_numpy(float))

order_err = t.error.abs().sort_values(ascending=False).index.tolist()
i_hi, i_lo = int(t.y_true.idxmax()), int(t.y_true.idxmin())
i_sur = next(i for i in order_err if i not in (i_hi, i_lo))
examples = [("highest_actual_score", i_hi), ("lowest_actual_score", i_lo), ("largest_error", i_sur)]

summ, long = [], []
for label, i in examples:
    c = pd.DataFrame({"feature": F, "feature_value": Xte.iloc[i].to_numpy(float), "shap": np.asarray(sv.values[i])})
    c["abs_shap"] = c.shap.abs()
    pos = c[c.shap > 0].sort_values("shap", ascending=False).head(3)
    neg = c[c.shap < 0].sort_values("shap").head(3)
    r = t.iloc[i]
    summ.append({"example": label, "Country": r.Country, "origin_year": int(r.Year), "target_year": int(r.TargetYear), "actual": r.y_true, "predicted": r.pred,
                 "error_pred_minus_actual": r.error, "model_average_forecast": float(np.asarray(sv.base_values)[i]),
                 "top_positive": "; ".join(f"{a} ({b:+.3f})" for a, b in zip(pos.feature, pos.shap)),
                 "top_negative": "; ".join(f"{a} ({b:+.3f})" for a, b in zip(neg.feature, neg.shap))})
    long.append(c.assign(example=label, Country=r.Country).sort_values("abs_shap", ascending=False))
    plt.figure(figsize=(9, 6))
    shap.plots.waterfall(sv[i], max_display=10, show=False)
    plt.title(f"{r.Country}: forecast for {int(r.TargetYear)} (actual {r.y_true:.2f}, predicted {r.pred:.2f})", fontsize=12)
    plt.gcf().savefig(os.path.join(P["figures"], f"shap_local_{label}.png"), bbox_inches="tight", dpi=150); plt.show()

summ = pd.DataFrame(summ)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
print(summ[["example", "Country", "origin_year", "target_year", "actual", "predicted", "error_pred_minus_actual", "model_average_forecast"]].round(3).to_string(index=False))
for r in summ.itertuples():
    print(f"\n{r.example} - {r.Country}\n  largest positive pushes: {r.top_positive}\n  largest negative pushes: {r.top_negative}")
mu.save_csv(summ, os.path.join(P["tables"], "shap_local_examples_summary.csv"), P)
mu.save_csv(pd.concat(long, ignore_index=True), os.path.join(P["tables"], "shap_local_contributions.csv"), P)
print("\nReminder: these numbers describe how this model produced each forecast. They do not say what would change a country's happiness.")

# VALIDATION
for r in summ.itertuples():
    row = long[[l for l, _ in examples].index(r.example)]
    assert abs(row.shap.sum() + r.model_average_forecast - r.predicted) < 1e-3, f"{r.example}: contributions do not add up to the prediction"
assert summ.Country.nunique() == 3
for label, _ in examples:
    assert os.path.getsize(os.path.join(P["figures"], f"shap_local_{label}.png")) > 10_000
print("CELL 51 PASSED")


In [ ]:
# ==================================================
# CELL 52 — TEST-SET ERROR ANALYSIS (saved Phase 2 chronological predictions)
#
# GOAL:
#   Find where the forecasts missed most, using the predictions already saved
#   in Phase 2 (nothing is recomputed or retrained).
#
# WHAT IS PRODUCED:
#   - the 10 countries with the largest absolute error, for each of the six models
#   - a per-model summary: average signed error (bias), MAE, RMSE, and in how
#     many countries the model beat Persistence
#   - errors grouped by the country's actual score level (three equal-size groups)
#   - errors by forecast year (from the rolling-origin backtest, because the
#     chronological test has only one forecast year)
#   Signed error = predicted - actual. Positive = the model forecast too high.
#
# CAUTION (important): with 133 countries and one test year we cannot say any
#   region or income group is systematically harder. We have no region column.
#   Repeated appearances in the worst-10 lists are described, not explained.
#
# EXPECTED OUTPUT: tables printed and saved, "CELL 52 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - FileNotFoundError predictions | Phase 2 outputs missing | `!ls results/predictions` | re-run the named CELL (28-40)
#   - AssertionError error-analysis RMSE != master table | prediction files from different runs | compare error_summary_by_model.csv with master_model_comparison.csv | re-run CELLS 28-41, then 52
# ==================================================
import os, sys, json, importlib
import numpy as np, pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
D = mu.load_phase1(P)
files = {"Persistence": "persistence_predictions.csv", "Linear Regression": "linear_regression_predictions.csv", "Random Forest": "random_forest_predictions.csv",
         "XGBoost": "xgboost_predictions.csv", "LSTM": "lstm_predictions.csv", "Stacked Ensemble": "stacked_predictions.csv"}
frames = []
for m, f in files.items():
    d = pd.read_csv(os.path.join(P["pred"], f)).query("split == 'test'").copy()
    mu.validate_pred_frame(d, D["truth"]["test"].merge(d[mu.KEYS], on=mu.KEYS), "test", m)
    d["model"] = m
    d["signed_error"] = d.y_pred - d.y_true
    d["abs_error"] = d.signed_error.abs()
    frames.append(d)
allp = pd.concat(frames, ignore_index=True)
pers = allp[allp.model == "Persistence"].set_index("Code").abs_error

# ---------- worst 10 per model ----------
worst = (allp.sort_values(["model", "abs_error"], ascending=[True, False]).groupby("model").head(10)
         [["model", "Country", "Code", "Year", "TargetYear", "y_true", "y_pred", "abs_error", "signed_error"]]
         .rename(columns={"Year": "origin_year", "TargetYear": "target_year", "y_true": "actual_score", "y_pred": "predicted_score"}))
worst["rank_within_model"] = worst.groupby("model").cumcount() + 1
worst = worst.sort_values(["model", "rank_within_model"]).reset_index(drop=True)
mu.save_csv(worst, os.path.join(P["tables"], "worst_10_country_errors.csv"), P)
master = pd.read_csv(os.path.join(P["tables"], "master_model_comparison.csv")).set_index("model")
best_val = master.drop("Persistence").val_RMSE.idxmin()
for m in (best_val, "Random Forest"):
    print(f"\nWorst 10 countries - {m} (test year, target 2024):")
    print(worst[worst.model == m].drop(columns=["model", "Code"]).round(3).to_string(index=False))

# ---------- per-model summary ----------
rows = []
for m, d in allp.groupby("model"):
    beat = np.mean([r.abs_error < pers[r.Code] for r in d.itertuples()]) if m != "Persistence" else np.nan
    rows.append({"model": m, "n_countries": len(d), "mean_signed_error_bias": d.signed_error.mean(), "MAE": d.abs_error.mean(), "RMSE": np.sqrt((d.signed_error ** 2).mean()),
                 "max_abs_error": d.abs_error.max(), "share_of_countries_better_than_persistence": beat})
summ = pd.DataFrame(rows).set_index("model").loc[list(files)].reset_index()
print("\nPer-model error summary (signed error = predicted - actual):\n", summ.round(4).to_string(index=False))
mu.save_csv(summ, os.path.join(P["tables"], "error_summary_by_model.csv"), P)

# ---------- by actual-score level ----------
tt = D["truth"]["test"][["Code", "y_true"]].copy()
tt["score_level"] = pd.qcut(tt.y_true, 3, labels=["lowest third", "middle third", "highest third"])
lv = allp.merge(tt[["Code", "score_level"]], on="Code")
bylevel = lv.groupby(["model", "score_level"], observed=True).agg(n=("abs_error", "size"), MAE=("abs_error", "mean"), bias=("signed_error", "mean")).reset_index()
print("\nError by actual-score level (each group has about", len(tt) // 3, "countries):\n", bylevel.pivot(index="model", columns="score_level", values="MAE").loc[list(files)].round(4).to_string())
mu.save_csv(bylevel, os.path.join(P["tables"], "error_by_score_level.csv"), P)

# ---------- recurring worst countries ----------
rec = worst.groupby("Country").model.nunique().sort_values(ascending=False)
print("\nCountries in the worst-10 list of several models (count of models, out of 6):\n", rec[rec >= 3].to_string() if (rec >= 3).any() else "none appear in 3 or more lists")
mu.save_csv(rec.rename("n_models_with_country_in_worst10").reset_index(), os.path.join(P["tables"], "recurring_worst_countries.csv"), P)

# ---------- by forecast year (rolling origin) ----------
roll_path = os.path.join(P["pred"], "rolling_origin_predictions.csv")
if os.path.exists(roll_path):
    r = pd.read_csv(roll_path)
    r["signed_error"] = r.y_pred - r.y_true
    by_year = r.groupby(["forecast_year", "model"]).agg(n=("signed_error", "size"), MAE=("signed_error", lambda s: s.abs().mean()),
                                                         RMSE=("signed_error", lambda s: np.sqrt((s ** 2).mean())), bias=("signed_error", "mean")).reset_index()
    print("\nBias (mean signed error) by forecast year, rolling origin:\n", by_year.pivot(index="forecast_year", columns="model", values="bias").round(3).to_string())
    mu.save_csv(by_year, os.path.join(P["tables"], "error_by_forecast_year.csv"), P)
else:
    print("\nrolling_origin_predictions.csv not found: run CELLS 46-47 for the by-year table.")

tmin = float(D["truth"]["train"].y_true.min())
below = D["truth"]["test"][D["truth"]["test"].y_true < tmin]
print(f"\nREADING THESE TABLES (description only): the lowest score a model saw as a training TARGET is {tmin:.3f}; "
      f"{len(below)} test countr{'y has' if len(below) == 1 else 'ies have'} an actual score below that"
      + (f" ({', '.join(below.Country)})." if len(below) else ".")
      + " Tree-based models cannot output values beyond the range of training targets, so such cases are limited for them by construction. "
      "This is not a claim about which regions or income groups are harder: there is no region column and only one test year.")
# VALIDATION
assert (worst.groupby("model").size() == 10).all() and len(worst) == 60
assert np.isfinite(summ.select_dtypes("number").to_numpy(float)[~np.isnan(summ.select_dtypes("number").to_numpy(float))]).all()
for m in files:                                  # recomputed RMSE must equal the master table
    assert abs(summ.set_index("model").loc[m, "RMSE"] - master.loc[m, "test_RMSE"]) < 1e-9, f"{m}: error analysis RMSE != master table"
print("CELL 52 PASSED")


In [ ]:
# ==================================================
# CELL 53 — FINAL FIGURE PACKAGE (7 figures)
#
# GOAL:
#   Produce the seven final figures with one consistent style. Figures that
#   already exist from earlier cells (random vs chronological, rolling RMSE
#   and MAE, SHAP importance) are COPIED under their final names, not
#   redrawn, so there are no duplicate graphs. Figures 1, 2 and 7 are drawn here.
#
#   final_fig1_six_model_test_comparison.png   MAE/RMSE of 6 models, 95% bootstrap bars
#   final_fig2_actual_vs_predicted.png         actual vs predicted (best fitted model by validation vs Persistence)
#   final_fig3_random_vs_chronological.png     (copy of CELL 45 figure)
#   final_fig4_rolling_rmse.png                (copy of CELL 48 figure)
#   final_fig5_rolling_mae.png                 (copy of CELL 48 figure)
#   final_fig6_shap_global_importance.png      (copy of CELL 50 figure)
#   final_fig7_worst_country_errors.png        10 largest test errors of the best fitted model
#
# EXPECTED OUTPUT: 7 files exist and are larger than 10 KB, "CELL 53 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - missing source figure (CELL 45/48/50) | the figure-creating cell was not run | `!ls results/figures` | run the cell named in the message, then 53
#   - KeyError Code / row mismatch | prediction file order differs | print both Code columns | re-run 43 and 53
# ==================================================
import os, sys, json, importlib, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
eu.style()
P = mu.make_paths(PROJECT_ROOT)
FIG = P["figures"]
master = pd.read_csv(os.path.join(P["tables"], "master_model_comparison.csv"))
boot = pd.read_csv(os.path.join(P["tables"], "chronological_test_bootstrap_ci.csv")).set_index("model")
order = list(master.model)
best_val = master[master.model != "Persistence"].sort_values("val_RMSE").iloc[0].model

# ---- Fig 1 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, met in zip(axes, ("MAE", "RMSE")):
    v = master.set_index("model").loc[order, f"test_{met}"].to_numpy()
    lo = boot.loc[order, f"{met}_lo"].to_numpy(); hi = boot.loc[order, f"{met}_hi"].to_numpy()
    ax.bar(order, v, color=[eu.COLORS[m] for m in order], yerr=[v - lo, hi - v], capsize=4)
    ax.axhline(master.set_index("model").loc["Persistence", f"test_{met}"], color="k", ls="--", lw=1, label="Persistence")
    ax.set_ylabel(f"{met} (happiness points)"); ax.set_xlabel("Model"); ax.set_title(f"Test {met} (95% country-bootstrap interval)")
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=20, ha="right")
    for i, x in enumerate(v):
        ax.text(i, hi[i] + 0.002, f"{x:.3f}", ha="center", fontsize=9)
    ax.legend()
fig.suptitle("Six models on the chronological test (133 countries, forecasting 2024)", fontsize=14)
f1 = os.path.join(FIG, "final_fig1_six_model_test_comparison.png"); fig.savefig(f1); plt.show()

# ---- Fig 2 ----
pf = {m: pd.read_csv(os.path.join(P["pred"], f)).query("split == 'test'").sort_values("Code").reset_index(drop=True)
      for m, f in (("Persistence", "persistence_predictions.csv"), (best_val, {"Linear Regression": "linear_regression_predictions.csv", "Random Forest": "random_forest_predictions.csv",
                                                                                   "XGBoost": "xgboost_predictions.csv", "LSTM": "lstm_predictions.csv", "Stacked Ensemble": "stacked_predictions.csv"}[best_val]))}
fig, ax = plt.subplots(figsize=(7.5, 7))
lo_, hi_ = 0.8, 8.4
ax.plot([lo_, hi_], [lo_, hi_], "k--", lw=1, label="perfect forecast")
ax.scatter(pf["Persistence"].y_true, pf["Persistence"].y_pred, s=22, alpha=0.6, color=eu.COLORS["Persistence"], label="Persistence")
ax.scatter(pf[best_val].y_true, pf[best_val].y_pred, s=22, alpha=0.7, color=eu.COLORS[best_val], label=best_val)
ax.set_xlim(lo_, hi_); ax.set_ylim(lo_, hi_)
ax.set_xlabel("Actual happiness in 2024 (0-10 scale)"); ax.set_ylabel("Predicted happiness for 2024 (0-10 scale)")
ax.set_title(f"Actual vs predicted, test set: {best_val} (best fitted model on validation) vs Persistence"); ax.legend()
f2 = os.path.join(FIG, "final_fig2_actual_vs_predicted.png"); fig.savefig(f2); plt.show()

# ---- copies ----
copies = {"random_vs_chronological.png": "final_fig3_random_vs_chronological.png", "rolling_origin_rmse.png": "final_fig4_rolling_rmse.png",
          "rolling_origin_mae.png": "final_fig5_rolling_mae.png", "shap_global_importance.png": "final_fig6_shap_global_importance.png"}
for src, dst in copies.items():
    assert os.path.exists(os.path.join(FIG, src)), f"{src} missing: run the cell that creates it (CELL 45, 48 or 50)."
    shutil.copy(os.path.join(FIG, src), os.path.join(FIG, dst))

# ---- Fig 7 ----
w = pd.read_csv(os.path.join(P["tables"], "worst_10_country_errors.csv"))
wm = w[w.model == best_val].sort_values("abs_error")
perr = pd.read_csv(os.path.join(P["pred"], "persistence_predictions.csv")).query("split == 'test'").assign(pe=lambda d: (d.y_pred - d.y_true).abs()).set_index("Code").pe
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(wm))
ax.barh(y + 0.2, wm.abs_error, 0.4, color=eu.COLORS[best_val], label=best_val)
ax.barh(y - 0.2, [perr[c] for c in wm.Code], 0.4, color=eu.COLORS["Persistence"], label="Persistence (same country)")
ax.set_yticks(y); ax.set_yticklabels([f"{c} ({s:+.2f})" for c, s in zip(wm.Country, wm.signed_error)])
ax.set_xlabel("Absolute error (happiness points)"); ax.set_ylabel("Country (signed error: predicted - actual)")
ax.set_title(f"10 largest test errors of {best_val} (forecast of 2024)"); ax.legend(loc="lower right")
f7 = os.path.join(FIG, "final_fig7_worst_country_errors.png"); fig.savefig(f7); plt.show()

names = ["final_fig1_six_model_test_comparison.png", "final_fig2_actual_vs_predicted.png", "final_fig3_random_vs_chronological.png", "final_fig4_rolling_rmse.png",
         "final_fig5_rolling_mae.png", "final_fig6_shap_global_importance.png", "final_fig7_worst_country_errors.png"]
for n in names:
    assert os.path.getsize(os.path.join(FIG, n)) > 10_000, f"{n} missing or too small"
print("Figures created:", *names, sep="\n  ")
print("\nCELL 53 PASSED")


In [ ]:
# ==================================================
# CELL 54 — FINAL EXPERIMENT VALIDATION (Phase 3)
#
# GOAL:
#   Verify every Phase 3 output, from the files on disk, and write a short
#   completion report. Cells 55-56 are reserved for your paper and are skipped.
#
# WHAT IS CHECKED:
#   random split : 10 seeds x 4 models finished, metrics finite, predictions
#                  match their identifiers, split files are consistent
#   rolling      : training targets always earlier than the forecast year,
#                  skipped years documented, 2023 fold == Phase 2 validation
#   SHAP         : right model, 18 features in the right order, values add up
#   errors       : error analysis recomputes the Phase 2 test RMSE
#   figures/CSV  : all exist, are readable, not empty
#   discipline   : hyperparameters unchanged from the validation-based Phase 2
#                  choice; selection tables hold no test columns; model for SHAP
#                  chosen on validation; generated text avoids causal wording
#
# EXPECTED OUTPUT: PASS table (all rows PASS), report, "CELL 54 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - a check says FAIL | a Phase 3 output is stale or missing | read the FAIL row | re-run the cell the check refers to, then 54 (never edit the check to pass)
#   - causal-wording hit | a generated file contains wording like 'causes' | the file:line is printed | rewrite that sentence in its generating cell, re-run, then 54
# ==================================================
import os, sys, json, importlib, re, glob
import numpy as np, pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu, experiment_utils as eu
importlib.reload(mu); importlib.reload(eu)
P = mu.make_paths(PROJECT_ROOT)
T, PR, FG = P["tables"], P["pred"], P["figures"]
rows = []
def add(n, ok, d=""):
    rows.append({"check": n, "result": "PASS" if ok else "FAIL", "detail": str(d)})
rd = lambda p: pd.read_csv(os.path.join(PROJECT_ROOT, p))
spec = json.load(open(os.path.join(P["v2"], "feature_spec.json")))
F, TG = spec["feature_cols"], spec["target_col"]
sup = rd("data/processed_v2/supervised_all.csv")
cfg = json.load(open(os.path.join(T, "phase3_experiment_config.json")))

# ---------- random split ----------
rs, rp, ra = rd("results/tables/random_split_results.csv"), rd("results/predictions/random_split_predictions.csv"), rd("results/predictions/random_split_assignments.csv")
add("random split: 10 seeds completed for every model", rs.groupby("model").seed.nunique().eq(10).all() and set(rs.model) == set(eu.MODELS_CHEAP), rs.groupby("model").seed.nunique().to_dict())
add("random split: all metrics present and finite", np.isfinite(rs[["MAE", "RMSE", "R2"]].to_numpy(float)).all() and len(rs) == 40)
mm = rp.merge(sup[["Code", "Year", "TargetYear", TG]], on=["Code", "Year", "TargetYear"], how="left")
add("random split: predictions match their identifiers and real targets", mm[TG].notna().all() and np.allclose(mm[TG], mm.y_true))
add("random split: each seed's test set has 222 rows, train 886, no overlap",
    all(set(g.partition) == {"random_train", "random_test"} and (g.partition == "random_test").sum() == 222 for _, g in ra.groupby("seed")) and
    all(not (set(zip(g[g.partition == "random_test"].Code, g[g.partition == "random_test"].Year)) & set(zip(g[g.partition == "random_train"].Code, g[g.partition == "random_train"].Year))) for _, g in ra.groupby("seed")))
rec = rp.groupby(["seed", "model"]).apply(lambda d: mu.metrics(d.y_true, d.y_pred)["RMSE"], include_groups=False).reset_index(name="rmse")
mg = rs.merge(rec, on=["seed", "model"])
add("random split: metrics recomputed from saved predictions match the results table", np.allclose(mg.RMSE, mg.rmse, atol=1e-9))
comp = rd("results/tables/random_vs_chronological_comparison.csv")
add("random vs chronological comparison table complete", len(comp) == 4 and np.isfinite(comp.select_dtypes("number").to_numpy(float)).all())

# ---------- rolling origin ----------
rr, rpred, el = rd("results/tables/rolling_origin_results.csv"), rd("results/predictions/rolling_origin_predictions.csv"), rd("results/tables/rolling_origin_eligibility.csv")
add("rolling: every training fold used only targets earlier than its forecast year", (rpred.train_max_target_year < rpred.forecast_year).all())
add("rolling: evaluation rows have TargetYear == forecast year", (rpred.TargetYear == rpred.forecast_year).all())
skipped = el[~el.eligible]
add("rolling: skipped years are documented with a reason", (rr[rr.status == "skipped"].forecast_year.sort_values().tolist() == skipped.forecast_year.sort_values().tolist()) and skipped.reason_if_skipped.astype(str).str.len().gt(0).all(),
    skipped.forecast_year.tolist())
add("rolling: every eligible year has all 4 models", rr[rr.status == "evaluated"].groupby("forecast_year").model.nunique().eq(4).all())
add("rolling: metrics finite", np.isfinite(rr[rr.status == "evaluated"][["MAE", "RMSE", "R2"]].to_numpy(float)).all())
master = rd("results/tables/master_model_comparison.csv").set_index("model")
if 2023 in set(rr.forecast_year):
    ok = all(abs(float(rr[(rr.forecast_year == 2023) & (rr.model == m)].RMSE.iloc[0]) - float(master.loc[m, "val_RMSE"])) < 1e-6 for m in eu.MODELS_CHEAP)
    add("rolling: 2023 fold reproduces the Phase 2 validation RMSE (independent cross-check)", ok)
chk = rpred.merge(sup[["Code", "Year", TG]], on=["Code", "Year"], how="left")
add("rolling: predictions match real targets", np.allclose(chk[TG], chk.y_true))

# ---------- SHAP ----------
sel = json.load(open(os.path.join(T, "shap_model_selection.json")))
gi = rd("results/tables/shap_global_importance.csv")
add("SHAP: importance table has exactly the 18 model features, no target column", sorted(gi.feature) == sorted(F) and TG not in set(gi.feature))
sv = rd("results/tables/shap_values_test.csv")
sc = [f"shap_{f}" for f in F]
add("SHAP: values table is (133 rows, 18 features) in model order", sv.shape[0] == 133 and list(sv[sc].columns) == sc)
add("SHAP: contributions + base value reproduce the model predictions", np.abs(sv[sc].sum(axis=1) + sv.shap_base_value - sv.model_prediction).max() < 1e-3)
add("SHAP: model chosen on chronological validation (not test, not random split)", "validation" in sel["selected_on"] and sel["chosen_model"] == min(sel["validation_rmse"], key=sel["validation_rmse"].get))
add("SHAP: local explanations add up", (lambda L, S: all(abs(L[L.example == e].shap.sum() + S.set_index("example").loc[e, "model_average_forecast"] - S.set_index("example").loc[e, "predicted"]) < 1e-3 for e in S.example))(
    rd("results/tables/shap_local_contributions.csv"), rd("results/tables/shap_local_examples_summary.csv")))

# ---------- error analysis ----------
es = rd("results/tables/error_summary_by_model.csv").set_index("model")
add("error analysis recomputes the saved Phase 2 test RMSE for all six models", all(abs(es.loc[m, "RMSE"] - master.loc[m, "test_RMSE"]) < 1e-9 for m in master.index))
w = rd("results/tables/worst_10_country_errors.csv")
add("worst-10 table: 10 rows for each of 6 models", w.groupby("model").size().eq(10).all() and w.model.nunique() == 6)

# ---------- files ----------
figs = ["random_vs_chronological.png", "rolling_origin_mae.png", "rolling_origin_rmse.png", "rolling_origin_r2.png", "shap_global_importance.png", "shap_beeswarm.png",
        "shap_local_highest_actual_score.png", "shap_local_lowest_actual_score.png", "shap_local_largest_error.png"] + [f"final_fig{i}_{n}.png" for i, n in
        enumerate(["six_model_test_comparison", "actual_vs_predicted", "random_vs_chronological", "rolling_rmse", "rolling_mae", "shap_global_importance", "worst_country_errors"], 1)]
bad = [f for f in figs if not (os.path.exists(os.path.join(FG, f)) and os.path.getsize(os.path.join(FG, f)) > 10_000)]
add("all 16 Phase 3 figures exist and are non-trivial", not bad, bad)
phase3_tables = ["random_split_results.csv", "random_vs_chronological_comparison.csv", "chronological_test_bootstrap_ci.csv", "rolling_origin_eligibility.csv", "rolling_origin_results.csv",
                 "rolling_origin_summary.csv", "random_vs_rolling_comparison.csv", "shap_global_importance.csv", "shap_global_importance_by_variable.csv", "shap_local_examples_summary.csv",
                 "worst_10_country_errors.csv", "error_summary_by_model.csv", "error_by_score_level.csv", "error_by_forecast_year.csv"]
unread = []
for f in phase3_tables:
    try:
        d = pd.read_csv(os.path.join(T, f)); assert len(d) > 0
    except Exception as e:
        unread.append(f)
add("all Phase 3 CSV tables load and are not empty", not unread, unread)

# ---------- discipline ----------
p2 = {"rf": json.load(open(os.path.join(P["models"], "random_forest_config.json"))), "xgb": json.load(open(os.path.join(P["models"], "xgboost_config.json")))}
cur = eu.load_fixed_params(P)
add("hyperparameters used in Phase 3 == Phase 2 validation-selected values (none retuned)",
    cur["rf"] == cfg["hyperparameters"]["rf"] and cur["xgb"] == cfg["hyperparameters"]["xgb"] and all(p2["rf"][k] == cur["rf"][k] for k in cur["rf"]))
sel_tables = ["rf_grid_results.csv", "xgb_grid_results.csv", "ridge_sensitivity_validation_only.csv", "lstm_candidate_validation.csv", "lstm_validation_comparison.csv"]
add("selection tables contain no test columns", not any("test" in c.lower() for f in sel_tables for c in pd.read_csv(os.path.join(T, f)).columns))
add("Phase 3 chronological comparisons read the saved Phase 2 predictions (test rows 133 each)", all(len(pd.read_csv(os.path.join(PR, f)).query("split == 'test'")) == 133 for f in
    ["persistence_predictions.csv", "linear_regression_predictions.csv", "random_forest_predictions.csv", "xgboost_predictions.csv", "lstm_predictions.csv", "stacked_predictions.csv"]))

# generated text: causal-language scan (files created by Phase 3 cells)
texts = [os.path.join(T, f) for f in ("phase3_experiment_summary.json", "shap_model_selection.json", "phase3_experiment_config.json")]
BANNED = re.compile(r"\b(causes|caused|causing|drives|driven by|leads to|results in|effect of|impact of|improves happiness|raises happiness)\b", re.I)
hits = []
for pth in texts + glob.glob(os.path.join(T, "shap_local_examples_summary.csv")):
    for i, line in enumerate(open(pth, errors="ignore"), 1):
        if BANNED.search(line):
            hits.append(f"{os.path.basename(pth)}:{i}")
add("generated Phase 3 text files contain no causal wording", not hits, hits)

out = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 90)
print(out.to_string(index=False))
mu.save_csv(out, os.path.join(T, "phase3_experiment_validation.csv"), P)

# ---------- completion report (numbers read from files) ----------
rs_mean = rs.groupby("model")[["MAE", "RMSE", "R2"]].mean()
rsum = rd("results/tables/rolling_origin_summary.csv").set_index("model")
rpt = ["# Phase 3 experiment completion report", "",
       f"- Checks passed: {int((out.result == 'PASS').sum())}/{len(out)}",
       f"- Random split: 10 seeds x 4 models; mean test RMSE " + ", ".join(f"{m} {rs_mean.loc[m, 'RMSE']:.4f}" for m in eu.MODELS_CHEAP),
       f"- Rolling origin: evaluated years {sorted(rr[rr.status == 'evaluated'].forecast_year.unique().tolist())}; skipped {skipped.forecast_year.tolist() or 'none'}",
       "- Rolling mean RMSE: " + ", ".join(f"{m} {rsum.loc[m, 'mean_RMSE']:.4f}" for m in eu.MODELS_CHEAP),
       f"- SHAP model: {sel['chosen_model']} (chosen on chronological validation); top feature {gi.iloc[0].feature} ({gi.iloc[0].share_pct:.1f}% of total mean |SHAP|)",
       "- Not repeated in Experiments A/B: LSTM and stacking (time budget).",
       "- All explanations describe model behaviour, not causes."]
open(os.path.join(T, "phase3_completion_report.md"), "w").write("\n".join(rpt) + "\n")
print("\n" + "\n".join(rpt))
assert (out.result == "PASS").all(), "A Phase 3 validation check FAILED. Re-run the cell named in the check, then CELL 54."
print("\nCELL 54 PASSED")


In [ ]:
# ==================================================
# CELL 57 — DEPENDENCY AND ENVIRONMENT AUDIT
#
# GOAL:
#   Record exactly which Python and package versions produced your results,
#   compare them with requirements.txt, fix requirements.txt ONLY where it
#   disagrees with the environment that was really used, and write
#   ENVIRONMENT.md so another student can recreate the setup.
#
# WHY:
#   Results can change slightly between package versions. A requirements file
#   that names different versions from the ones you ran is misleading.
#
# WHAT HAPPENS:
#   importlib.metadata.version(name)  asks Python which version is installed (read-only)
#   python -m pip check               asks pip whether installed packages conflict (read-only)
#   requirements.txt is rewritten only if needed, after a backup copy in
#   results/_backups/. Nothing is reinstalled.
#   Set UPDATE_REQUIREMENTS = False to only report.
#
# EXPECTED OUTPUT:
#   a version table (status OK / UPDATED / ADDED), the pip check message,
#   "CELL 57 PASSED".
#
# COMMON FAILURES AND FIXES  (error | why it happens | diagnostic | exact fix and cells to re-run):
#   - pip check lists conflicts | Colab has many preinstalled packages | read the lines that name our packages | ignore unrelated ones; if a package of ours is named, `!pip install -U <package>` then restart
#   - PermissionError writing requirements.txt | Drive hiccup | `!ls -l requirements.txt` | re-run 57
# ==================================================
import os, sys, json, importlib, platform, subprocess, shutil, datetime
import importlib.metadata as md
import pandas as pd

PROJECT_ROOT = globals().get("PROJECT_ROOT", "/content/drive/MyDrive/CountryWellness/Country-Wellness-Score-Prediction")
assert os.path.isdir(PROJECT_ROOT), "Project folder not found. Run SETUP, then CELL 43."
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import modeling_utils as mu
importlib.reload(mu)
P = mu.make_paths(PROJECT_ROOT)
UPDATE_REQUIREMENTS = True
REQ = os.path.join(PROJECT_ROOT, "requirements.txt")

USED = ["pandas", "numpy", "scikit-learn", "xgboost", "tensorflow", "keras", "shap", "matplotlib", "seaborn", "joblib", "statsmodels"]
ALIAS = {"tensorflow": ("tensorflow", "tensorflow-cpu")}


def installed(pkg):
    for name in ALIAS.get(pkg, (pkg,)):
        try:
            return name, md.version(name)
        except md.PackageNotFoundError:
            continue
    return pkg, None


pinned = {}
if os.path.exists(REQ):
    for line in open(REQ):
        line = line.strip()
        if line and not line.startswith("#") and "==" in line:
            k, v = line.split("==", 1)
            pinned[k.strip()] = v.strip()
print("Python", platform.python_version(), "|", platform.platform(), "| CPUs:", os.cpu_count())

rows, new_pins = [], dict(pinned)
for pkg in USED:
    name, ver = installed(pkg)
    pin = pinned.get(pkg) or pinned.get(name)
    if ver is None:
        status = "NOT INSTALLED (not used in this run)" if pkg == "statsmodels" else "NOT INSTALLED"
    elif pin is None:
        status = "ADDED to requirements" if UPDATE_REQUIREMENTS else "missing from requirements"
        new_pins[pkg if name == pkg else name] = ver
    elif pin != ver:
        status = "UPDATED pin" if UPDATE_REQUIREMENTS else "pin differs from installed"
        new_pins[pkg if pkg in pinned else name] = ver
    else:
        status = "OK"
    rows.append({"package": pkg, "installed": ver or "-", "pinned_in_requirements": pin or "-", "status": status})
tbl = pd.DataFrame(rows)
print("\n", tbl.to_string(index=False))
mu.save_csv(tbl, os.path.join(P["tables"], "environment_audit.csv"), P)
for must in ("pandas", "numpy", "scikit-learn", "xgboost", "shap", "joblib"):
    assert installed(must)[1] is not None, f"{must} is not installed. Run  !pip install {must}  then restart the runtime."

changed = new_pins != pinned
if changed and UPDATE_REQUIREMENTS:
    os.makedirs(P["backups"], exist_ok=True)
    if os.path.exists(REQ):
        shutil.copy(REQ, os.path.join(P["backups"], datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_requirements.txt"))
    with open(REQ, "w") as f:
        f.write("\n".join(f"{k}=={v}" for k, v in sorted(new_pins.items())) + "\n")
    print("\nrequirements.txt updated (backup saved). New content:\n", open(REQ).read())
else:
    print("\nrequirements.txt already matches the running environment; not modified." if not changed else "\n(report only; requirements.txt not changed)")

# compatibility check (read-only). Unrelated Colab packages may be listed; only conflicts naming our packages matter.
pc = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
pip_msg = (pc.stdout + pc.stderr).strip()
ours = [l for l in pip_msg.splitlines() if any(l.lower().startswith(u.lower().replace("-", "_")) or l.lower().startswith(u.lower()) for u in USED)]
print("\npip check:", pip_msg[:600] if pip_msg else "(no output)")
print("Conflicts that involve our packages:", ours if ours else "none")

env_md = ["# Environment used for the reported results", "", f"Generated by CELL 57 on {datetime.date.today().isoformat()}.", "",
          f"- Python: {platform.python_version()}", f"- Platform: {platform.platform()}", f"- CPUs: {os.cpu_count()} (no GPU needed; everything runs on CPU)",
          "- Random seed: 42 (all experiments)", "", "| Package | Version |", "|---|---|"]
for pkg in USED:
    n, v = installed(pkg)
    if v:
        env_md.append(f"| {pkg} | {v} |")
env_md += ["", "## Recreate", "```", "python -m venv .venv && source .venv/bin/activate", "pip install -r requirements.txt", "```",
           "The project was developed in Google Colab with Google Drive. Different package versions can change numbers in the last decimals.",
           "", "## pip check", "```", pip_msg or "No broken requirements found.", "```"]
open(os.path.join(PROJECT_ROOT, "ENVIRONMENT.md"), "w").write("\n".join(env_md) + "\n")
print("\nWrote ENVIRONMENT.md")

# VALIDATION
final_req = {l.split("==")[0] for l in open(REQ) if "==" in l}
for pkg in ("pandas", "numpy", "scikit-learn", "xgboost", "shap", "joblib", "matplotlib"):
    assert pkg in final_req, f"{pkg} is not pinned in requirements.txt"
assert os.path.exists(os.path.join(PROJECT_ROOT, "ENVIRONMENT.md"))
print("CELL 57 PASSED")
